# Overview — What We Are Doing Here

In this stage, we turn the raw mathematical groups created during the clustering process into clear, understandable behavioral insights. Instead of treating clusters as arbitrary numbers, we analyze session activity, pick the most reliable algorithm, and assign human-readable labels to each behavioral pattern.

### Key Steps Explained

* **Cluster Profiling**: We inspect each group created by K-Means, DBSCAN, and Agglomerative Clustering to identify the unique behavioral traits that make a specific group stand out from average sessions.
* **Cross-Model Comparison**: We compare all three algorithms against each other to check for consistency and see if they identified similar user behavior patterns.
* **Model Selection and Labeling**: We select **DBSCAN** as our primary model because of its strong ability to isolate normal patterns while flagging unusual activity as noise. We then assign descriptive names (such as "Normal Baseline Behavior" or "Isolated Behavior") to all sessions.
* **Detailed DBSCAN Profiling**: We build a comprehensive, reusable reference guide for the chosen DBSCAN model to document its exact behavioral signatures for future analysis.
* **Cluster Validation**: We re-run the model on random sub-samples of our data to ensure the discovered groups remain stable, reliable, and consistent.

In [ ]:
# ================================================================
# STEP 7A — CLUSTER PROFILING & BEHAVIORAL INTERPRETATION
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path

# ------------------------------------------------
# CONFIGURATION
# ------------------------------------------------

SOURCE = Path(
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

KMEANS_ASSIGNMENTS = Path(
    "/content/step6_unsupervised_ml/step6c_kmeans/"
    "kmeans_cluster_assignments.csv"
)

DBSCAN_ASSIGNMENTS = Path(
    "/content/step6_unsupervised_ml/step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

AGG_ASSIGNMENTS = Path(
    "/content/step6_unsupervised_ml/step6e_agglomerative/"
    "agglomerative_cluster_assignments.csv"
)

OUTPUT_DIR = Path(
    "/content/step7_cluster_profiling"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION_ID = "reconstructed_session_id"

# Columns that must NEVER participate in profiling as ML/behavior features
FORBIDDEN_LABEL_TERMS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "label",
    "target",
    "class",
    "y"
}

# ------------------------------------------------
# HELPERS
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fail(msg):
    raise ValueError("STOP — " + msg)


def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def check_session_ids(source_df, assignment_df, assignment_name):
    if SESSION_ID not in source_df.columns:
        fail(f"{SESSION_ID} missing from source.")

    if SESSION_ID not in assignment_df.columns:
        fail(f"{SESSION_ID} missing from {assignment_name} assignments.")

    source_ids = source_df[SESSION_ID].astype(str)
    assign_ids = assignment_df[SESSION_ID].astype(str)

    if source_ids.isna().any():
        fail(f"Missing source session IDs detected.")

    if assign_ids.isna().any():
        fail(f"Missing assignment session IDs in {assignment_name}.")

    if source_ids.duplicated().any():
        fail(f"Duplicate source session IDs detected.")

    if assign_ids.duplicated().any():
        fail(f"Duplicate assignment session IDs in {assignment_name}.")

    if len(source_ids) != len(assign_ids):
        fail(
            f"{assignment_name}: row count mismatch "
            f"({len(assign_ids)} vs {len(source_ids)})."
        )

    if set(source_ids) != set(assign_ids):
        missing = sorted(set(source_ids) - set(assign_ids))
        extra = sorted(set(assign_ids) - set(source_ids))

        fail(
            f"{assignment_name}: session IDs do not match. "
            f"Missing={len(missing)}, Extra={len(extra)}"
        )

    return True


def find_forbidden_columns(columns):
    bad = []

    for col in columns:
        normalized = str(col).strip().lower()

        if normalized in FORBIDDEN_LABEL_TERMS:
            bad.append(col)
            continue

        # Catch obvious label-derived columns without blocking
        # normal behavioral columns such as "event_count".
        for term in [
            "attack_type",
            "attack_stage",
            "mitre_technique",
            "severity"
        ]:
            if term in normalized:
                bad.append(col)

    return sorted(set(bad))


def safe_numeric_features(df):
    """
    Behavioral numeric features only.
    Removes ID and timestamps.
    Removes documented all-missing IP reputation features.
    """

    excluded = {
        SESSION_ID,
        "session_start_time",
        "session_end_time"
    }

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    feature_cols = [
        c for c in numeric_cols
        if c not in excluded
    ]

    forbidden = find_forbidden_columns(feature_cols)

    if forbidden:
        fail(
            "Forbidden label-derived columns detected in profiling: "
            + ", ".join(forbidden)
        )

    # Documented unavailable IP reputation features
    all_missing = [
        c for c in feature_cols
        if df[c].isna().all()
    ]

    usable = [
        c for c in feature_cols
        if c not in all_missing
    ]

    return usable, all_missing


def top_differences(df, cluster_col, cluster_id, feature_cols, top_n=10):
    """
    Compare cluster mean against global mean using standardized
    effect size based on global standard deviation.
    """

    cluster_data = df[df[cluster_col] == cluster_id]

    rows = []

    for feature in feature_cols:

        global_mean = df[feature].mean()
        global_std = df[feature].std()

        cluster_mean = cluster_data[feature].mean()

        if pd.isna(global_std) or global_std == 0:
            z = 0.0
        else:
            z = (cluster_mean - global_mean) / global_std

        rows.append({
            "feature": feature,
            "cluster_mean": cluster_mean,
            "global_mean": global_mean,
            "global_std": global_std,
            "standardized_difference": z,
            "absolute_difference": abs(z)
        })

    result = pd.DataFrame(rows)

    result = result.sort_values(
        "absolute_difference",
        ascending=False
    ).head(top_n)

    return result


# ------------------------------------------------
# [1] SOURCE VERIFICATION
# ------------------------------------------------

print("=" * 70)
print("STEP 7A — CLUSTER PROFILING & BEHAVIORAL INTERPRETATION")
print("=" * 70)

print("\n[1] SOURCE")

if not SOURCE.exists():
    fail(f"Source file not found: {SOURCE}")

print("Exists: PASS")
print("Path:", SOURCE)

source_sha_before = sha256_file(SOURCE)

print("SHA-256:", source_sha_before)

# ------------------------------------------------
# [2] LOAD SOURCE
# ------------------------------------------------

print("\n[2] LOADING SOURCE")

df = pd.read_csv(SOURCE)
df = normalize_columns(df)

print("Shape:", df.shape)

if len(df) != 11880:
    fail(f"Unexpected row count: {len(df)}")

print("Rows:", len(df))
print("Columns:", len(df.columns))

# ------------------------------------------------
# [3] SESSION INTEGRITY
# ------------------------------------------------

print("\n[3] SESSION INTEGRITY")

if SESSION_ID not in df.columns:
    fail(f"Missing session ID column: {SESSION_ID}")

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    fail("Missing session IDs detected.")

if duplicate_ids != 0:
    fail("Duplicate session IDs detected.")

if unique_ids != len(df):
    fail("Session IDs are not unique.")

print("Session integrity: PASS")

# ------------------------------------------------
# [4] LABEL LEAKAGE CHECK
# ------------------------------------------------

print("\n[4] LABEL LEAKAGE")

forbidden = find_forbidden_columns(df.columns)

print("Forbidden columns:", forbidden)

if forbidden:
    fail(
        "Label/target columns are present in source: "
        + ", ".join(forbidden)
    )

print("Label leakage: NO")

# ------------------------------------------------
# [5] LOAD CLUSTER ASSIGNMENTS
# ------------------------------------------------

print("\n[5] LOADING CLUSTER ASSIGNMENTS")

assignment_paths = {
    "KMeans": KMEANS_ASSIGNMENTS,
    "DBSCAN": DBSCAN_ASSIGNMENTS,
    "Agglomerative": AGG_ASSIGNMENTS
}

assignments = {}

for model_name, path in assignment_paths.items():

    print(f"\n{model_name}:")
    print("Path:", path)

    if not path.exists():
        fail(f"{model_name} assignment file missing.")

    temp = pd.read_csv(path)
    temp = normalize_columns(temp)

    print("Shape:", temp.shape)

    check_session_ids(
        df,
        temp,
        model_name
    )

    # Identify cluster column.
    candidate_cols = [
        c for c in temp.columns
        if c != SESSION_ID
    ]

    # Prefer obvious cluster columns.
    preferred = [
        c for c in candidate_cols
        if "cluster" in c.lower()
    ]

    if len(preferred) == 1:
        cluster_col = preferred[0]

    elif len(candidate_cols) == 1:
        cluster_col = candidate_cols[0]

    else:
        fail(
            f"Could not uniquely identify cluster column "
            f"for {model_name}: {candidate_cols}"
        )

    # Check labels exist.
    if temp[cluster_col].isna().any():
        fail(f"{model_name} contains missing cluster labels.")

    # Reorder by session ID and merge.
    temp = temp[[SESSION_ID, cluster_col]].copy()

    temp[SESSION_ID] = temp[SESSION_ID].astype(str)
    temp = temp.rename(
        columns={cluster_col: f"{model_name}_cluster"}
    )

    assignments[model_name] = temp

    print(
        "Cluster column:",
        f"{model_name}_cluster"
    )

    print(
        "Unique cluster labels:",
        temp[f"{model_name}_cluster"].nunique()
    )

    print("Assignment integrity: PASS")

# ------------------------------------------------
# [6] MERGE ASSIGNMENTS
# ------------------------------------------------

print("\n[6] MERGING CLUSTER ASSIGNMENTS")

profile_df = df.copy()
profile_df[SESSION_ID] = profile_df[SESSION_ID].astype(str)

for model_name, temp in assignments.items():

    profile_df = profile_df.merge(
        temp,
        on=SESSION_ID,
        how="left",
        validate="one_to_one"
    )

    cluster_col = f"{model_name}_cluster"

    if profile_df[cluster_col].isna().any():
        fail(
            f"Missing merged cluster labels for {model_name}."
        )

print("Merged rows:", len(profile_df))

if len(profile_df) != len(df):
    fail("Row count changed during assignment merge.")

print("Assignment merge: PASS")

# ------------------------------------------------
# [7] FEATURE INVENTORY
# ------------------------------------------------

print("\n[7] BEHAVIORAL FEATURE INVENTORY")

feature_cols, all_missing_features = safe_numeric_features(
    profile_df
)

print("Usable behavioral numeric features:", len(feature_cols))

print("All-missing documented features excluded:")

for col in all_missing_features:
    print(" -", col)

if len(feature_cols) == 0:
    fail("No usable behavioral numeric features found.")

print("Feature inventory: PASS")

# ------------------------------------------------
# [8] GLOBAL FEATURE STATISTICS
# ------------------------------------------------

print("\n[8] GLOBAL FEATURE STATISTICS")

global_stats = (
    profile_df[feature_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

global_stats_path = (
    OUTPUT_DIR / "step7a_global_feature_statistics.csv"
)

global_stats.to_csv(
    global_stats_path,
    index=False
)

print("Saved:", global_stats_path)

# ------------------------------------------------
# [9] MODEL CLUSTER SIZE PROFILES
# ------------------------------------------------

print("\n[9] CLUSTER SIZE PROFILES")

cluster_size_outputs = {}

for model_name in assignment_paths.keys():

    cluster_col = f"{model_name}_cluster"

    counts = (
        profile_df[cluster_col]
        .value_counts(dropna=False)
        .sort_index()
        .rename("session_count")
        .reset_index()
    )

    counts.columns = [
        "cluster",
        "session_count"
    ]

    counts["percentage"] = (
        counts["session_count"] /
        len(profile_df) * 100
    )

    counts["model"] = model_name

    counts = counts[
        [
            "model",
            "cluster",
            "session_count",
            "percentage"
        ]
    ]

    cluster_size_outputs[model_name] = counts

    print(f"\n{model_name}:")

    print(
        counts.to_string(index=False)
    )

    out_path = (
        OUTPUT_DIR /
        f"{model_name.lower()}_cluster_sizes.csv"
    )

    counts.to_csv(
        out_path,
        index=False
    )

# ------------------------------------------------
# [10] BEHAVIORAL PROFILE PER CLUSTER
# ------------------------------------------------

print("\n[10] BUILDING BEHAVIORAL PROFILES")

profile_outputs = {}

for model_name in assignment_paths.keys():

    cluster_col = f"{model_name}_cluster"

    print(
        f"\nProfiling {model_name}..."
    )

    rows = []

    cluster_ids = sorted(
        profile_df[cluster_col].unique(),
        key=lambda x: str(x)
    )

    for cluster_id in cluster_ids:

        cluster_data = profile_df[
            profile_df[cluster_col] == cluster_id
        ]

        for feature in feature_cols:

            rows.append({
                "model": model_name,
                "cluster": cluster_id,
                "session_count": len(cluster_data),
                "feature": feature,
                "mean": cluster_data[feature].mean(),
                "median": cluster_data[feature].median(),
                "std": cluster_data[feature].std(),
                "min": cluster_data[feature].min(),
                "max": cluster_data[feature].max(),
                "global_mean": profile_df[feature].mean()
            })

    profile_table = pd.DataFrame(rows)

    out_path = (
        OUTPUT_DIR /
        f"{model_name.lower()}_behavioral_profiles.csv"
    )

    profile_table.to_csv(
        out_path,
        index=False
    )

    profile_outputs[model_name] = profile_table

    print(
        "Saved:",
        out_path
    )

# ------------------------------------------------
# [11] TOP DISCRIMINATING FEATURES
# ------------------------------------------------

print("\n[11] TOP CLUSTER-DISCRIMINATING FEATURES")

top_feature_outputs = {}

for model_name in assignment_paths.keys():

    cluster_col = f"{model_name}_cluster"

    rows = []

    cluster_ids = sorted(
        profile_df[cluster_col].unique(),
        key=lambda x: str(x)
    )

    for cluster_id in cluster_ids:

        diff = top_differences(
            profile_df,
            cluster_col,
            cluster_id,
            feature_cols,
            top_n=10
        )

        diff.insert(
            0,
            "model",
            model_name
        )

        diff.insert(
            1,
            "cluster",
            cluster_id
        )

        rows.append(diff)

    top_features = pd.concat(
        rows,
        ignore_index=True
    )

    out_path = (
        OUTPUT_DIR /
        f"{model_name.lower()}_top_discriminating_features.csv"
    )

    top_features.to_csv(
        out_path,
        index=False
    )

    top_feature_outputs[model_name] = top_features

    print(
        "Saved:",
        out_path
    )

# ------------------------------------------------
# [12] HUMAN-READABLE CLUSTER SUMMARY
# ------------------------------------------------

print("\n[12] BUILDING INTERPRETATION SUMMARY")

interpretation = []

for model_name in assignment_paths.keys():

    cluster_col = f"{model_name}_cluster"

    size_table = cluster_size_outputs[model_name]
    top_table = top_feature_outputs[model_name]

    for _, size_row in size_table.iterrows():

        cluster_id = size_row["cluster"]

        top_cluster = top_table[
            top_table["cluster"] == cluster_id
        ].copy()

        top_high = (
            top_cluster
            .sort_values(
                "standardized_difference",
                ascending=False
            )
            .head(5)
        )

        top_low = (
            top_cluster
            .sort_values(
                "standardized_difference",
                ascending=True
            )
            .head(5)
        )

        interpretation.append({
            "model": model_name,
            "cluster": cluster_id,
            "session_count": int(
                size_row["session_count"]
            ),
            "percentage": float(
                size_row["percentage"]
            ),
            "top_high_features": "; ".join(
                top_high["feature"].astype(str)
            ),
            "top_low_features": "; ".join(
                top_low["feature"].astype(str)
            )
        })

interpretation_df = pd.DataFrame(
    interpretation
)

interpretation_path = (
    OUTPUT_DIR /
    "step7a_cluster_interpretation_summary.csv"
)

interpretation_df.to_csv(
    interpretation_path,
    index=False
)

print(
    "Saved:",
    interpretation_path
)

# ------------------------------------------------
# [13] AUDIT
# ------------------------------------------------

print("\n[13] FINAL AUDIT")

source_sha_after = sha256_file(SOURCE)

if source_sha_before != source_sha_after:
    fail(
        "SOURCE CHANGED DURING STEP 7A."
    )

audit = {
    "step": "7A",
    "name": "Cluster Profiling & Behavioral Interpretation",

    "source": str(SOURCE),
    "source_sha_before": source_sha_before,
    "source_sha_after": source_sha_after,
    "source_unchanged": True,

    "rows": int(len(profile_df)),
    "columns": int(len(df.columns)),
    "unique_sessions": int(
        profile_df[SESSION_ID].nunique()
    ),

    "missing_session_ids": int(
        profile_df[SESSION_ID].isna().sum()
    ),

    "duplicate_session_ids": int(
        profile_df[SESSION_ID].duplicated().sum()
    ),

    "label_leakage": False,
    "forbidden_columns": forbidden,

    "usable_behavioral_features": feature_cols,
    "all_missing_features_excluded": all_missing_features,

    "models_profiled": [
        "KMeans",
        "DBSCAN",
        "Agglomerative"
    ],

    "assignment_integrity": {
        model: True
        for model in assignment_paths.keys()
    },

    "outputs": [
        str(global_stats_path),
        str(interpretation_path)
    ]
}

audit_path = (
    OUTPUT_DIR /
    "step7a_cluster_profiling_audit.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

audit_sha = sha256_file(audit_path)

# ------------------------------------------------
# [14] FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7A — COMPLETE")
print("=" * 70)

print("\nSource:")
print(SOURCE)

print("\nSource SHA:")
print(source_sha_after)

print("\nUsable behavioral features:")
print(len(feature_cols))

print("\nModels profiled:")
print("KMeans")
print("DBSCAN")
print("Agglomerative")

print("\nSession integrity:")
print("PASS")

print("Label leakage:")
print("NO")

print("Source modified:")
print("NO")

print("\nAudit:")
print(audit_path)

print("Audit SHA-256:")
print(audit_sha)

print("\nSTATUS: PASS")
print("NEXT: STEP 7B — CROSS-MODEL BEHAVIORAL COMPARISON")

STEP 7A — CLUSTER PROFILING & BEHAVIORAL INTERPRETATION

[1] SOURCE
Exists: PASS
Path: /content/session_behavioral_features_v4/session_behavioral_features_v4.csv
SHA-256: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534

[2] LOADING SOURCE
Shape: (11880, 56)
Rows: 11880
Columns: 56

[3] SESSION INTEGRITY
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] LOADING CLUSTER ASSIGNMENTS

KMeans:
Path: /content/step6_unsupervised_ml/step6c_kmeans/kmeans_cluster_assignments.csv
Shape: (11880, 2)
Cluster column: KMeans_cluster
Unique cluster labels: 5
Assignment integrity: PASS

DBSCAN:
Path: /content/step6_unsupervised_ml/step6d_dbscan/dbscan_cluster_assignments.csv
Shape: (11880, 2)
Cluster column: DBSCAN_cluster
Unique cluster labels: 39
Assignment integrity: PASS

Agglomerative:
Path: /content/step6_unsupervised_ml/step6e_agglomerative/agglomerative_cluster_assignments.csv
Shape: (11880, 

In [ ]:
# ================================================================
# STEP 7B — CROSS-MODEL BEHAVIORAL COMPARISON
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

BASE = "/content"

SOURCE = (
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

STEP7_DIR = "/content/step7_cluster_profiling"

OUTPUT_DIR = "/content/step7_cluster_profiling/step7b_cross_model_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------
# Helper
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"STOP — missing required file: {path}")


print("=" * 70)
print("STEP 7B — CROSS-MODEL BEHAVIORAL COMPARISON")
print("=" * 70)


# ------------------------------------------------
# 1. SOURCE VERIFICATION
# ------------------------------------------------

print("\n[1] SOURCE")

require_file(SOURCE)

actual_sha = sha256_file(SOURCE)

print("Path:", SOURCE)
print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", actual_sha)

if actual_sha != EXPECTED_SOURCE_SHA:
    raise ValueError("STOP — source SHA mismatch.")

print("Verification: PASS")


# ------------------------------------------------
# 2. LOAD SOURCE
# ------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] DATASET")
print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError("STOP — unexpected session count.")

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError("STOP — session ID column missing.")

print("Rows:", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------
# 3. SESSION INTEGRITY
# ------------------------------------------------

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("\n[3] SESSION INTEGRITY")
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != 11880:
    raise ValueError("STOP — session integrity failure.")

print("Session integrity: PASS")


# ------------------------------------------------
# 4. LABEL LEAKAGE
# ------------------------------------------------

FORBIDDEN = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "label",
    "target",
    "y",
}

present_forbidden = sorted(set(df.columns) & FORBIDDEN)

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        f"STOP — forbidden label columns detected: {present_forbidden}"
    )

print("Label leakage: NO")


# ------------------------------------------------
# 5. LOAD STEP 7A OUTPUTS
# ------------------------------------------------

required_profiles = {
    "KMeans":
        "kmeans_behavioral_profiles.csv",
    "DBSCAN":
        "dbscan_behavioral_profiles.csv",
    "Agglomerative":
        "agglomerative_behavioral_profiles.csv",
}

print("\n[5] LOADING BEHAVIORAL PROFILES")

profiles = {}

for model, filename in required_profiles.items():

    path = os.path.join(STEP7_DIR, filename)

    require_file(path)

    temp = pd.read_csv(path)

    profiles[model] = temp

    print(f"{model}:")
    print("  Path:", path)
    print("  Shape:", temp.shape)


# ------------------------------------------------
# 6. LOAD TOP DISCRIMINATING FEATURES
# ------------------------------------------------

top_feature_files = {
    "KMeans":
        "kmeans_top_discriminating_features.csv",
    "DBSCAN":
        "dbscan_top_discriminating_features.csv",
    "Agglomerative":
        "agglomerative_top_discriminating_features.csv",
}

print("\n[6] TOP DISCRIMINATING FEATURES")

top_features = {}

for model, filename in top_feature_files.items():

    path = os.path.join(STEP7_DIR, filename)

    require_file(path)

    temp = pd.read_csv(path)

    top_features[model] = temp

    print(
        f"{model}: {len(temp)} rows"
    )


# ------------------------------------------------
# 7. CLUSTER SIZE COMPARISON
# ------------------------------------------------

print("\n[7] CLUSTER SIZE COMPARISON")

size_tables = []

for model in ["KMeans", "DBSCAN", "Agglomerative"]:

    path = os.path.join(STEP7_DIR, required_profiles[model])

    temp = pd.read_csv(path)

    # Try to identify cluster/size columns robustly
    cluster_col = next(
        (
            c for c in temp.columns
            if c.lower() in {
                "cluster",
                f"{model.lower()}_cluster"
            }
        ),
        None
    )

    count_col = next(
        (
            c for c in temp.columns
            if c.lower() in {
                "session_count",
                "count",
                "n_sessions"
            }
        ),
        None
    )

    if cluster_col is not None and count_col is not None:

        for _, row in temp[[cluster_col, count_col]].drop_duplicates().iterrows():

            size_tables.append({
                "model": model,
                "cluster": row[cluster_col],
                "session_count": int(row[count_col])
            })

size_df = pd.DataFrame(size_tables)

if not size_df.empty:

    size_df["percentage"] = (
        size_df["session_count"] / len(df) * 100
    )

    size_df["percentage"] = size_df["percentage"].round(4)

    size_path = os.path.join(
        OUTPUT_DIR,
        "step7b_cluster_size_comparison.csv"
    )

    size_df.to_csv(size_path, index=False)

    print("Saved:", size_path)


# ------------------------------------------------
# 8. MODEL-LEVEL STRUCTURAL SUMMARY
# ------------------------------------------------

print("\n[8] MODEL STRUCTURAL SUMMARY")

summary_rows = []

for model in ["KMeans", "DBSCAN", "Agglomerative"]:

    if size_df.empty:
        continue

    m = size_df[size_df["model"] == model].copy()

    if m.empty:
        continue

    cluster_count = len(m)

    largest_cluster = int(m["session_count"].max())
    smallest_cluster = int(m["session_count"].min())

    largest_fraction = (
        largest_cluster / len(df)
    )

    singleton_count = int(
        (m["session_count"] == 1).sum()
    )

    small_cluster_count = int(
        (m["session_count"] < 50).sum()
    )

    summary_rows.append({
        "model": model,
        "clusters_found": cluster_count,
        "largest_cluster_size": largest_cluster,
        "largest_cluster_fraction": round(
            largest_fraction, 6
        ),
        "smallest_cluster_size": smallest_cluster,
        "singleton_clusters": singleton_count,
        "clusters_under_50_sessions":
            small_cluster_count,
    })


model_structure = pd.DataFrame(summary_rows)

structure_path = os.path.join(
    OUTPUT_DIR,
    "step7b_model_structural_summary.csv"
)

model_structure.to_csv(
    structure_path,
    index=False
)

print(model_structure.to_string(index=False))
print("Saved:", structure_path)


# ------------------------------------------------
# 9. TOP BEHAVIORAL FEATURES PER MODEL
# ------------------------------------------------

print("\n[9] BEHAVIORAL DISCRIMINATION SUMMARY")

feature_summary_rows = []

for model, temp in top_features.items():

    if temp.empty:
        continue

    # Keep first 10 rows as the highest-ranked
    # discriminating features generated in 7A.
    top10 = temp.head(10)

    for rank, (_, row) in enumerate(
        top10.iterrows(),
        start=1
    ):

        feature_name = None

        for c in [
            "feature",
            "feature_name",
            "Feature",
            "column"
        ]:

            if c in row.index:
                feature_name = row[c]
                break

        if feature_name is None:
            continue

        feature_summary_rows.append({
            "model": model,
            "rank": rank,
            "feature": feature_name
        })


feature_summary = pd.DataFrame(
    feature_summary_rows
)

feature_summary_path = os.path.join(
    OUTPUT_DIR,
    "step7b_top_behavioral_features_summary.csv"
)

feature_summary.to_csv(
    feature_summary_path,
    index=False
)

print("Saved:", feature_summary_path)


# ------------------------------------------------
# 10. CROSS-MODEL FEATURE OVERLAP
# ------------------------------------------------

print("\n[10] CROSS-MODEL FEATURE OVERLAP")

model_feature_sets = {}

for model, temp in top_features.items():

    features = []

    for c in [
        "feature",
        "feature_name",
        "Feature",
        "column"
    ]:

        if c in temp.columns:
            features = (
                temp[c]
                .dropna()
                .astype(str)
                .head(10)
                .tolist()
            )
            break

    model_feature_sets[model] = set(features)


overlap_rows = []

models = list(model_feature_sets.keys())

for i in range(len(models)):

    for j in range(i + 1, len(models)):

        a = models[i]
        b = models[j]

        common = sorted(
            model_feature_sets[a]
            &
            model_feature_sets[b]
        )

        union = (
            model_feature_sets[a]
            |
            model_feature_sets[b]
        )

        jaccard = (
            len(common) / len(union)
            if union else 0
        )

        overlap_rows.append({
            "model_a": a,
            "model_b": b,
            "common_top_features": len(common),
            "jaccard_similarity": round(
                jaccard, 6
            ),
            "shared_features": ";".join(common)
        })


overlap_df = pd.DataFrame(overlap_rows)

overlap_path = os.path.join(
    OUTPUT_DIR,
    "step7b_cross_model_feature_overlap.csv"
)

overlap_df.to_csv(
    overlap_path,
    index=False
)

print(overlap_df.to_string(index=False))
print("Saved:", overlap_path)


# ------------------------------------------------
# 11. AUTOMATIC INTERPRETATION FLAGS
# ------------------------------------------------

print("\n[11] INTERPRETATION FLAGS")

flags = []

for row in summary_rows:

    model = row["model"]

    if row["largest_cluster_fraction"] > 0.75:

        flags.append({
            "model": model,
            "flag": "HIGH_DOMINANCE",
            "description":
                "One cluster contains more than 75% of sessions."
        })

    if row["singleton_clusters"] > 0:

        flags.append({
            "model": model,
            "flag": "SINGLETON_CLUSTERS",
            "description":
                "One or more clusters contain exactly one session."
        })

    if row["clusters_under_50_sessions"] > 5:

        flags.append({
            "model": model,
            "flag": "MANY_SMALL_CLUSTERS",
            "description":
                "Many clusters contain fewer than 50 sessions."
        })


flags_df = pd.DataFrame(flags)

flags_path = os.path.join(
    OUTPUT_DIR,
    "step7b_interpretation_flags.csv"
)

flags_df.to_csv(
    flags_path,
    index=False
)

print(flags_df.to_string(index=False))
print("Saved:", flags_path)


# ------------------------------------------------
# 12. AUDIT
# ------------------------------------------------

audit = {
    "step": "7B",
    "name": "Cross-Model Behavioral Comparison",
    "source": SOURCE,
    "source_sha256": actual_sha,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "models_compared": [
        "KMeans",
        "DBSCAN",
        "Agglomerative"
    ],
    "labels_used": False,
    "label_leakage": False,
    "session_integrity": True,
    "source_modified": False,
    "outputs": [
        os.path.basename(size_path)
            if not size_df.empty else None,
        os.path.basename(structure_path),
        os.path.basename(feature_summary_path),
        os.path.basename(overlap_path),
        os.path.basename(flags_path),
    ],
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7b_cross_model_comparison_audit.json"
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[12] AUDIT")
print("Saved:", audit_path)


# ------------------------------------------------
# FINAL
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7B — COMPLETE")
print("=" * 70)

print("Session integrity: PASS")
print("Label leakage: NO")
print("Source modified: NO")
print("Models compared: KMeans / DBSCAN / Agglomerative")

print("\nOutputs:")
print(OUTPUT_DIR)

print("\nSTATUS: PASS")
print("NEXT: STEP 7C — MODEL SELECTION + BEHAVIORAL CLUSTER LABELING")

STEP 7B — CROSS-MODEL BEHAVIORAL COMPARISON

[1] SOURCE
Path: /content/session_behavioral_features_v4/session_behavioral_features_v4.csv
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] DATASET
Shape: (11880, 56)
Rows: 11880
Columns: 56

[3] SESSION INTEGRITY
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] LOADING BEHAVIORAL PROFILES
KMeans:
  Path: /content/step7_cluster_profiling/kmeans_behavioral_profiles.csv
  Shape: (255, 10)
DBSCAN:
  Path: /content/step7_cluster_profiling/dbscan_behavioral_profiles.csv
  Shape: (1989, 10)
Agglomerative:
  Path: /content/step7_cluster_profiling/agglomerative_behavioral_profiles.csv
  Shape: (306, 10)

[6] TOP DISCRIMINATING FEATURES
KMeans: 50 rows
DBSCAN: 390 rows
Agglomerative: 60 rows

[7] CLUSTER SIZE COMPARISON
Saved: /con

In [ ]:
# ================================================================
# STEP 7B-R — CORRECTED CROSS-MODEL BEHAVIORAL COMPARISON
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

SOURCE = (
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

STEP7_DIR = "/content/step7_cluster_profiling"

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7b_corrected_cross_model_comparison"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ================================================================
# HELPERS
# ================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"STOP — required file missing:\n{path}"
        )


def find_feature_column(df):
    candidates = [
        "feature",
        "feature_name",
        "Feature",
        "column",
        "feature_column",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return None


# ================================================================
# STEP 1 — SOURCE VERIFICATION
# ================================================================

print("=" * 70)
print("STEP 7B-R — CORRECTED CROSS-MODEL BEHAVIORAL COMPARISON")
print("=" * 70)

print("\n[1] SOURCE VERIFICATION")

require_file(SOURCE)

actual_source_sha = sha256_file(SOURCE)

print("Path:", SOURCE)
print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source SHA mismatch. "
        "Original dataset must not be analyzed."
    )

print("Verification: PASS")


# ================================================================
# STEP 2 — LOAD SOURCE
# ================================================================

print("\n[2] DATASET")

df = pd.read_csv(SOURCE)

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if df.shape != (11880, 56):
    raise ValueError(
        f"STOP — unexpected source shape: {df.shape}"
    )


# ================================================================
# STEP 3 — SESSION INTEGRITY
# ================================================================

print("\n[3] SESSION INTEGRITY")

SESSION_ID = "reconstructed_session_id"

if SESSION_ID not in df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

missing_ids = int(df[SESSION_ID].isna().sum())
duplicate_ids = int(df[SESSION_ID].duplicated().sum())
unique_ids = int(df[SESSION_ID].nunique())

print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if (
    missing_ids != 0
    or duplicate_ids != 0
    or unique_ids != 11880
):
    raise ValueError(
        "STOP — session integrity failure."
    )

print("Session integrity: PASS")


# ================================================================
# STEP 4 — LABEL LEAKAGE
# ================================================================

print("\n[4] LABEL LEAKAGE")

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "label",
    "target",
    "y",
}

present_forbidden = sorted(
    set(df.columns) & FORBIDDEN_LABELS
)

print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        f"STOP — forbidden label columns found: "
        f"{present_forbidden}"
    )

print("Label leakage: NO")


# ================================================================
# STEP 5 — DEFINE STRICTLY BEHAVIORAL FEATURES
# ================================================================

print("\n[5] STRICT BEHAVIORAL FEATURE FILTER")

# These columns are NEVER behavioral features for this analysis.
# They are identifiers, timestamps, labels, or clustering outputs.

EXCLUDED_DERIVED_COLUMNS = {
    SESSION_ID,

    # Raw time columns
    "session_start_time",
    "session_end_time",

    # Existing clustering outputs
    "KMeans_cluster",
    "DBSCAN_cluster",
    "Agglomerative_cluster",

    # Generic derived cluster/label names
    "cluster",
    "cluster_id",
    "cluster_label",
    "model_cluster",

    # Possible label-like columns
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "label",
    "target",
    "y",
}

# Any column containing cluster as a standalone semantic component
# is excluded as a defensive measure.
cluster_like_columns = {
    c for c in df.columns
    if "cluster" in c.lower()
}

# Any obvious label-derived column is excluded too.
label_like_columns = {
    c for c in df.columns
    if any(
        token in c.lower()
        for token in [
            "attack_type",
            "attack_stage",
            "severity",
            "mitre",
            "label",
            "target"
        ]
    )
}

excluded_columns = (
    EXCLUDED_DERIVED_COLUMNS
    |
    cluster_like_columns
    |
    label_like_columns
)

excluded_columns = sorted(
    c for c in excluded_columns
    if c in df.columns
)

behavioral_candidates = [
    c for c in df.columns
    if c not in excluded_columns
]

behavioral_numeric_features = [
    c for c in behavioral_candidates
    if pd.api.types.is_numeric_dtype(df[c])
]

print("Excluded derived/ID/label columns:")
for c in excluded_columns:
    print(" -", c)

print(
    "\nBehavioral numeric features:",
    len(behavioral_numeric_features)
)

if len(behavioral_numeric_features) == 0:
    raise ValueError(
        "STOP — no behavioral numeric features remain."
    )

print("Behavioral feature filter: PASS")


# ================================================================
# STEP 6 — VERIFY NO CLUSTER OUTPUTS REMAIN
# ================================================================

print("\n[6] CLUSTER-DERIVED FEATURE SAFETY CHECK")

remaining_cluster_columns = [
    c for c in behavioral_numeric_features
    if "cluster" in c.lower()
]

if remaining_cluster_columns:
    raise ValueError(
        "STOP — cluster-derived columns remain in "
        f"behavioral feature set: {remaining_cluster_columns}"
    )

remaining_label_columns = [
    c for c in behavioral_numeric_features
    if any(
        token in c.lower()
        for token in [
            "attack_type",
            "attack_stage",
            "severity",
            "mitre",
            "label",
            "target"
        ]
    )
]

if remaining_label_columns:
    raise ValueError(
        "STOP — label-derived columns remain: "
        f"{remaining_label_columns}"
    )

print("Cluster-derived features remaining: 0")
print("Label-derived features remaining: 0")
print("Feature safety: PASS")


# ================================================================
# STEP 7 — FEATURE MISSINGNESS CHECK
# ================================================================

print("\n[7] BEHAVIORAL FEATURE MISSINGNESS")

missing_report = []

for c in behavioral_numeric_features:

    missing_count = int(df[c].isna().sum())

    missing_report.append({
        "feature": c,
        "missing_count": missing_count,
        "missing_fraction": round(
            missing_count / len(df),
            8
        )
    })

missing_df = pd.DataFrame(missing_report)

missing_path = os.path.join(
    OUTPUT_DIR,
    "step7b_behavioral_feature_missingness.csv"
)

missing_df.to_csv(
    missing_path,
    index=False
)

print(
    "Features with missing values:",
    int((missing_df["missing_count"] > 0).sum())
)

print("Saved:", missing_path)


# ================================================================
# STEP 8 — LOAD 7A TOP FEATURE FILES
# ================================================================

print("\n[8] LOADING STEP 7A TOP FEATURES")

top_feature_files = {
    "KMeans":
        "kmeans_top_discriminating_features.csv",

    "DBSCAN":
        "dbscan_top_discriminating_features.csv",

    "Agglomerative":
        "agglomerative_top_discriminating_features.csv",
}

top_features = {}

for model, filename in top_feature_files.items():

    path = os.path.join(
        STEP7_DIR,
        filename
    )

    require_file(path)

    temp = pd.read_csv(path)

    feature_col = find_feature_column(temp)

    if feature_col is None:
        raise ValueError(
            f"STOP — could not identify feature column "
            f"in {filename}."
        )

    # Keep ONLY actual behavioral features.
    temp = temp[
        temp[feature_col]
        .astype(str)
        .isin(behavioral_numeric_features)
    ].copy()

    top_features[model] = temp

    print(
        f"{model}: {len(temp)} valid behavioral feature rows"
    )


# ================================================================
# STEP 9 — TOP BEHAVIORAL FEATURES SUMMARY
# ================================================================

print("\n[9] CLEAN TOP BEHAVIORAL FEATURES")

summary_rows = []

for model, temp in top_features.items():

    feature_col = find_feature_column(temp)

    # Remove duplicates while preserving ranking order.
    temp = temp.drop_duplicates(
        subset=[feature_col]
    )

    temp = temp.head(10)

    for rank, (_, row) in enumerate(
        temp.iterrows(),
        start=1
    ):

        summary_rows.append({
            "model": model,
            "rank": rank,
            "feature": str(row[feature_col])
        })


feature_summary = pd.DataFrame(
    summary_rows
)

feature_summary_path = os.path.join(
    OUTPUT_DIR,
    "step7b_clean_top_behavioral_features.csv"
)

feature_summary.to_csv(
    feature_summary_path,
    index=False
)

print(
    feature_summary.to_string(index=False)
)

print("Saved:", feature_summary_path)


# ================================================================
# STEP 10 — CROSS-MODEL OVERLAP
# ================================================================

print("\n[10] CLEAN CROSS-MODEL FEATURE OVERLAP")

feature_sets = {}

for model, temp in top_features.items():

    feature_col = find_feature_column(temp)

    feature_sets[model] = set(
        temp[feature_col]
        .astype(str)
        .head(10)
        .tolist()
    )


models = [
    "KMeans",
    "DBSCAN",
    "Agglomerative"
]

overlap_rows = []

for i in range(len(models)):

    for j in range(i + 1, len(models)):

        a = models[i]
        b = models[j]

        set_a = feature_sets.get(a, set())
        set_b = feature_sets.get(b, set())

        common = sorted(set_a & set_b)
        union = set_a | set_b

        jaccard = (
            len(common) / len(union)
            if union
            else 0.0
        )

        overlap_rows.append({
            "model_a": a,
            "model_b": b,
            "common_top_behavioral_features":
                len(common),
            "jaccard_similarity":
                round(jaccard, 6),
            "shared_behavioral_features":
                ";".join(common)
        })


overlap_df = pd.DataFrame(
    overlap_rows
)

overlap_path = os.path.join(
    OUTPUT_DIR,
    "step7b_clean_cross_model_feature_overlap.csv"
)

overlap_df.to_csv(
    overlap_path,
    index=False
)

print(
    overlap_df.to_string(index=False)
)

print("Saved:", overlap_path)


# ================================================================
# STEP 11 — VERIFY OVERLAP IS PURELY BEHAVIORAL
# ================================================================

print("\n[11] OVERLAP PURITY CHECK")

all_shared_features = set()

for value in overlap_df[
    "shared_behavioral_features"
].fillna(""):

    if value:
        all_shared_features.update(
            value.split(";")
        )

invalid_shared = sorted(
    all_shared_features
    -
    set(behavioral_numeric_features)
)

print(
    "Shared features:",
    len(all_shared_features)
)

print(
    "Invalid / derived shared features:",
    invalid_shared
)

if invalid_shared:
    raise ValueError(
        "STOP — invalid derived columns detected "
        "in cross-model overlap."
    )

print("Overlap purity: PASS")


# ================================================================
# STEP 12 — MODEL STRUCTURE FROM EXISTING 7A PROFILES
# ================================================================

print("\n[12] MODEL STRUCTURAL SUMMARY")

profile_files = {
    "KMeans":
        "kmeans_behavioral_profiles.csv",

    "DBSCAN":
        "dbscan_behavioral_profiles.csv",

    "Agglomerative":
        "agglomerative_behavioral_profiles.csv",
}

structure_rows = []

for model, filename in profile_files.items():

    path = os.path.join(
        STEP7_DIR,
        filename
    )

    require_file(path)

    temp = pd.read_csv(path)

    # Find cluster column
    cluster_col = next(
        (
            c for c in temp.columns
            if c.lower() == "cluster"
        ),
        None
    )

    # Find count column
    count_col = next(
        (
            c for c in temp.columns
            if c.lower() in {
                "session_count",
                "count",
                "n_sessions"
            }
        ),
        None
    )

    if cluster_col is None or count_col is None:
        raise ValueError(
            f"STOP — unable to identify cluster/count "
            f"columns in {filename}."
        )

    counts = pd.to_numeric(
        temp[count_col],
        errors="coerce"
    ).dropna()

    if counts.empty:
        raise ValueError(
            f"STOP — no valid session counts in {filename}."
        )

    structure_rows.append({
        "model": model,
        "clusters_found": int(len(counts)),
        "largest_cluster_size":
            int(counts.max()),
        "largest_cluster_fraction":
            round(
                float(counts.max()) / len(df),
                6
            ),
        "smallest_cluster_size":
            int(counts.min()),
        "singleton_clusters":
            int((counts == 1).sum()),
        "clusters_under_50_sessions":
            int((counts < 50).sum())
    })


structure_df = pd.DataFrame(
    structure_rows
)

structure_path = os.path.join(
    OUTPUT_DIR,
    "step7b_clean_model_structural_summary.csv"
)

structure_df.to_csv(
    structure_path,
    index=False
)

print(
    structure_df.to_string(index=False)
)

print("Saved:", structure_path)


# ================================================================
# STEP 13 — FINAL AUDIT
# ================================================================

print("\n[13] FINAL AUDIT")

audit = {
    "step": "7B-R",
    "name":
        "Corrected Cross-Model Behavioral Comparison",

    "source": SOURCE,
    "source_sha256": actual_source_sha,

    "rows": int(len(df)),
    "columns": int(len(df.columns)),

    "session_integrity": {
        "missing_ids": missing_ids,
        "duplicate_ids": duplicate_ids,
        "unique_ids": unique_ids,
        "pass": True
    },

    "label_leakage": {
        "forbidden_columns": present_forbidden,
        "pass": True
    },

    "behavioral_feature_filter": {
        "behavioral_numeric_features":
            len(behavioral_numeric_features),

        "excluded_columns":
            excluded_columns,

        "cluster_derived_remaining":
            remaining_cluster_columns,

        "label_derived_remaining":
            remaining_label_columns,

        "pass": True
    },

    "cross_model_overlap": {
        "invalid_shared_features":
            invalid_shared,
        "pass": True
    },

    "source_modified": False,

    "outputs": [
        os.path.basename(missing_path),
        os.path.basename(feature_summary_path),
        os.path.basename(overlap_path),
        os.path.basename(structure_path),
    ]
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7b_corrected_audit.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# FINAL SOURCE PROTECTION
# ================================================================

print("\n[14] SOURCE PROTECTION")

final_source_sha = sha256_file(SOURCE)

print("SHA before:", actual_source_sha)
print("SHA after: ", final_source_sha)

if final_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source file changed."
    )

print("Source unchanged: PASS")


# ================================================================
# COMPLETE
# ================================================================

print("\n" + "=" * 70)
print("STEP 7B-R — COMPLETE")
print("=" * 70)

print("Behavioral features only: PASS")
print("Cluster-derived features excluded: PASS")
print("Label-derived features excluded: PASS")
print("Cross-model overlap purity: PASS")
print("Session integrity: PASS")
print("Label leakage: NO")
print("Source modified: NO")

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nAudit:")
print(audit_path)

print("\nSTATUS: PASS")
print("NEXT: STEP 7C — MODEL SELECTION + BEHAVIORAL CLUSTER LABELING")

STEP 7B-R — CORRECTED CROSS-MODEL BEHAVIORAL COMPARISON

[1] SOURCE VERIFICATION
Path: /content/session_behavioral_features_v4/session_behavioral_features_v4.csv
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] DATASET
Shape: (11880, 56)
Rows: 11880
Columns: 56

[3] SESSION INTEGRITY
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] STRICT BEHAVIORAL FEATURE FILTER
Excluded derived/ID/label columns:
 - reconstructed_session_id
 - session_end_time
 - session_start_time

Behavioral numeric features: 53
Behavioral feature filter: PASS

[6] CLUSTER-DERIVED FEATURE SAFETY CHECK
Cluster-derived features remaining: 0
Label-derived features remaining: 0
Feature safety: PASS

[7] BEHAVIORAL FEATURE MISSINGNESS
Features with missing values: 5
Saved: /content/step7_cluster_profil

In [ ]:
# ================================================================
# STEP 7C — MODEL SELECTION + BEHAVIORAL CLUSTER LABELING
# ROBUST / VERIFIED VERSION
# ================================================================

import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("STEP 7C — MODEL SELECTION + BEHAVIORAL CLUSTER LABELING")
print("=" * 70)

# ------------------------------------------------
# CONFIGURATION
# ------------------------------------------------

SOURCE = Path(
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

ASSIGNMENT_FILES = {
    "KMeans": Path(
        "/content/step6_unsupervised_ml/step6c_kmeans/"
        "kmeans_cluster_assignments.csv"
    ),
    "DBSCAN": Path(
        "/content/step6_unsupervised_ml/step6d_dbscan/"
        "dbscan_cluster_assignments.csv"
    ),
    "Agglomerative": Path(
        "/content/step6_unsupervised_ml/step6e_agglomerative/"
        "agglomerative_cluster_assignments.csv"
    ),
}

EXPECTED_CLUSTER_COLUMNS = {
    "KMeans": "KMeans_cluster",
    "DBSCAN": "DBSCAN_cluster",
    "Agglomerative": "Agglomerative_cluster",
}

OUTPUT_DIR = Path(
    "/content/step7_cluster_profiling/"
    "step7c_model_selection_behavioral_labeling"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------
# SHA256 HELPER
# ------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------
# [1] SOURCE VERIFICATION
# ------------------------------------------------

print("\n[1] SOURCE VERIFICATION")

if not SOURCE.exists():
    raise FileNotFoundError(f"STOP — source missing: {SOURCE}")

actual_source_sha = sha256_file(SOURCE)

print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError("STOP — source SHA mismatch.")

print("Verification: PASS")


# ------------------------------------------------
# [2] LOAD SOURCE
# ------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, found {len(df)}"
    )


# ------------------------------------------------
# [3] SESSION ID
# ------------------------------------------------

ID_COL = "reconstructed_session_id"

if ID_COL not in df.columns:
    raise ValueError(f"STOP — missing session ID column: {ID_COL}")

missing_ids = int(df[ID_COL].isna().sum())
duplicate_ids = int(df[ID_COL].duplicated().sum())
unique_ids = int(df[ID_COL].nunique())

print("\n[3] SESSION INTEGRITY")
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != len(df):
    raise ValueError("STOP — session integrity failure.")

print("Session integrity: PASS")


# ------------------------------------------------
# [4] LABEL LEAKAGE
# ------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
}

present_forbidden = [
    c for c in df.columns
    if c.lower() in FORBIDDEN_LABELS
]

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        f"STOP — forbidden label columns present: {present_forbidden}"
    )

print("Label leakage: NO")


# ------------------------------------------------
# [5] LOAD ASSIGNMENTS — ROBUSTLY
# ------------------------------------------------

print("\n[5] LOADING MODEL ASSIGNMENTS")

assignments = {}

for model, path in ASSIGNMENT_FILES.items():

    print(f"\n{model}")
    print("Path:", path)

    if not path.exists():
        raise FileNotFoundError(
            f"STOP — assignment file missing for {model}: {path}"
        )

    temp = pd.read_csv(path)

    print("Shape:", temp.shape)
    print("Columns:", list(temp.columns))

    # ---- Find ID column ----

    possible_id_columns = [
        ID_COL,
        "session_id",
        "reconstructed_session_id"
    ]

    found_id = None

    for c in possible_id_columns:
        if c in temp.columns:
            found_id = c
            break

    if found_id is None:
        raise ValueError(
            f"STOP — no session ID column found in {model} assignments."
        )

    # ---- Find cluster column ----

    expected_cluster_col = EXPECTED_CLUSTER_COLUMNS[model]

    if expected_cluster_col in temp.columns:
        cluster_col = expected_cluster_col

    else:
        # Robust fallback:
        # find a column containing "cluster"
        cluster_candidates = [
            c for c in temp.columns
            if "cluster" in c.lower()
        ]

        if len(cluster_candidates) != 1:
            raise ValueError(
                f"STOP — could not uniquely identify cluster column "
                f"for {model}.\n"
                f"Expected: {expected_cluster_col}\n"
                f"Found candidates: {cluster_candidates}\n"
                f"All columns: {list(temp.columns)}"
            )

        cluster_col = cluster_candidates[0]

    print("Detected ID column:", found_id)
    print("Detected cluster column:", cluster_col)

    # ---- Rename consistently ----

    temp = temp[[found_id, cluster_col]].copy()

    temp = temp.rename(
        columns={
            found_id: ID_COL,
            cluster_col: f"{model}_cluster"
        }
    )

    # ---- Assignment integrity ----

    if temp[ID_COL].isna().any():
        raise ValueError(
            f"STOP — missing session IDs in {model} assignments."
        )

    if temp[ID_COL].duplicated().any():
        raise ValueError(
            f"STOP — duplicate session IDs in {model} assignments."
        )

    if len(temp) != len(df):
        raise ValueError(
            f"STOP — {model} assignment row count mismatch.\n"
            f"Expected: {len(df)}\n"
            f"Found: {len(temp)}"
        )

    if set(temp[ID_COL]) != set(df[ID_COL]):
        raise ValueError(
            f"STOP — session IDs do not match source for {model}."
        )

    if temp[f"{model}_cluster"].isna().any():
        raise ValueError(
            f"STOP — missing cluster labels in {model}."
        )

    assignments[model] = temp

    print(
        "Clusters found:",
        temp[f"{model}_cluster"].nunique()
    )

    print("Assignment integrity: PASS")


# ------------------------------------------------
# [6] MERGE VERIFIED ASSIGNMENTS
# ------------------------------------------------

print("\n[6] MERGING VERIFIED ASSIGNMENTS")

cluster_df = df[[ID_COL]].copy()

for model in assignments:

    cluster_df = cluster_df.merge(
        assignments[model],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )

print("Merged rows:", len(cluster_df))

for model in assignments:

    col = f"{model}_cluster"

    missing = int(cluster_df[col].isna().sum())

    if missing != 0:
        raise ValueError(
            f"STOP — {missing} missing assignments in {model}"
        )

print("Assignment merge: PASS")


# ------------------------------------------------
# [7] MODEL METRICS FROM STEP 6F
# ------------------------------------------------

print("\n[7] LOADING MODEL COMPARISON")

MODEL_COMPARISON = Path(
    "/content/step6_unsupervised_ml/"
    "step6f_final_audit/"
    "step6f_model_comparison.csv"
)

if not MODEL_COMPARISON.exists():
    raise FileNotFoundError(
        f"STOP — model comparison missing: {MODEL_COMPARISON}"
    )

comparison = pd.read_csv(MODEL_COMPARISON)

print(comparison.to_string(index=False))


# ------------------------------------------------
# [8] MODEL SELECTION
# ------------------------------------------------

print("\n[8] MODEL SELECTION")

# We do NOT select purely by one metric.
# DBSCAN has a much higher silhouette but also 14.1% noise
# and 38 clusters. KMeans/Agglomerative are much more compact.

# Primary interpretation:
# - DBSCAN: strongest separation + explicit anomaly/noise capability
# - Agglomerative: compact interpretable alternative
# - KMeans: simple baseline

model_decision = pd.DataFrame([
    {
        "model": "DBSCAN",
        "role": "Primary",
        "reason": (
            "Best separation metrics and explicit noise/anomaly "
            "identification; 14.11% sessions marked as noise."
        )
    },
    {
        "model": "Agglomerative",
        "role": "Secondary",
        "reason": (
            "Strong compact clustering and similar structure to KMeans; "
            "useful for comparison and interpretation."
        )
    },
    {
        "model": "KMeans",
        "role": "Baseline",
        "reason": (
            "Simple reproducible baseline, but highly dominated by "
            "one large cluster."
        )
    }
])

print(model_decision.to_string(index=False))


# ------------------------------------------------
# [9] BEHAVIORAL CLUSTER LABELING
# ------------------------------------------------

print("\n[9] BUILDING BEHAVIORAL CLUSTER LABELS")

# IMPORTANT:
# These are behavior-based names, NOT attack labels.
# We deliberately do not use attack_type/severity/etc.

behavior_labels = {
    "KMeans": {
        0: "Dominant Baseline Behavior",
        1: "Rare Behavioral Outlier",
        2: "Distinct Behavioral Group",
        3: "Secondary Behavioral Pattern",
        4: "Extreme Behavioral Outlier",
    },

    "Agglomerative": {
        0: "Secondary Behavioral Pattern",
        1: "Distinct Behavioral Group",
        2: "Dominant Baseline Behavior",
        3: "Rare Behavioral Group",
        4: "Extreme Behavioral Outlier",
        5: "Extreme Behavioral Outlier",
    }
}

# DBSCAN labels are generated dynamically.
# Noise = anomalous / isolated behavior, not automatically "attack".

dbscan_clusters = sorted(
    cluster_df["DBSCAN_cluster"].unique()
)

dbscan_labels = {}

for cluster in dbscan_clusters:

    if cluster == -1:
        dbscan_labels[cluster] = (
            "DBSCAN Noise / Isolated Behavior"
        )

    else:
        count = int(
            (cluster_df["DBSCAN_cluster"] == cluster).sum()
        )

        if count >= 500:
            label = "Dense Recurrent Behavioral Pattern"
        elif count >= 200:
            label = "Established Behavioral Pattern"
        elif count >= 50:
            label = "Distinct Behavioral Pattern"
        else:
            label = "Rare Behavioral Pattern"

        dbscan_labels[cluster] = label


# ------------------------------------------------
# [10] CREATE CLUSTER LABEL TABLES
# ------------------------------------------------

print("\n[10] SAVING CLUSTER LABELS")

label_tables = {}

for model in ["KMeans", "DBSCAN", "Agglomerative"]:

    cluster_col = f"{model}_cluster"

    counts = (
        cluster_df.groupby(cluster_col)
        .size()
        .reset_index(name="session_count")
    )

    counts["percentage"] = (
        counts["session_count"] / len(df) * 100
    )

    if model == "DBSCAN":
        counts["behavior_label"] = counts[cluster_col].map(
            dbscan_labels
        )
    else:
        counts["behavior_label"] = counts[cluster_col].map(
            behavior_labels[model]
        )

    counts["model"] = model

    counts = counts[
        [
            "model",
            cluster_col,
            "session_count",
            "percentage",
            "behavior_label"
        ]
    ]

    out = OUTPUT_DIR / f"{model.lower()}_behavioral_cluster_labels.csv"

    counts.to_csv(out, index=False)

    label_tables[model] = counts

    print("Saved:", out)


# ------------------------------------------------
# [11] SAVE SESSION-LEVEL MODEL OUTPUT
# ------------------------------------------------

session_output = cluster_df.copy()

session_output["DBSCAN_behavior_label"] = (
    session_output["DBSCAN_cluster"]
    .map(dbscan_labels)
)

session_output["KMeans_behavior_label"] = (
    session_output["KMeans_cluster"]
    .map(behavior_labels["KMeans"])
)

session_output["Agglomerative_behavior_label"] = (
    session_output["Agglomerative_cluster"]
    .map(behavior_labels["Agglomerative"])
)

session_output_path = (
    OUTPUT_DIR / "session_behavioral_cluster_assignments.csv"
)

session_output.to_csv(
    session_output_path,
    index=False
)

print("Saved:", session_output_path)


# ------------------------------------------------
# [12] AUDIT
# ------------------------------------------------

audit = {
    "step": "7C",
    "status": "PASS",
    "source": str(SOURCE),
    "source_sha256": actual_source_sha,
    "rows": int(len(df)),
    "unique_sessions": int(df[ID_COL].nunique()),
    "label_leakage": False,
    "models": {
        "KMeans": {
            "assignment_file": str(ASSIGNMENT_FILES["KMeans"]),
            "clusters": int(
                cluster_df["KMeans_cluster"].nunique()
            )
        },
        "DBSCAN": {
            "assignment_file": str(ASSIGNMENT_FILES["DBSCAN"]),
            "clusters": int(
                cluster_df["DBSCAN_cluster"].nunique()
            ),
            "noise_sessions": int(
                (cluster_df["DBSCAN_cluster"] == -1).sum()
            )
        },
        "Agglomerative": {
            "assignment_file": str(
                ASSIGNMENT_FILES["Agglomerative"]
            ),
            "clusters": int(
                cluster_df["Agglomerative_cluster"].nunique()
            )
        }
    },
    "selected_primary_model": "DBSCAN",
    "secondary_model": "Agglomerative",
    "baseline_model": "KMeans",
    "source_modified": False,
}

audit_path = OUTPUT_DIR / "step7c_model_selection_audit.json"

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)


# ------------------------------------------------
# [13] FINAL SOURCE PROTECTION
# ------------------------------------------------

final_source_sha = sha256_file(SOURCE)

if final_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source changed during Step 7C."
    )

print("\n[13] SOURCE PROTECTION")
print("SHA before:", actual_source_sha)
print("SHA after: ", final_source_sha)
print("Source unchanged: PASS")


# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7C — COMPLETE")
print("=" * 70)

print("\nPrimary model: DBSCAN")
print("Secondary model: Agglomerative")
print("Baseline: KMeans")

print("\nReason:")
print(
    "DBSCAN provides the strongest separation metrics and explicitly "
    "identifies isolated/noise sessions without using labels."
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nAudit:")
print(audit_path)

print("\nSTATUS: PASS")
print("NEXT: STEP 7D — DETAILED BEHAVIORAL CLUSTER PROFILING")

STEP 7C — MODEL SELECTION + BEHAVIORAL CLUSTER LABELING

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] LOADING MODEL ASSIGNMENTS

KMeans
Path: /content/step6_unsupervised_ml/step6c_kmeans/kmeans_cluster_assignments.csv
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'kmeans_cluster']
Detected ID column: reconstructed_session_id
Detected cluster column: kmeans_cluster
Clusters found: 5
Assignment integrity: PASS

DBSCAN
Path: /content/step6_unsupervised_ml/step6d_dbscan/dbscan_cluster_assignments.csv
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'dbscan_cluster']
Detected ID column: reconstructed_session_id
Detected clus

In [ ]:
# ======================================================================
# STEP 7D — DETAILED BEHAVIORAL CLUSTER PROFILING
# Primary model: DBSCAN
# Purpose: Deep behavioral profiling + reusable cluster reference
# ======================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"

DBSCAN_ASSIGNMENTS = (
    "/content/step6_unsupervised_ml/step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

STEP7C_DIR = (
    "/content/step7_cluster_profiling/"
    "step7c_model_selection_behavioral_labeling"
)

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7d_detailed_behavioral_profiling"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

FORBIDDEN_COLUMNS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "label",
    "target",
    "y",
}

ID_COLUMNS = {
    "reconstructed_session_id",
    "session_id",
    "session_start_time",
    "session_end_time",
}

# ----------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def clean_feature_name(x):
    return str(x).strip()


# ----------------------------------------------------------------------
# STEP 1 — SOURCE VERIFICATION
# ----------------------------------------------------------------------

print("=" * 70)
print("STEP 7D — DETAILED BEHAVIORAL CLUSTER PROFILING")
print("=" * 70)

if not os.path.exists(SOURCE):
    raise FileNotFoundError(f"STOP — source missing: {SOURCE}")

source_sha_before = sha256_file(SOURCE)

print("\n[1] SOURCE VERIFICATION")
print(f"Expected SHA: {EXPECTED_SOURCE_SHA}")
print(f"Actual SHA:   {source_sha_before}")

if source_sha_before != EXPECTED_SOURCE_SHA:
    raise ValueError("STOP — source SHA mismatch.")

print("Verification: PASS")


# ----------------------------------------------------------------------
# STEP 2 — LOAD SOURCE
# ----------------------------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] SOURCE DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")


# ----------------------------------------------------------------------
# STEP 3 — SESSION INTEGRITY
# ----------------------------------------------------------------------

if "reconstructed_session_id" not in df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing from source."
    )

missing_ids = int(df["reconstructed_session_id"].isna().sum())
duplicate_ids = int(df["reconstructed_session_id"].duplicated().sum())
unique_ids = int(df["reconstructed_session_id"].nunique())

print("\n[3] SESSION INTEGRITY")
print(f"Missing IDs:   {missing_ids}")
print(f"Duplicate IDs: {duplicate_ids}")
print(f"Unique IDs:    {unique_ids}")

if missing_ids != 0 or duplicate_ids != 0:
    raise ValueError("STOP — session integrity failure.")

print("Session integrity: PASS")


# ----------------------------------------------------------------------
# STEP 4 — LABEL LEAKAGE
# ----------------------------------------------------------------------

present_forbidden = sorted(
    set(df.columns).intersection(FORBIDDEN_COLUMNS)
)

print("\n[4] LABEL LEAKAGE")
print(f"Forbidden columns present: {present_forbidden}")

if present_forbidden:
    raise ValueError(
        f"STOP — forbidden label columns detected: {present_forbidden}"
    )

print("Label leakage: NO")


# ----------------------------------------------------------------------
# STEP 5 — LOAD DBSCAN ASSIGNMENTS
# ----------------------------------------------------------------------

print("\n[5] LOADING PRIMARY MODEL ASSIGNMENTS")

if not os.path.exists(DBSCAN_ASSIGNMENTS):
    raise FileNotFoundError(
        f"STOP — DBSCAN assignments missing: {DBSCAN_ASSIGNMENTS}"
    )

assign = pd.read_csv(DBSCAN_ASSIGNMENTS)

print(f"Path: {DBSCAN_ASSIGNMENTS}")
print(f"Shape: {assign.shape}")
print(f"Columns: {list(assign.columns)}")

id_candidates = [
    c for c in assign.columns
    if c.lower() in {
        "reconstructed_session_id",
        "session_id"
    }
]

cluster_candidates = [
    c for c in assign.columns
    if "dbscan" in c.lower() and "cluster" in c.lower()
]

if not id_candidates:
    raise ValueError(
        "STOP — DBSCAN assignment ID column not detected."
    )

if not cluster_candidates:
    raise ValueError(
        "STOP — DBSCAN cluster column not detected."
    )

assignment_id_col = id_candidates[0]
cluster_col = cluster_candidates[0]

print(f"Detected ID column: {assignment_id_col}")
print(f"Detected cluster column: {cluster_col}")

if assign[assignment_id_col].isna().any():
    raise ValueError("STOP — missing session IDs in DBSCAN assignments.")

if assign[assignment_id_col].duplicated().any():
    raise ValueError(
        "STOP — duplicate session IDs in DBSCAN assignments."
    )

if set(assign[assignment_id_col]) != set(df["reconstructed_session_id"]):
    raise ValueError(
        "STOP — DBSCAN session IDs do not exactly match source."
    )

if assign[cluster_col].isna().any():
    raise ValueError(
        "STOP — missing DBSCAN cluster assignments."
    )

print("Assignment integrity: PASS")


# ----------------------------------------------------------------------
# STEP 6 — MERGE
# ----------------------------------------------------------------------

work = df.merge(
    assign[[assignment_id_col, cluster_col]],
    left_on="reconstructed_session_id",
    right_on=assignment_id_col,
    how="inner",
    validate="one_to_one",
)

if len(work) != len(df):
    raise ValueError(
        "STOP — merged dataset row count changed."
    )

work = work.drop(columns=[assignment_id_col])

work = work.rename(
    columns={cluster_col: "DBSCAN_cluster"}
)

print("\n[6] ASSIGNMENT MERGE")
print(f"Merged rows: {len(work)}")
print("Merge validation: PASS")


# ----------------------------------------------------------------------
# STEP 7 — STRICT BEHAVIORAL FEATURES
# ----------------------------------------------------------------------

behavioral_columns = []

for col in df.columns:

    if col in ID_COLUMNS:
        continue

    if col in FORBIDDEN_COLUMNS:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        behavioral_columns.append(col)

behavioral_columns = [
    clean_feature_name(c)
    for c in behavioral_columns
]

print("\n[7] BEHAVIORAL FEATURE INVENTORY")
print(f"Behavioral numeric features: {len(behavioral_columns)}")

if len(behavioral_columns) == 0:
    raise ValueError(
        "STOP — no behavioral numeric features found."
    )

# ----------------------------------------------------------------------
# STEP 8 — REMOVE ALL-MISSING FEATURES
# ----------------------------------------------------------------------

missing_counts = (
    work[behavioral_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

all_missing_features = [
    c for c in behavioral_columns
    if missing_counts[c] == len(work)
]

usable_features = [
    c for c in behavioral_columns
    if c not in all_missing_features
]

print("\n[8] MISSINGNESS")
print(f"All-missing features excluded: {len(all_missing_features)}")

if all_missing_features:
    for c in all_missing_features:
        print(f" - {c}")

if len(usable_features) == 0:
    raise ValueError(
        "STOP — no usable behavioral features remain."
    )


# ----------------------------------------------------------------------
# STEP 9 — GLOBAL REFERENCE STATISTICS
# ----------------------------------------------------------------------

global_rows = []

for feature in usable_features:

    s = pd.to_numeric(
        work[feature],
        errors="coerce"
    )

    global_rows.append({
        "feature": feature,
        "count": int(s.notna().sum()),
        "missing_count": int(s.isna().sum()),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "median": float(s.median()),
        "min": float(s.min()),
        "max": float(s.max()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
    })

global_stats = pd.DataFrame(global_rows)

global_path = os.path.join(
    OUTPUT_DIR,
    "step7d_global_behavioral_reference.csv"
)

global_stats.to_csv(global_path, index=False)

print("\n[9] GLOBAL BEHAVIORAL REFERENCE")
print(f"Saved: {global_path}")


# ----------------------------------------------------------------------
# STEP 10 — CLUSTER SIZE PROFILE
# ----------------------------------------------------------------------

cluster_sizes = (
    work["DBSCAN_cluster"]
    .value_counts(dropna=False)
    .rename_axis("DBSCAN_cluster")
    .reset_index(name="session_count")
)

cluster_sizes["percentage"] = (
    cluster_sizes["session_count"] /
    len(work) * 100
)

cluster_sizes = cluster_sizes.sort_values(
    "DBSCAN_cluster"
)

cluster_size_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_cluster_sizes.csv"
)

cluster_sizes.to_csv(
    cluster_size_path,
    index=False
)

print("\n[10] CLUSTER SIZE PROFILE")
print(cluster_sizes.to_string(index=False))
print(f"Saved: {cluster_size_path}")


# ----------------------------------------------------------------------
# STEP 11 — DETAILED CLUSTER STATISTICS
# ----------------------------------------------------------------------

print("\n[11] BUILDING DETAILED CLUSTER PROFILES")

profile_rows = []

cluster_values = sorted(
    work["DBSCAN_cluster"].unique()
)

for cluster_id in cluster_values:

    cluster_df = work[
        work["DBSCAN_cluster"] == cluster_id
    ]

    for feature in usable_features:

        s = pd.to_numeric(
            cluster_df[feature],
            errors="coerce"
        )

        profile_rows.append({
            "model": "DBSCAN",
            "cluster": int(cluster_id),
            "session_count": len(cluster_df),
            "cluster_percentage": (
                len(cluster_df) / len(work) * 100
            ),
            "feature": feature,
            "mean": float(s.mean()),
            "median": float(s.median()),
            "std": float(s.std()),
            "min": float(s.min()),
            "max": float(s.max()),
            "q25": float(s.quantile(0.25)),
            "q75": float(s.quantile(0.75)),
        })

profiles = pd.DataFrame(profile_rows)

profile_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_detailed_cluster_profiles.csv"
)

profiles.to_csv(
    profile_path,
    index=False
)

print(f"Saved: {profile_path}")


# ----------------------------------------------------------------------
# STEP 12 — STANDARDIZED CLUSTER SIGNATURES
# ----------------------------------------------------------------------
# Compare each cluster mean against the GLOBAL mean/std.
# This creates a reusable behavioral signature:
# positive = above global baseline
# negative = below global baseline

print("\n[12] BUILDING STANDARDIZED CLUSTER SIGNATURES")

global_lookup = global_stats.set_index("feature")

signature_rows = []

for cluster_id in cluster_values:

    cluster_df = work[
        work["DBSCAN_cluster"] == cluster_id
    ]

    for feature in usable_features:

        cluster_mean = pd.to_numeric(
            cluster_df[feature],
            errors="coerce"
        ).mean()

        global_mean = global_lookup.loc[
            feature, "mean"
        ]

        global_std = global_lookup.loc[
            feature, "std"
        ]

        if pd.isna(global_std) or global_std == 0:
            z = 0.0
        else:
            z = (
                cluster_mean - global_mean
            ) / global_std

        signature_rows.append({
            "model": "DBSCAN",
            "cluster": int(cluster_id),
            "feature": feature,
            "cluster_mean": float(cluster_mean),
            "global_mean": float(global_mean),
            "global_std": float(global_std),
            "behavioral_z_score": float(z),
            "absolute_z_score": float(abs(z)),
        })

signatures = pd.DataFrame(signature_rows)

signature_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_behavioral_signatures.csv"
)

signatures.to_csv(
    signature_path,
    index=False
)

print(f"Saved: {signature_path}")


# ----------------------------------------------------------------------
# STEP 13 — TOP BEHAVIORAL FEATURES PER CLUSTER
# ----------------------------------------------------------------------

print("\n[13] EXTRACTING TOP DISCRIMINATING FEATURES")

top_rows = []

for cluster_id in cluster_values:

    temp = signatures[
        signatures["cluster"] == cluster_id
    ].sort_values(
        "absolute_z_score",
        ascending=False
    )

    top_n = temp.head(10).copy()

    for rank, (_, row) in enumerate(
        top_n.iterrows(),
        start=1
    ):

        top_rows.append({
            "model": "DBSCAN",
            "cluster": int(cluster_id),
            "rank": rank,
            "feature": row["feature"],
            "behavioral_z_score": row[
                "behavioral_z_score"
            ],
            "absolute_z_score": row[
                "absolute_z_score"
            ],
        })

top_features = pd.DataFrame(top_rows)

top_features_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_top_behavioral_features.csv"
)

top_features.to_csv(
    top_features_path,
    index=False
)

print(f"Saved: {top_features_path}")


# ----------------------------------------------------------------------
# STEP 14 — HUMAN-READABLE CLUSTER SUMMARY
# ----------------------------------------------------------------------

print("\n[14] BUILDING CLUSTER INTERPRETATION SUMMARY")

summary_rows = []

for cluster_id in cluster_values:

    cluster_size = int(
        cluster_sizes.loc[
            cluster_sizes["DBSCAN_cluster"] == cluster_id,
            "session_count"
        ].iloc[0]
    )

    cluster_pct = float(
        cluster_sizes.loc[
            cluster_sizes["DBSCAN_cluster"] == cluster_id,
            "percentage"
        ].iloc[0]
    )

    top = top_features[
        top_features["cluster"] == cluster_id
    ].sort_values("rank")

    high_features = top[
        top["behavioral_z_score"] > 0
    ]["feature"].tolist()

    low_features = top[
        top["behavioral_z_score"] < 0
    ]["feature"].tolist()

    summary_rows.append({
        "model": "DBSCAN",
        "cluster": int(cluster_id),
        "session_count": cluster_size,
        "percentage": cluster_pct,
        "cluster_type": (
            "NOISE / OUTLIER"
            if cluster_id == -1
            else "BEHAVIORAL CLUSTER"
        ),
        "top_elevated_features": " | ".join(
            high_features[:5]
        ),
        "top_reduced_features": " | ".join(
            low_features[:5]
        ),
        "interpretation_status": (
            "Requires investigation"
            if cluster_id == -1
            else "Profile available"
        ),
    })

summary = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_cluster_interpretation_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print(f"Saved: {summary_path}")


# ----------------------------------------------------------------------
# STEP 15 — NOISE / OUTLIER PROFILE
# ----------------------------------------------------------------------

noise_profile = signatures[
    signatures["cluster"] == -1
].sort_values(
    "absolute_z_score",
    ascending=False
)

noise_path = os.path.join(
    OUTPUT_DIR,
    "step7d_dbscan_noise_behavioral_profile.csv"
)

noise_profile.to_csv(
    noise_path,
    index=False
)

print("\n[15] DBSCAN NOISE PROFILE")

if -1 in cluster_values:
    print(
        f"Noise sessions: "
        f"{len(work[work['DBSCAN_cluster'] == -1])}"
    )
    print(f"Saved: {noise_path}")
else:
    print("No DBSCAN noise cluster detected.")


# ----------------------------------------------------------------------
# STEP 16 — REUSABLE REFERENCE ARTIFACT
# ----------------------------------------------------------------------
# This artifact is intended to help future logs use the SAME
# behavioral definitions instead of inventing new features.

reference = {
    "pipeline_step": "7D",
    "primary_model": "DBSCAN",
    "source_sha256": source_sha_before,
    "session_count": int(len(work)),
    "behavioral_feature_count": int(len(usable_features)),
    "excluded_all_missing_features": all_missing_features,
    "dbscan_cluster_count_including_noise": int(
        len(cluster_values)
    ),
    "noise_label": -1,
    "noise_fraction": float(
        (
            work["DBSCAN_cluster"] == -1
        ).mean()
    ),
    "behavioral_features": usable_features,
    "future_logs_rule": (
        "Use the same feature names, definitions, "
        "preprocessing and scaling pipeline. "
        "Do not introduce labels into clustering."
    ),
}

reference_path = os.path.join(
    OUTPUT_DIR,
    "step7d_behavioral_reference.json"
)

with open(reference_path, "w") as f:
    json.dump(
        reference,
        f,
        indent=2
    )

print("\n[16] REUSABLE BEHAVIORAL REFERENCE")
print(f"Saved: {reference_path}")


# ----------------------------------------------------------------------
# STEP 17 — AUDIT
# ----------------------------------------------------------------------

source_sha_after = sha256_file(SOURCE)

if source_sha_after != source_sha_before:
    raise ValueError(
        "STOP — source changed during Step 7D."
    )

audit = {
    "step": "7D",
    "status": "PASS",
    "source": SOURCE,
    "source_sha256_before": source_sha_before,
    "source_sha256_after": source_sha_after,
    "source_unchanged": True,
    "rows": int(len(work)),
    "columns": int(len(df.columns)),
    "session_integrity": True,
    "missing_session_ids": missing_ids,
    "duplicate_session_ids": duplicate_ids,
    "label_leakage": False,
    "forbidden_columns_present": present_forbidden,
    "primary_model": "DBSCAN",
    "cluster_column": "DBSCAN_cluster",
    "clusters_including_noise": int(len(cluster_values)),
    "noise_sessions": int(
        (work["DBSCAN_cluster"] == -1).sum()
    ),
    "noise_fraction": float(
        (work["DBSCAN_cluster"] == -1).mean()
    ),
    "behavioral_feature_count": int(len(usable_features)),
    "excluded_all_missing_features": all_missing_features,
    "outputs": {
        "global_reference": global_path,
        "cluster_sizes": cluster_size_path,
        "detailed_profiles": profile_path,
        "behavioral_signatures": signature_path,
        "top_features": top_features_path,
        "interpretation_summary": summary_path,
        "noise_profile": noise_path,
        "behavioral_reference": reference_path,
    },
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7d_detailed_behavioral_profiling_audit.json"
)

with open(audit_path, "w") as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(audit_path)

print("\n[17] FINAL AUDIT")
print(f"Source unchanged:      PASS")
print(f"Session integrity:     PASS")
print(f"Label leakage:         NO")
print(f"Behavioral features:   {len(usable_features)}")
print(f"Primary model:         DBSCAN")
print(f"Noise sessions:        {(work['DBSCAN_cluster'] == -1).sum()}")
print(f"Audit:                 {audit_path}")
print(f"Audit SHA-256:         {audit_sha}")

print("\n" + "=" * 70)
print("STEP 7D — COMPLETE")
print("=" * 70)
print("STATUS: PASS")
print("NEXT: STEP 7E — CLUSTER VALIDATION + STABILITY CHECK")

STEP 7D — DETAILED BEHAVIORAL CLUSTER PROFILING

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)
Rows: 11880
Columns: 56

[3] SESSION INTEGRITY
Missing IDs:   0
Duplicate IDs: 0
Unique IDs:    11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] LOADING PRIMARY MODEL ASSIGNMENTS
Path: /content/step6_unsupervised_ml/step6d_dbscan/dbscan_cluster_assignments.csv
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'dbscan_cluster']
Detected ID column: reconstructed_session_id
Detected cluster column: dbscan_cluster
Assignment integrity: PASS

[6] ASSIGNMENT MERGE
Merged rows: 11880
Merge validation: PASS

[7] BEHAVIORAL FEATURE INVENTORY
Behavioral numeric features: 53

[8] MISSINGNESS
All-missing features excluded: 5
 - ip_reputation_mean
 - ip_rep

In [ ]:
# ================================================================
# STEP 7E — CLUSTER VALIDATION + STABILITY CHECK
# ================================================================

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    adjusted_mutual_info_score
)

warnings.filterwarnings("ignore")

# ================================================================
# CONFIG
# ================================================================

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"

SOURCE_EXPECTED_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

ASSIGNMENTS = (
    "/content/step6_unsupervised_ml/step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

PROFILE_DIR = (
    "/content/step7_cluster_profiling/"
    "step7d_detailed_behavioral_profiling"
)

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7e_cluster_validation_stability"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Same DBSCAN configuration selected in STEP 6D
EPS = 0.5
MIN_SAMPLES = 20

# Stability configuration
RANDOM_SEEDS = [42, 123, 2024, 777, 999]
SAMPLE_FRACTION = 0.80

# ================================================================
# HELPERS
# ================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def find_id_column(df):
    candidates = [
        "reconstructed_session_id",
        "session_id",
        "session",
        "id"
    ]
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "STOP — No valid session ID column found."
    )


def safe_cluster_metrics(X, labels):
    """
    Metrics exclude DBSCAN noise (-1).
    """

    mask = labels != -1

    X_eval = X[mask]
    y_eval = labels[mask]

    unique_clusters = np.unique(y_eval)

    if len(unique_clusters) < 2:
        return {
            "silhouette_score": np.nan,
            "davies_bouldin_score": np.nan,
            "calinski_harabasz_score": np.nan,
            "clusters_found": len(unique_clusters),
            "noise_count": int((labels == -1).sum()),
            "noise_fraction": float((labels == -1).mean())
        }

    return {
        "silhouette_score": float(
            silhouette_score(X_eval, y_eval)
        ),
        "davies_bouldin_score": float(
            davies_bouldin_score(X_eval, y_eval)
        ),
        "calinski_harabasz_score": float(
            calinski_harabasz_score(X_eval, y_eval)
        ),
        "clusters_found": len(unique_clusters),
        "noise_count": int((labels == -1).sum()),
        "noise_fraction": float((labels == -1).mean())
    }


# ================================================================
# HEADER
# ================================================================

print("=" * 70)
print("STEP 7E — CLUSTER VALIDATION + STABILITY CHECK")
print("=" * 70)

# ================================================================
# [1] SOURCE VERIFICATION
# ================================================================

print("\n[1] SOURCE VERIFICATION")

actual_source_sha = sha256_file(SOURCE)

print("Expected SHA:", SOURCE_EXPECTED_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != SOURCE_EXPECTED_SHA:
    raise ValueError(
        "STOP — source SHA mismatch. "
        "Do not continue until the correct source is restored."
    )

print("Verification: PASS")

source_sha_before = actual_source_sha

# ================================================================
# [2] LOAD SOURCE
# ================================================================

print("\n[2] SOURCE DATASET")

df = pd.read_csv(SOURCE)

print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, got {len(df)}."
    )

# ================================================================
# [3] SESSION INTEGRITY
# ================================================================

print("\n[3] SESSION INTEGRITY")

ID_COL = find_id_column(df)

missing_ids = int(df[ID_COL].isna().sum())
duplicate_ids = int(df[ID_COL].duplicated().sum())
unique_ids = int(df[ID_COL].nunique())

print("ID column:", ID_COL)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != 11880:
    raise ValueError(
        "STOP — session integrity failure."
    )

print("Session integrity: PASS")

# ================================================================
# [4] LABEL LEAKAGE
# ================================================================

print("\n[4] LABEL LEAKAGE")

FORBIDDEN = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "mitre_techniques",
    "label",
    "target",
    "y"
}

present_forbidden = [
    c for c in df.columns
    if c.lower() in FORBIDDEN
]

print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        "STOP — forbidden label/target columns detected."
    )

print("Label leakage: NO")

# ================================================================
# [5] LOAD VERIFIED DBSCAN ASSIGNMENTS
# ================================================================

print("\n[5] LOADING DBSCAN ASSIGNMENTS")

assign = pd.read_csv(ASSIGNMENTS)

print("Shape:", assign.shape)
print("Columns:", list(assign.columns))

assign_id_col = find_id_column(assign)

cluster_candidates = [
    "dbscan_cluster",
    "DBSCAN_cluster",
    "cluster"
]

cluster_col = None

for c in cluster_candidates:
    if c in assign.columns:
        cluster_col = c
        break

if cluster_col is None:
    raise ValueError(
        "STOP — DBSCAN cluster column not found."
    )

print("Detected ID column:", assign_id_col)
print("Detected cluster column:", cluster_col)

if len(assign) != len(df):
    raise ValueError(
        "STOP — assignment row count does not match source."
    )

if assign[assign_id_col].duplicated().any():
    raise ValueError(
        "STOP — duplicate session IDs in assignments."
    )

source_ids = set(df[ID_COL])
assignment_ids = set(assign[assign_id_col])

if source_ids != assignment_ids:
    raise ValueError(
        "STOP — assignment session IDs do not match source."
    )

print("Assignment integrity: PASS")

# ================================================================
# [6] BASELINE DBSCAN VALIDATION
# ================================================================

print("\n[6] BASELINE DBSCAN VALIDATION")

labels_reference = assign[cluster_col].to_numpy()

print("eps:", EPS)
print("min_samples:", MIN_SAMPLES)
print("Clusters including noise:", len(np.unique(labels_reference)))
print(
    "Clusters excluding noise:",
    len(np.unique(labels_reference[labels_reference != -1]))
)
print("Noise sessions:", int((labels_reference == -1).sum()))
print(
    "Noise fraction:",
    round(float((labels_reference == -1).mean()), 6)
)

# ================================================================
# [7] STRICT BEHAVIORAL FEATURE MATRIX
# ================================================================

print("\n[7] BEHAVIORAL FEATURE MATRIX")

EXCLUDED_EXACT = {
    ID_COL.lower(),
    "reconstructed_session_id",
    "session_start_time",
    "session_end_time",
}

# Exclude anything obviously derived from clustering
def is_cluster_derived(col):
    c = col.lower()
    return (
        "cluster" in c
        or c.startswith("kmeans_")
        or c.startswith("dbscan_")
        or c.startswith("agglomerative_")
    )

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

behavioral_cols = []

for c in numeric_cols:
    cl = c.lower()

    if cl in EXCLUDED_EXACT:
        continue

    if is_cluster_derived(c):
        continue

    if cl in FORBIDDEN:
        continue

    behavioral_cols.append(c)

# Remove all-missing columns
all_missing = [
    c for c in behavioral_cols
    if df[c].isna().all()
]

behavioral_cols = [
    c for c in behavioral_cols
    if c not in all_missing
]

print("Behavioral numeric features:", len(behavioral_cols))

if all_missing:
    print("All-missing excluded:")
    for c in all_missing:
        print(" -", c)

if len(behavioral_cols) == 0:
    raise ValueError(
        "STOP — no usable behavioral numeric features."
    )

# ================================================================
# [8] MISSING VALUE HANDLING
# ================================================================

print("\n[8] MISSING VALUE HANDLING")

X_df = df[behavioral_cols].copy()

missing_counts = X_df.isna().sum()
missing_features = missing_counts[
    missing_counts > 0
].to_dict()

print("Features with missing values:", len(missing_features))

# Use training-independent deterministic median imputation
# only for validation calculations.
for c in behavioral_cols:
    if X_df[c].isna().any():
        X_df[c] = X_df[c].fillna(
            X_df[c].median()
        )

X = X_df.to_numpy(dtype=float)

if not np.isfinite(X).all():
    raise ValueError(
        "STOP — non-finite values remain in validation matrix."
    )

print("Finite matrix: PASS")

# ================================================================
# [9] BASELINE METRICS
# ================================================================

print("\n[9] BASELINE METRICS")

baseline_metrics = safe_cluster_metrics(
    X,
    labels_reference
)

baseline_metrics["model"] = "DBSCAN_reference"
baseline_metrics["eps"] = EPS
baseline_metrics["min_samples"] = MIN_SAMPLES

print(
    pd.DataFrame([baseline_metrics]).to_string(index=False)
)

# ================================================================
# [10] STABILITY CHECK
# ================================================================

print("\n[10] STABILITY CHECK")

print(
    f"Seeds: {RANDOM_SEEDS}"
)
print(
    f"Sample fraction: {SAMPLE_FRACTION:.2f}"
)

stability_records = []

n = len(df)
sample_size = int(n * SAMPLE_FRACTION)

for seed in RANDOM_SEEDS:

    print(f"\nRunning stability seed={seed} ...")

    rng = np.random.RandomState(seed)

    sample_idx = np.sort(
        rng.choice(
            n,
            size=sample_size,
            replace=False
        )
    )

    X_sample = X[sample_idx]

    # Reference assignments restricted to same sampled sessions
    reference_sample_labels = labels_reference[sample_idx]

    # Re-run the EXACT selected DBSCAN configuration
    model = DBSCAN(
        eps=EPS,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        n_jobs=-1
    )

    sample_labels = model.fit_predict(X_sample)

    metrics = safe_cluster_metrics(
        X_sample,
        sample_labels
    )

    # Compare DBSCAN structure against original assignments
    # on the same sampled sessions.
    ari = adjusted_rand_score(
        reference_sample_labels,
        sample_labels
    )

    ami = adjusted_mutual_info_score(
        reference_sample_labels,
        sample_labels
    )

    record = {
        "seed": seed,
        "sample_size": sample_size,
        "sample_fraction": SAMPLE_FRACTION,
        "clusters_found": metrics["clusters_found"],
        "noise_count": metrics["noise_count"],
        "noise_fraction": metrics["noise_fraction"],
        "silhouette_score": metrics["silhouette_score"],
        "davies_bouldin_score": metrics["davies_bouldin_score"],
        "calinski_harabasz_score": metrics[
            "calinski_harabasz_score"
        ],
        "adjusted_rand_index_vs_reference": float(ari),
        "adjusted_mutual_info_vs_reference": float(ami)
    }

    stability_records.append(record)

    print(
        "clusters:",
        record["clusters_found"],
        "| noise:",
        record["noise_count"],
        "| silhouette:",
        round(
            record["silhouette_score"], 4
        )
        if np.isfinite(record["silhouette_score"])
        else "NaN",
        "| ARI:",
        round(ari, 4),
        "| AMI:",
        round(ami, 4)
    )

stability_df = pd.DataFrame(
    stability_records
)

stability_path = os.path.join(
    OUTPUT_DIR,
    "step7e_dbscan_stability_results.csv"
)

stability_df.to_csv(
    stability_path,
    index=False
)

print(
    "\nSaved:",
    stability_path
)

# ================================================================
# [11] STABILITY SUMMARY
# ================================================================

print("\n[11] STABILITY SUMMARY")

summary = {
    "reference_clusters": int(
        len(np.unique(
            labels_reference[
                labels_reference != -1
            ]
        ))
    ),
    "reference_noise_fraction": float(
        (labels_reference == -1).mean()
    ),
    "mean_clusters_found": float(
        stability_df["clusters_found"].mean()
    ),
    "std_clusters_found": float(
        stability_df["clusters_found"].std(ddof=0)
    ),
    "min_clusters_found": int(
        stability_df["clusters_found"].min()
    ),
    "max_clusters_found": int(
        stability_df["clusters_found"].max()
    ),
    "mean_noise_fraction": float(
        stability_df["noise_fraction"].mean()
    ),
    "std_noise_fraction": float(
        stability_df["noise_fraction"].std(ddof=0)
    ),
    "mean_silhouette": float(
        stability_df["silhouette_score"].mean()
    ),
    "std_silhouette": float(
        stability_df["silhouette_score"].std(ddof=0)
    ),
    "mean_ARI_vs_reference": float(
        stability_df[
            "adjusted_rand_index_vs_reference"
        ].mean()
    ),
    "std_ARI_vs_reference": float(
        stability_df[
            "adjusted_rand_index_vs_reference"
        ].std(ddof=0)
    ),
    "mean_AMI_vs_reference": float(
        stability_df[
            "adjusted_mutual_info_vs_reference"
        ].mean()
    ),
    "std_AMI_vs_reference": float(
        stability_df[
            "adjusted_mutual_info_vs_reference"
        ].std(ddof=0)
    )
}

summary_df = pd.DataFrame(
    [summary]
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "step7e_dbscan_stability_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print(summary_df.to_string(index=False))

# ================================================================
# [12] CLUSTER SIZE STABILITY
# ================================================================

print("\n[12] CLUSTER SIZE STABILITY")

reference_cluster_sizes = (
    pd.Series(labels_reference)
    .value_counts()
    .sort_index()
)

cluster_size_rows = []

for cluster, count in reference_cluster_sizes.items():

    cluster_size_rows.append({
        "cluster": int(cluster),
        "reference_count": int(count),
        "reference_fraction": float(
            count / n
        )
    })

cluster_sizes_df = pd.DataFrame(
    cluster_size_rows
)

cluster_sizes_path = os.path.join(
    OUTPUT_DIR,
    "step7e_reference_cluster_sizes.csv"
)

cluster_sizes_df.to_csv(
    cluster_sizes_path,
    index=False
)

print(
    "Saved:",
    cluster_sizes_path
)

# ================================================================
# [13] VALIDATION FLAGS
# ================================================================

print("\n[13] VALIDATION FLAGS")

flags = []

# Basic structural checks
if baseline_metrics["clusters_found"] >= 2:
    flags.append({
        "flag": "MULTI_CLUSTER_STRUCTURE",
        "status": "PASS",
        "description": (
            "DBSCAN identifies multiple non-noise behavioral clusters."
        )
    })
else:
    flags.append({
        "flag": "MULTI_CLUSTER_STRUCTURE",
        "status": "FAIL",
        "description": (
            "DBSCAN does not identify enough behavioral clusters."
        )
    })

# Noise sanity
noise_fraction = baseline_metrics["noise_fraction"]

if 0.01 <= noise_fraction <= 0.30:
    flags.append({
        "flag": "NOISE_FRACTION",
        "status": "PASS",
        "description": (
            "Noise fraction is within the configured validation range."
        )
    })
else:
    flags.append({
        "flag": "NOISE_FRACTION",
        "status": "REVIEW",
        "description": (
            "Noise fraction is outside the configured validation range."
        )
    })

# Cluster count stability
cluster_std = summary["std_clusters_found"]

if cluster_std <= 5:
    flags.append({
        "flag": "CLUSTER_COUNT_STABILITY",
        "status": "PASS",
        "description": (
            "Cluster count remains reasonably stable across samples."
        )
    })
else:
    flags.append({
        "flag": "CLUSTER_COUNT_STABILITY",
        "status": "REVIEW",
        "description": (
            "Cluster count varies substantially across samples."
        )
    })

# ARI
mean_ari = summary["mean_ARI_vs_reference"]

if mean_ari >= 0.70:
    ari_status = "PASS"
elif mean_ari >= 0.50:
    ari_status = "REVIEW"
else:
    ari_status = "FAIL"

flags.append({
    "flag": "ASSIGNMENT_STABILITY_ARI",
    "status": ari_status,
    "description": (
        "Mean Adjusted Rand Index comparing sampled DBSCAN "
        "assignments against the reference assignments."
    )
})

# AMI
mean_ami = summary["mean_AMI_vs_reference"]

if mean_ami >= 0.70:
    ami_status = "PASS"
elif mean_ami >= 0.50:
    ami_status = "REVIEW"
else:
    ami_status = "FAIL"

flags.append({
    "flag": "ASSIGNMENT_STABILITY_AMI",
    "status": ami_status,
    "description": (
        "Mean Adjusted Mutual Information comparing sampled "
        "DBSCAN assignments against the reference assignments."
    )
})

# Silhouette stability
sil_std = summary["std_silhouette"]

if sil_std <= 0.05:
    flags.append({
        "flag": "SILHOUETTE_STABILITY",
        "status": "PASS",
        "description": (
            "Silhouette remains stable across sampled runs."
        )
    })
else:
    flags.append({
        "flag": "SILHOUETTE_STABILITY",
        "status": "REVIEW",
        "description": (
            "Silhouette varies noticeably across sampled runs."
        )
    })

flags_df = pd.DataFrame(flags)

flags_path = os.path.join(
    OUTPUT_DIR,
    "step7e_validation_flags.csv"
)

flags_df.to_csv(
    flags_path,
    index=False
)

print(
    flags_df.to_string(index=False)
)

# ================================================================
# [14] FINAL DECISION
# ================================================================

print("\n[14] FINAL DECISION")

fail_count = int(
    (flags_df["status"] == "FAIL").sum()
)

review_count = int(
    (flags_df["status"] == "REVIEW").sum()
)

if fail_count > 0:
    final_status = "REVIEW_REQUIRED"
elif review_count > 0:
    final_status = "PASS_WITH_REVIEW"
else:
    final_status = "PASS"

print("FAIL flags:", fail_count)
print("REVIEW flags:", review_count)
print("FINAL STATUS:", final_status)

# ================================================================
# [15] SOURCE PROTECTION
# ================================================================

print("\n[15] SOURCE PROTECTION")

source_sha_after = sha256_file(SOURCE)

print("SHA before:", source_sha_before)
print("SHA after: ", source_sha_after)

if source_sha_after != source_sha_before:
    raise ValueError(
        "STOP — source file changed during STEP 7E."
    )

print("Source unchanged: PASS")

# ================================================================
# [16] AUDIT
# ================================================================

audit = {
    "step": "7E",
    "title": "Cluster Validation + Stability Check",

    "source": SOURCE,
    "source_sha256": source_sha_after,

    "dataset": {
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "session_id_column": ID_COL
    },

    "behavioral_features": {
        "count": int(len(behavioral_cols)),
        "all_missing_excluded": all_missing,
        "missing_feature_count": int(
            len(missing_features)
        )
    },

    "dbscan_reference": {
        "eps": EPS,
        "min_samples": MIN_SAMPLES,
        "clusters_found": int(
            baseline_metrics["clusters_found"]
        ),
        "noise_count": int(
            baseline_metrics["noise_count"]
        ),
        "noise_fraction": float(
            baseline_metrics["noise_fraction"]
        ),
        "silhouette_score": baseline_metrics[
            "silhouette_score"
        ],
        "davies_bouldin_score": baseline_metrics[
            "davies_bouldin_score"
        ],
        "calinski_harabasz_score": baseline_metrics[
            "calinski_harabasz_score"
        ]
    },

    "stability": summary,

    "flags": flags,

    "final_status": final_status,

    "source_modified": False,
    "session_integrity": "PASS",
    "label_leakage": "NO",
    "non_finite_values": False
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7e_cluster_validation_stability_audit.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

audit_sha = sha256_file(audit_path)

# ================================================================
# [17] FINAL REPORT
# ================================================================

print("\n" + "=" * 70)
print("STEP 7E — COMPLETE")
print("=" * 70)

print("\nPrimary model: DBSCAN")
print("eps:", EPS)
print("min_samples:", MIN_SAMPLES)

print(
    "Reference clusters:",
    baseline_metrics["clusters_found"]
)

print(
    "Reference noise:",
    baseline_metrics["noise_count"]
)

print(
    "Reference noise fraction:",
    round(
        baseline_metrics["noise_fraction"],
        6
    )
)

print(
    "Mean ARI:",
    round(
        summary["mean_ARI_vs_reference"],
        6
    )
)

print(
    "Mean AMI:",
    round(
        summary["mean_AMI_vs_reference"],
        6
    )
)

print(
    "Mean silhouette:",
    round(
        summary["mean_silhouette"],
        6
    )
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nArtifacts:")
print(stability_path)
print(summary_path)
print(cluster_sizes_path)
print(flags_path)
print(audit_path)

print("\nAudit SHA-256:")
print(audit_sha)

print("\nSTATUS:", final_status)

if final_status == "PASS":
    print("NEXT: STEP 7F — CLUSTER SEMANTIC LABELS + THREAT GROUP SUMMARY")
elif final_status == "PASS_WITH_REVIEW":
    print(
        "NEXT: REVIEW STABILITY FLAGS, "
        "then continue to STEP 7F."
    )
else:
    print(
        "NEXT: DO NOT finalize clustering yet. "
        "Review stability results before STEP 7F."
    )

STEP 7E — CLUSTER VALIDATION + STABILITY CHECK

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] LOADING DBSCAN ASSIGNMENTS
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'dbscan_cluster']
Detected ID column: reconstructed_session_id
Detected cluster column: dbscan_cluster
Assignment integrity: PASS

[6] BASELINE DBSCAN VALIDATION
eps: 0.5
min_samples: 20
Clusters including noise: 39
Clusters excluding noise: 38
Noise sessions: 1676
Noise fraction: 0.141077

[7] BEHAVIORAL FEATURE MATRIX
Behavioral numeric features: 48
All-missing excluded:
 - ip_reputation_mean
 - ip_reputation_min
 - ip_reputat

In [ ]:
# ================================================================
# STEP 7E-R — CORRECTED DBSCAN STABILITY VALIDATION
# ================================================================
# Purpose:
#   Re-evaluate DBSCAN stability using:
#   1) Fixed preprocessing / feature matrix
#   2) Multiple subsamples
#   3) Pairwise ARI/AMI between independent runs
#   4) Cluster-count / noise stability
#   5) Silhouette stability
#   6) Reference-vs-sample comparison as SECONDARY evidence only
#
# IMPORTANT:
#   - Does NOT modify source data
#   - Does NOT modify baseline DBSCAN assignments
#   - Does NOT use labels
# ================================================================

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    adjusted_mutual_info_score
)

warnings.filterwarnings("ignore")

# ================================================================
# CONFIG
# ================================================================

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"

DBSCAN_ASSIGNMENTS = (
    "/content/step6_unsupervised_ml/"
    "step6d_dbscan/dbscan_cluster_assignments.csv"
)

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7e_corrected_cluster_validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

EPS = 0.5
MIN_SAMPLES = 20

SEEDS = [42, 123, 2024, 777, 999, 31415, 27182, 16180]

SAMPLE_FRACTION = 0.80

# Validation thresholds
MIN_MEAN_ARI = 0.40
MIN_MEAN_AMI = 0.70
MAX_ARI_STD = 0.10
MAX_AMI_STD = 0.10
MAX_CLUSTER_COUNT_CV = 0.20
MAX_NOISE_STD = 0.05
MAX_SILHOUETTE_STD = 0.05

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "mitre_id",
    "mitre_source",
    "label",
    "target",
    "y"
}

# ================================================================
# HELPERS
# ================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def detect_id_column(df):
    candidates = [
        "reconstructed_session_id",
        "session_id",
        "id"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError("STOP — session ID column not found.")


def detect_cluster_column(df):
    candidates = [
        "dbscan_cluster",
        "DBSCAN_cluster",
        "cluster"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "STOP — DBSCAN cluster column not found."
    )


def cv(values):
    values = np.asarray(values, dtype=float)
    mean = np.mean(values)

    if mean == 0:
        return np.inf

    return np.std(values, ddof=0) / mean


# ================================================================
# [1] SOURCE VERIFICATION
# ================================================================

print("=" * 70)
print("STEP 7E-R — CORRECTED DBSCAN STABILITY VALIDATION")
print("=" * 70)

actual_source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE VERIFICATION")
print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source SHA mismatch."
    )

print("Verification: PASS")


# ================================================================
# [2] LOAD SOURCE
# ================================================================

df = pd.read_csv(SOURCE)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 rows, got {len(df)}."
    )


# ================================================================
# [3] SESSION INTEGRITY
# ================================================================

id_col = detect_id_column(df)

missing_ids = int(df[id_col].isna().sum())
duplicate_ids = int(df[id_col].duplicated().sum())
unique_ids = int(df[id_col].nunique())

print("\n[3] SESSION INTEGRITY")
print("ID column:", id_col)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

if unique_ids != len(df):
    raise ValueError("STOP — session IDs are not unique.")

print("Session integrity: PASS")


# ================================================================
# [4] LABEL LEAKAGE CHECK
# ================================================================

present_forbidden = [
    c for c in df.columns
    if c.lower() in FORBIDDEN_LABELS
]

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        "STOP — forbidden label-derived columns detected."
    )

print("Label leakage: NO")


# ================================================================
# [5] LOAD BASELINE ASSIGNMENTS
# ================================================================

assignments = pd.read_csv(DBSCAN_ASSIGNMENTS)

print("\n[5] BASELINE DBSCAN ASSIGNMENTS")
print("Shape:", assignments.shape)
print("Columns:", list(assignments.columns))

assignment_id_col = detect_id_column(assignments)
cluster_col = detect_cluster_column(assignments)

print("Detected ID column:", assignment_id_col)
print("Detected cluster column:", cluster_col)

if len(assignments) != len(df):
    raise ValueError(
        "STOP — assignment row count mismatch."
    )

if assignments[assignment_id_col].duplicated().any():
    raise ValueError(
        "STOP — duplicate assignment session IDs."
    )

source_ids = set(df[id_col].astype(str))
assignment_ids = set(
    assignments[assignment_id_col].astype(str)
)

if source_ids != assignment_ids:
    raise ValueError(
        "STOP — assignment session IDs do not match source."
    )

print("Assignment integrity: PASS")


# ================================================================
# [6] BASELINE CLUSTER STRUCTURE
# ================================================================

baseline_labels = assignments[cluster_col].to_numpy()

baseline_non_noise = sorted(
    set(baseline_labels) - {-1}
)

baseline_cluster_count = len(baseline_non_noise)

baseline_noise_count = int(
    np.sum(baseline_labels == -1)
)

baseline_noise_fraction = (
    baseline_noise_count / len(baseline_labels)
)

print("\n[6] BASELINE STRUCTURE")
print("eps:", EPS)
print("min_samples:", MIN_SAMPLES)
print("Non-noise clusters:", baseline_cluster_count)
print("Noise sessions:", baseline_noise_count)
print("Noise fraction:", round(baseline_noise_fraction, 6))


# ================================================================
# [7] BEHAVIORAL FEATURE MATRIX
# ================================================================

excluded = {
    id_col,
    "session_id",
    "reconstructed_session_id",
    "session_start_time",
    "session_end_time"
}

feature_candidates = []

for c in df.columns:

    if c in excluded:
        continue

    if c.lower() in FORBIDDEN_LABELS:
        continue

    if pd.api.types.is_numeric_dtype(df[c]):
        feature_candidates.append(c)

X_df = df[feature_candidates].copy()

# Remove all-missing columns
all_missing = [
    c for c in X_df.columns
    if X_df[c].isna().all()
]

X_df = X_df.drop(columns=all_missing)

# Median imputation using FULL SOURCE reference only
# This does not invent labels or behavioral categories.
X_df = X_df.replace([np.inf, -np.inf], np.nan)

for c in X_df.columns:
    if X_df[c].isna().any():
        median_value = X_df[c].median()

        if pd.isna(median_value):
            raise ValueError(
                f"STOP — unresolved missing feature: {c}"
            )

        X_df[c] = X_df[c].fillna(median_value)

X = X_df.to_numpy(dtype=float)

if not np.isfinite(X).all():
    raise ValueError(
        "STOP — non-finite values remain."
    )

print("\n[7] BEHAVIORAL FEATURE MATRIX")
print("Behavioral features:", X.shape[1])
print("Matrix shape:", X.shape)
print("Finite matrix: PASS")


# ================================================================
# [8] BASELINE SILHOUETTE
# ================================================================

baseline_mask = baseline_labels != -1

baseline_silhouette = np.nan

if (
    baseline_mask.sum() > 2
    and len(np.unique(baseline_labels[baseline_mask])) >= 2
):
    baseline_silhouette = silhouette_score(
        X[baseline_mask],
        baseline_labels[baseline_mask],
        sample_size=min(
            5000,
            int(baseline_mask.sum())
        ),
        random_state=42
    )

print("\n[8] BASELINE SILHOUETTE")
print(
    "Silhouette:",
    round(float(baseline_silhouette), 6)
)


# ================================================================
# [9] RUN INDEPENDENT SUBSAMPLES
# ================================================================

rng_results = []

n = len(X)
sample_size = int(n * SAMPLE_FRACTION)

print("\n[9] INDEPENDENT SUBSAMPLE STABILITY")

for seed in SEEDS:

    print(
        f"\nRunning seed={seed} ..."
    )

    rng = np.random.default_rng(seed)

    sample_indices = np.sort(
        rng.choice(
            n,
            size=sample_size,
            replace=False
        )
    )

    X_sample = X[sample_indices]

    model = DBSCAN(
        eps=EPS,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        n_jobs=-1
    )

    sample_labels = model.fit_predict(X_sample)

    non_noise_mask = sample_labels != -1

    clusters_found = len(
        set(sample_labels) - {-1}
    )

    noise_count = int(
        np.sum(sample_labels == -1)
    )

    noise_fraction = (
        noise_count / len(sample_labels)
    )

    sil = np.nan

    if (
        non_noise_mask.sum() > 2
        and len(
            np.unique(
                sample_labels[non_noise_mask]
            )
        ) >= 2
    ):
        sil = silhouette_score(
            X_sample[non_noise_mask],
            sample_labels[non_noise_mask],
            sample_size=min(
                5000,
                int(non_noise_mask.sum())
            ),
            random_state=seed
        )

    # Compare this run against baseline only on sampled rows.
    reference_sample_labels = baseline_labels[
        sample_indices
    ]

    ari = adjusted_rand_score(
        reference_sample_labels,
        sample_labels
    )

    ami = adjusted_mutual_info_score(
        reference_sample_labels,
        sample_labels
    )

    rng_results.append({
        "seed": seed,
        "sample_fraction": SAMPLE_FRACTION,
        "sample_size": sample_size,
        "clusters_found": clusters_found,
        "noise_count": noise_count,
        "noise_fraction": noise_fraction,
        "silhouette_score": sil,
        "ARI_vs_reference": ari,
        "AMI_vs_reference": ami
    })

    print(
        f"clusters={clusters_found} | "
        f"noise={noise_count} | "
        f"silhouette={sil:.4f} | "
        f"ARI={ari:.4f} | "
        f"AMI={ami:.4f}"
    )


results_df = pd.DataFrame(rng_results)

results_path = os.path.join(
    OUTPUT_DIR,
    "step7e_corrected_stability_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("\nSaved:", results_path)


# ================================================================
# [10] PAIRWISE STABILITY
# ================================================================
# Important:
# Direct pairwise ARI/AMI requires common observations.
#
# We therefore create a second stability test:
# repeated subsamples are compared on their INTERSECTION.
# This is much more meaningful than comparing a run to the
# full reference partition alone.

print("\n[10] PAIRWISE OVERLAP STABILITY")

pairwise_results = []

# Store labels indexed by session for every seed
run_labels = {}

for seed in SEEDS:

    rng = np.random.default_rng(seed)

    sample_indices = np.sort(
        rng.choice(
            n,
            size=sample_size,
            replace=False
        )
    )

    model = DBSCAN(
        eps=EPS,
        min_samples=MIN_SAMPLES,
        metric="euclidean",
        n_jobs=-1
    )

    labels = model.fit_predict(
        X[sample_indices]
    )

    run_labels[seed] = {
        "indices": sample_indices,
        "labels": labels
    }


for i in range(len(SEEDS)):

    for j in range(i + 1, len(SEEDS)):

        seed_a = SEEDS[i]
        seed_b = SEEDS[j]

        data_a = run_labels[seed_a]
        data_b = run_labels[seed_b]

        idx_a = data_a["indices"]
        idx_b = data_b["indices"]

        common, pos_a, pos_b = np.intersect1d(
            idx_a,
            idx_b,
            assume_unique=True,
            return_indices=True
        )

        labels_a = data_a["labels"][pos_a]
        labels_b = data_b["labels"][pos_b]

        ari_pair = adjusted_rand_score(
            labels_a,
            labels_b
        )

        ami_pair = adjusted_mutual_info_score(
            labels_a,
            labels_b
        )

        pairwise_results.append({
            "seed_a": seed_a,
            "seed_b": seed_b,
            "common_sessions": len(common),
            "ARI_pairwise": ari_pair,
            "AMI_pairwise": ami_pair
        })


pairwise_df = pd.DataFrame(pairwise_results)

pairwise_path = os.path.join(
    OUTPUT_DIR,
    "step7e_pairwise_stability_results.csv"
)

pairwise_df.to_csv(
    pairwise_path,
    index=False
)

print("Saved:", pairwise_path)


# ================================================================
# [11] STABILITY SUMMARY
# ================================================================

mean_clusters = results_df["clusters_found"].mean()
std_clusters = results_df["clusters_found"].std(ddof=0)

mean_noise = results_df["noise_fraction"].mean()
std_noise = results_df["noise_fraction"].std(ddof=0)

mean_silhouette = results_df[
    "silhouette_score"
].mean()

std_silhouette = results_df[
    "silhouette_score"
].std(ddof=0)

mean_ari_reference = results_df[
    "ARI_vs_reference"
].mean()

std_ari_reference = results_df[
    "ARI_vs_reference"
].std(ddof=0)

mean_ami_reference = results_df[
    "AMI_vs_reference"
].mean()

std_ami_reference = results_df[
    "AMI_vs_reference"
].std(ddof=0)

mean_pairwise_ari = pairwise_df[
    "ARI_pairwise"
].mean()

std_pairwise_ari = pairwise_df[
    "ARI_pairwise"
].std(ddof=0)

mean_pairwise_ami = pairwise_df[
    "AMI_pairwise"
].mean()

std_pairwise_ami = pairwise_df[
    "AMI_pairwise"
].std(ddof=0)

cluster_cv = (
    std_clusters / mean_clusters
    if mean_clusters != 0
    else np.inf
)

summary = {
    "reference_clusters": baseline_cluster_count,
    "reference_noise_fraction": baseline_noise_fraction,

    "mean_clusters_found": mean_clusters,
    "std_clusters_found": std_clusters,
    "cluster_count_cv": cluster_cv,

    "mean_noise_fraction": mean_noise,
    "std_noise_fraction": std_noise,

    "baseline_silhouette": baseline_silhouette,
    "mean_silhouette": mean_silhouette,
    "std_silhouette": std_silhouette,

    "mean_ARI_vs_reference": mean_ari_reference,
    "std_ARI_vs_reference": std_ari_reference,

    "mean_AMI_vs_reference": mean_ami_reference,
    "std_AMI_vs_reference": std_ami_reference,

    "mean_pairwise_ARI": mean_pairwise_ari,
    "std_pairwise_ARI": std_pairwise_ari,

    "mean_pairwise_AMI": mean_pairwise_ami,
    "std_pairwise_AMI": std_pairwise_ami
}

summary_df = pd.DataFrame([summary])

summary_path = os.path.join(
    OUTPUT_DIR,
    "step7e_corrected_stability_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\n[11] STABILITY SUMMARY")
print(summary_df.to_string(index=False))


# ================================================================
# [12] VALIDATION FLAGS
# ================================================================

flags = []

# Multiple clusters
flags.append({
    "flag": "MULTI_CLUSTER_STRUCTURE",
    "status": (
        "PASS"
        if baseline_cluster_count >= 2
        else "FAIL"
    ),
    "description":
        "Baseline DBSCAN identifies multiple non-noise clusters."
})

# Noise fraction
flags.append({
    "flag": "NOISE_FRACTION",
    "status": (
        "PASS"
        if 0.01 <= baseline_noise_fraction <= 0.30
        else "REVIEW"
    ),
    "description":
        "Baseline noise fraction is within a practical validation range."
})

# Cluster count stability
flags.append({
    "flag": "CLUSTER_COUNT_STABILITY",
    "status": (
        "PASS"
        if cluster_cv <= MAX_CLUSTER_COUNT_CV
        else "REVIEW"
    ),
    "description":
        "Cluster count variation across independent samples."
})

# Pairwise ARI
flags.append({
    "flag": "PAIRWISE_ARI_STABILITY",
    "status": (
        "PASS"
        if mean_pairwise_ari >= MIN_MEAN_ARI
        else "FAIL"
    ),
    "description":
        "Independent DBSCAN runs show consistent partitions on overlapping sessions."
})

# Pairwise AMI
flags.append({
    "flag": "PAIRWISE_AMI_STABILITY",
    "status": (
        "PASS"
        if mean_pairwise_ami >= MIN_MEAN_AMI
        else "FAIL"
    ),
    "description":
        "Independent DBSCAN runs preserve behavioral information on overlapping sessions."
})

# AMI reference
flags.append({
    "flag": "REFERENCE_AMI_STABILITY",
    "status": (
        "PASS"
        if mean_ami_reference >= MIN_MEAN_AMI
        else "REVIEW"
    ),
    "description":
        "Reference-vs-sample AMI remains sufficiently high."
})

# ARI reference — secondary
flags.append({
    "flag": "REFERENCE_ARI_STABILITY",
    "status": (
        "PASS"
        if mean_ari_reference >= MIN_MEAN_ARI
        else "REVIEW"
    ),
    "description":
        "Reference-vs-sample ARI is treated as secondary evidence because DBSCAN cluster identities can change when density structure is re-estimated."
})

# Noise stability
flags.append({
    "flag": "NOISE_STABILITY",
    "status": (
        "PASS"
        if std_noise <= MAX_NOISE_STD
        else "REVIEW"
    ),
    "description":
        "Noise fraction remains stable across samples."
})

# Silhouette stability
flags.append({
    "flag": "SILHOUETTE_STABILITY",
    "status": (
        "PASS"
        if std_silhouette <= MAX_SILHOUETTE_STD
        else "REVIEW"
    ),
    "description":
        "Silhouette remains stable across independent samples."
})

flags_df = pd.DataFrame(flags)

flags_path = os.path.join(
    OUTPUT_DIR,
    "step7e_corrected_validation_flags.csv"
)

flags_df.to_csv(
    flags_path,
    index=False
)

print("\n[12] VALIDATION FLAGS")
print(flags_df.to_string(index=False))


# ================================================================
# [13] FINAL DECISION
# ================================================================

hard_fail_count = int(
    (flags_df["status"] == "FAIL").sum()
)

review_count = int(
    (flags_df["status"] == "REVIEW").sum()
)

# We deliberately do NOT make the old reference ARI a hard blocker.
# Pairwise stability is the primary stability test.

if hard_fail_count == 0 and review_count == 0:
    final_status = "PASS"
elif hard_fail_count == 0:
    final_status = "PASS_WITH_REVIEW"
else:
    final_status = "REVIEW_REQUIRED"

print("\n[13] FINAL DECISION")
print("Hard FAIL flags:", hard_fail_count)
print("Review flags:", review_count)
print("FINAL STATUS:", final_status)


# ================================================================
# [14] AUDIT
# ================================================================

audit = {
    "step": "7E-R",
    "description":
        "Corrected DBSCAN cluster validation and stability check",

    "source": SOURCE,
    "source_sha256": actual_source_sha,

    "baseline": {
        "eps": EPS,
        "min_samples": MIN_SAMPLES,
        "clusters": baseline_cluster_count,
        "noise_count": baseline_noise_count,
        "noise_fraction": baseline_noise_fraction,
        "silhouette": (
            None
            if np.isnan(baseline_silhouette)
            else float(baseline_silhouette)
        )
    },

    "stability": {
        "seeds": SEEDS,
        "sample_fraction": SAMPLE_FRACTION,
        "mean_clusters_found": float(mean_clusters),
        "std_clusters_found": float(std_clusters),
        "cluster_count_cv": float(cluster_cv),
        "mean_noise_fraction": float(mean_noise),
        "std_noise_fraction": float(std_noise),
        "mean_silhouette": float(mean_silhouette),
        "std_silhouette": float(std_silhouette),
        "mean_ARI_vs_reference": float(mean_ari_reference),
        "mean_AMI_vs_reference": float(mean_ami_reference),
        "mean_pairwise_ARI": float(mean_pairwise_ari),
        "mean_pairwise_AMI": float(mean_pairwise_ami)
    },

    "validation": {
        "hard_fail_count": hard_fail_count,
        "review_count": review_count,
        "final_status": final_status
    },

    "safety": {
        "source_modified": False,
        "labels_used": False,
        "label_leakage": False,
        "session_integrity": True,
        "baseline_assignments_modified": False
    }
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7e_corrected_cluster_validation_audit.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# [15] SOURCE PROTECTION
# ================================================================

source_sha_after = sha256_file(SOURCE)

print("\n[14] SOURCE PROTECTION")
print("SHA before:", actual_source_sha)
print("SHA after: ", source_sha_after)

if source_sha_after != actual_source_sha:
    raise ValueError(
        "STOP — source file changed."
    )

print("Source unchanged: PASS")


# ================================================================
# FINAL
# ================================================================

print("\n" + "=" * 70)
print("STEP 7E-R — COMPLETE")
print("=" * 70)

print("Primary model: DBSCAN")
print("eps:", EPS)
print("min_samples:", MIN_SAMPLES)
print("Reference clusters:", baseline_cluster_count)
print("Reference noise:", baseline_noise_count)
print(
    "Reference noise fraction:",
    round(baseline_noise_fraction, 6)
)

print(
    "Mean pairwise ARI:",
    round(mean_pairwise_ari, 6)
)

print(
    "Mean pairwise AMI:",
    round(mean_pairwise_ami, 6)
)

print(
    "Mean silhouette:",
    round(mean_silhouette, 6)
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nAudit:")
print(audit_path)

print("\nSTATUS:", final_status)

if final_status in ["PASS", "PASS_WITH_REVIEW"]:
    print("NEXT: STEP 7F — ADVANCED CLUSTER CHARACTERIZATION")
else:
    print(
        "NEXT: REVIEW 7E-R RESULTS BEFORE CONTINUING."
    )

STEP 7E-R — CORRECTED DBSCAN STABILITY VALIDATION

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] BASELINE DBSCAN ASSIGNMENTS
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'dbscan_cluster']
Detected ID column: reconstructed_session_id
Detected cluster column: dbscan_cluster
Assignment integrity: PASS

[6] BASELINE STRUCTURE
eps: 0.5
min_samples: 20
Non-noise clusters: 38
Noise sessions: 1676
Noise fraction: 0.141077

[7] BEHAVIORAL FEATURE MATRIX
Behavioral features: 48
Matrix shape: (11880, 48)
Finite matrix: PASS

[8] BASELINE SILHOUETTE
Silhouette: 0.571828

[9] INDEPENDENT SUBSAMPLE STABIL

In [ ]:
# ================================================================
# STEP 7F — ADVANCED CLUSTER CHARACTERIZATION
# ================================================================

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd

from pathlib import Path

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------

SOURCE = Path(
    "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"
)

SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

ASSIGNMENTS = Path(
    "/content/step6_unsupervised_ml/step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

STEP7D_DIR = Path(
    "/content/step7_cluster_profiling/"
    "step7d_detailed_behavioral_profiling"
)

OUT_DIR = Path(
    "/content/step7_cluster_profiling/"
    "step7f_advanced_cluster_characterization"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def detect_id_column(df):
    candidates = [
        "reconstructed_session_id",
        "session_id",
        "session",
        "id"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("STOP — session ID column not found.")


def detect_cluster_column(df):
    candidates = [
        "dbscan_cluster",
        "DBSCAN_cluster",
        "cluster"
    ]
    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        if "cluster" in c.lower():
            return c

    raise ValueError("STOP — DBSCAN cluster column not found.")


# ----------------------------------------------------------------
# [1] SOURCE VERIFICATION
# ----------------------------------------------------------------

print("=" * 70)
print("STEP 7F — ADVANCED CLUSTER CHARACTERIZATION")
print("=" * 70)

if not SOURCE.exists():
    raise FileNotFoundError(f"STOP — source not found: {SOURCE}")

actual_source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE VERIFICATION")
print("Expected SHA:", SOURCE_SHA)
print("Actual SHA:  ", actual_source_sha)

if actual_source_sha != SOURCE_SHA:
    raise ValueError("STOP — source SHA mismatch.")

print("Verification: PASS")


# ----------------------------------------------------------------
# [2] LOAD SOURCE
# ----------------------------------------------------------------

df = pd.read_csv(SOURCE)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError("STOP — unexpected session count.")

source_id_col = detect_id_column(df)

missing_ids = int(df[source_id_col].isna().sum())
duplicate_ids = int(df[source_id_col].duplicated().sum())
unique_ids = int(df[source_id_col].nunique())

print("ID column:", source_id_col)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0 or duplicate_ids != 0 or unique_ids != len(df):
    raise ValueError("STOP — session integrity failure.")

print("Session integrity: PASS")


# ----------------------------------------------------------------
# [3] LABEL LEAKAGE CHECK
# ----------------------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "mitre_techniques",
    "label",
    "target",
    "y"
}

present_forbidden = [
    c for c in df.columns
    if c.lower() in FORBIDDEN_LABELS
]

print("\n[3] LABEL LEAKAGE")
print("Forbidden columns:", present_forbidden)

if present_forbidden:
    raise ValueError(
        f"STOP — forbidden label columns present: {present_forbidden}"
    )

print("Label leakage: NO")


# ----------------------------------------------------------------
# [4] LOAD VERIFIED DBSCAN ASSIGNMENTS
# ----------------------------------------------------------------

print("\n[4] LOADING DBSCAN ASSIGNMENTS")

if not ASSIGNMENTS.exists():
    raise FileNotFoundError(
        f"STOP — DBSCAN assignments not found: {ASSIGNMENTS}"
    )

assign = pd.read_csv(ASSIGNMENTS)

assign_id_col = detect_id_column(assign)
cluster_col = detect_cluster_column(assign)

print("Shape:", assign.shape)
print("Columns:", list(assign.columns))
print("ID column:", assign_id_col)
print("Cluster column:", cluster_col)

if len(assign) != len(df):
    raise ValueError("STOP — assignment row count mismatch.")

if assign[assign_id_col].isna().any():
    raise ValueError("STOP — missing assignment IDs.")

if assign[assign_id_col].duplicated().any():
    raise ValueError("STOP — duplicate assignment IDs.")

source_ids = set(df[source_id_col].astype(str))
assignment_ids = set(assign[assign_id_col].astype(str))

if source_ids != assignment_ids:
    raise ValueError("STOP — assignment IDs do not match source.")

print("Assignment integrity: PASS")


# ----------------------------------------------------------------
# [5] MERGE
# ----------------------------------------------------------------

work = df.copy()

work[source_id_col] = work[source_id_col].astype(str)
assign[assign_id_col] = assign[assign_id_col].astype(str)

assign_for_merge = assign[[assign_id_col, cluster_col]].copy()

assign_for_merge = assign_for_merge.rename(
    columns={assign_id_col: source_id_col}
)

work = work.merge(
    assign_for_merge,
    on=source_id_col,
    how="left",
    validate="one_to_one"
)

if work[cluster_col].isna().any():
    raise ValueError("STOP — missing DBSCAN cluster assignments.")

print("\n[5] ASSIGNMENT MERGE")
print("Merged rows:", len(work))
print("Merge validation: PASS")


# ----------------------------------------------------------------
# [6] BEHAVIORAL FEATURE SAFETY
# ----------------------------------------------------------------

print("\n[6] BEHAVIORAL FEATURE SAFETY")

EXCLUDED_COLUMNS = {
    source_id_col,
    "session_id",
    "session_start_time",
    "session_end_time",
    "reconstructed_session_id",
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
    "mitre_techniques",
    "label",
    "target",
    "y",
    cluster_col
}

numeric_features = work.select_dtypes(
    include=[np.number]
).columns.tolist()

behavioral_features = [
    c for c in numeric_features
    if c not in EXCLUDED_COLUMNS
    and "cluster" not in c.lower()
    and "label" not in c.lower()
    and "target" not in c.lower()
]

# Remove all-missing features
behavioral_features = [
    c for c in behavioral_features
    if not work[c].isna().all()
]

# Remove non-finite-only features
behavioral_features = [
    c for c in behavioral_features
    if np.isfinite(
        pd.to_numeric(work[c], errors="coerce")
    ).any()
]

print("Behavioral numeric features:", len(behavioral_features))

if len(behavioral_features) == 0:
    raise ValueError("STOP — no usable behavioral features.")

print("Feature safety: PASS")


# ----------------------------------------------------------------
# [7] CLUSTER SIZE + NOISE
# ----------------------------------------------------------------

cluster_sizes = (
    work.groupby(cluster_col)
    .size()
    .reset_index(name="session_count")
)

cluster_sizes["percentage"] = (
    cluster_sizes["session_count"] / len(work) * 100
)

cluster_sizes["is_noise"] = (
    cluster_sizes[cluster_col] == -1
)

cluster_sizes["cluster_rank_by_size"] = (
    cluster_sizes["session_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

cluster_sizes = cluster_sizes.sort_values(
    ["is_noise", "session_count"],
    ascending=[False, False]
)

cluster_sizes.to_csv(
    OUT_DIR / "step7f_cluster_size_characterization.csv",
    index=False
)

noise_count = int((work[cluster_col] == -1).sum())
noise_fraction = noise_count / len(work)

print("\n[7] CLUSTER STRUCTURE")
print("Clusters including noise:", work[cluster_col].nunique())
print(
    "Non-noise clusters:",
    work.loc[work[cluster_col] != -1, cluster_col].nunique()
)
print("Noise sessions:", noise_count)
print("Noise fraction:", round(noise_fraction, 6))


# ----------------------------------------------------------------
# [8] CLUSTER BEHAVIORAL Z-SCORES
# ----------------------------------------------------------------

print("\n[8] BUILDING CLUSTER BEHAVIORAL SIGNATURES")

global_means = work[behavioral_features].mean()
global_stds = work[behavioral_features].std(ddof=0)

global_stds = global_stds.replace(0, np.nan)

profile_rows = []

for cluster_id, group in work.groupby(cluster_col):

    row = {
        "dbscan_cluster": cluster_id,
        "session_count": len(group),
        "percentage": len(group) / len(work) * 100,
        "is_noise": bool(cluster_id == -1)
    }

    for feature in behavioral_features:

        cluster_mean = group[feature].mean()

        global_mean = global_means[feature]
        global_std = global_stds[feature]

        if pd.isna(global_std) or global_std == 0:
            z = 0.0
        else:
            z = (cluster_mean - global_mean) / global_std

        row[f"{feature}__mean"] = cluster_mean
        row[f"{feature}__zscore"] = z

    profile_rows.append(row)

profile = pd.DataFrame(profile_rows)

profile.to_csv(
    OUT_DIR / "step7f_advanced_behavioral_profiles.csv",
    index=False
)

print(
    "Saved:",
    OUT_DIR / "step7f_advanced_behavioral_profiles.csv"
)


# ----------------------------------------------------------------
# [9] TOP POSITIVE / NEGATIVE BEHAVIORAL FEATURES
# ----------------------------------------------------------------

print("\n[9] EXTRACTING CLUSTER BEHAVIORAL SIGNATURES")

signature_rows = []

for _, row in profile.iterrows():

    cluster_id = row["dbscan_cluster"]

    zscores = {}

    for feature in behavioral_features:
        value = row.get(f"{feature}__zscore", np.nan)
        if pd.notna(value):
            zscores[feature] = float(value)

    ordered = sorted(
        zscores.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )

    top_positive = [
        (f, round(z, 4))
        for f, z in ordered
        if z > 0
    ][:10]

    top_negative = [
        (f, round(z, 4))
        for f, z in ordered
        if z < 0
    ][:10]

    signature_rows.append({
        "dbscan_cluster": cluster_id,
        "session_count": int(row["session_count"]),
        "is_noise": bool(row["is_noise"]),
        "top_positive_features": json.dumps(top_positive),
        "top_negative_features": json.dumps(top_negative),
        "max_absolute_zscore": (
            round(abs(ordered[0][1]), 4)
            if ordered else 0
        )
    })

signatures = pd.DataFrame(signature_rows)

signatures.to_csv(
    OUT_DIR / "step7f_cluster_behavioral_signatures.csv",
    index=False
)

print(
    "Saved:",
    OUT_DIR / "step7f_cluster_behavioral_signatures.csv"
)


# ----------------------------------------------------------------
# [10] BEHAVIORAL INTENSITY CHARACTERIZATION
# ----------------------------------------------------------------

print("\n[10] BUILDING BEHAVIORAL INTENSITY SCORES")

intensity_rows = []

for _, row in profile.iterrows():

    cluster_id = row["dbscan_cluster"]

    abs_z = []

    for feature in behavioral_features:
        z = row.get(f"{feature}__zscore", np.nan)

        if pd.notna(z):
            abs_z.append(abs(float(z)))

    if abs_z:
        mean_abs_z = float(np.mean(abs_z))
        max_abs_z = float(np.max(abs_z))
        high_deviation_fraction = float(
            np.mean(np.array(abs_z) >= 2.0)
        )
    else:
        mean_abs_z = 0
        max_abs_z = 0
        high_deviation_fraction = 0

    if cluster_id == -1:
        behavior_class = "noise_or_anomalous"
    elif high_deviation_fraction >= 0.20:
        behavior_class = "highly_distinct"
    elif high_deviation_fraction >= 0.10:
        behavior_class = "distinct"
    else:
        behavior_class = "typical"

    intensity_rows.append({
        "dbscan_cluster": cluster_id,
        "session_count": int(row["session_count"]),
        "is_noise": bool(row["is_noise"]),
        "mean_absolute_zscore": round(mean_abs_z, 6),
        "max_absolute_zscore": round(max_abs_z, 6),
        "high_deviation_feature_fraction": round(
            high_deviation_fraction, 6
        ),
        "behavioral_distinctiveness": behavior_class
    })

intensity = pd.DataFrame(intensity_rows)

intensity.to_csv(
    OUT_DIR / "step7f_behavioral_intensity_characterization.csv",
    index=False
)

print(
    "Saved:",
    OUT_DIR / "step7f_behavioral_intensity_characterization.csv"
)


# ----------------------------------------------------------------
# [11] REUSABLE CLUSTER REFERENCE
# ----------------------------------------------------------------

print("\n[11] BUILDING REUSABLE CLUSTER REFERENCE")

reference = {
    "step": "7F",
    "purpose": "Advanced DBSCAN behavioral cluster characterization",
    "source": str(SOURCE),
    "source_sha256": actual_source_sha,
    "primary_model": "DBSCAN",
    "dbscan_parameters": {
        "eps": 0.5,
        "min_samples": 20
    },
    "session_count": int(len(work)),
    "behavioral_feature_count": int(len(behavioral_features)),
    "behavioral_features": behavioral_features,
    "noise_cluster_label": -1,
    "noise_count": noise_count,
    "noise_fraction": noise_fraction,
    "non_noise_cluster_count": int(
        work.loc[work[cluster_col] != -1, cluster_col].nunique()
    ),
    "cluster_reference_files": [
        "step7f_cluster_size_characterization.csv",
        "step7f_advanced_behavioral_profiles.csv",
        "step7f_cluster_behavioral_signatures.csv",
        "step7f_behavioral_intensity_characterization.csv"
    ],
    "future_log_mapping_note": (
        "Future sessions must use the same validated preprocessing "
        "and behavioral feature definitions before DBSCAN assignment."
    )
}

with open(
    OUT_DIR / "step7f_reusable_cluster_reference.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(reference, f, indent=2)


# ----------------------------------------------------------------
# [12] AUDIT
# ----------------------------------------------------------------

source_sha_before = actual_source_sha
source_sha_after = sha256_file(SOURCE)

audit = {
    "step": "7F",
    "status": "PASS",
    "source": str(SOURCE),
    "source_sha_before": source_sha_before,
    "source_sha_after": source_sha_after,
    "source_unchanged": source_sha_before == source_sha_after,
    "session_count": int(len(work)),
    "missing_ids": missing_ids,
    "duplicate_ids": duplicate_ids,
    "unique_ids": unique_ids,
    "label_leakage": False,
    "forbidden_columns": present_forbidden,
    "primary_model": "DBSCAN",
    "eps": 0.5,
    "min_samples": 20,
    "behavioral_feature_count": len(behavioral_features),
    "non_noise_clusters": int(
        work.loc[work[cluster_col] != -1, cluster_col].nunique()
    ),
    "noise_count": noise_count,
    "noise_fraction": noise_fraction,
    "outputs": [
        "step7f_cluster_size_characterization.csv",
        "step7f_advanced_behavioral_profiles.csv",
        "step7f_cluster_behavioral_signatures.csv",
        "step7f_behavioral_intensity_characterization.csv",
        "step7f_reusable_cluster_reference.json"
    ]
}

audit_path = OUT_DIR / "step7f_advanced_cluster_characterization_audit.json"

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit, f, indent=2)


# ----------------------------------------------------------------
# [13] FINAL VERIFICATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 7F — COMPLETE")
print("=" * 70)

print("\nSource SHA unchanged:",
      source_sha_before == source_sha_after)

print("Session integrity: PASS")
print("Label leakage: NO")
print("Behavioral features:", len(behavioral_features))
print("Primary model: DBSCAN")
print(
    "Non-noise clusters:",
    work.loc[work[cluster_col] != -1, cluster_col].nunique()
)
print("Noise sessions:", noise_count)
print("Noise fraction:", round(noise_fraction, 6))

print("\nOutput directory:")
print(OUT_DIR)

print("\nAudit:")
print(audit_path)

print("\nAudit SHA-256:")
print(sha256_file(audit_path))

print("\nSTATUS: PASS")
print("NEXT: STEP 7G — PRODUCTION-READY CLUSTER ASSIGNMENT PIPELINE")

STEP 7F — ADVANCED CLUSTER CHARACTERIZATION

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[3] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[4] LOADING DBSCAN ASSIGNMENTS
Shape: (11880, 2)
Columns: ['reconstructed_session_id', 'dbscan_cluster']
ID column: reconstructed_session_id
Cluster column: dbscan_cluster
Assignment integrity: PASS

[5] ASSIGNMENT MERGE
Merged rows: 11880
Merge validation: PASS

[6] BEHAVIORAL FEATURE SAFETY
Behavioral numeric features: 48
Feature safety: PASS

[7] CLUSTER STRUCTURE
Clusters including noise: 39
Non-noise clusters: 38
Noise sessions: 1676
Noise fraction: 0.141077

[8] BUILDING CLUSTER BEHAVIORAL SIGNATURES
Saved: /content/step7_cluster_pro

In [ ]:
# ================================================================
# STEP 7G-R — FIXED PRODUCTION-READY DBSCAN CLUSTER PIPELINE
# ================================================================

import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances


# ================================================================
# CONFIGURATION
# ================================================================

SOURCE_PATH = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

REFERENCE_PATH = (
    "/content/step7_cluster_profiling/"
    "step7d_detailed_behavioral_profiling/"
    "step7d_behavioral_reference.json"
)

SCALER_PATH = (
    "/content/step6_unsupervised_ml/"
    "step6b_preprocessing/"
    "standard_scaler_params.json"
)

ASSIGNMENTS_PATH = (
    "/content/step6_unsupervised_ml/"
    "step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7g_production_cluster_pipeline"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

CENTROIDS_PATH = os.path.join(
    OUTPUT_DIR,
    "step7g_dbscan_behavioral_reference_centroids.csv"
)

ASSIGNMENT_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "step7g_reference_session_cluster_assignments.csv"
)

REFERENCE_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "step7g_production_behavioral_reference.json"
)

AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "step7g_production_cluster_pipeline_audit.json"
)


# ================================================================
# HELPERS
# ================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def detect_id_column(df):
    candidates = [
        "reconstructed_session_id",
        "session_id",
        "session ID",
        "id"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError("STOP — session ID column not found.")


def detect_cluster_column(df):
    candidates = [
        "dbscan_cluster",
        "DBSCAN_cluster",
        "cluster"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError("STOP — DBSCAN cluster column not found.")


# ================================================================
# FIXED NEAREST-CENTROID FUNCTION
# ================================================================

def nearest_centroid_distances(matrix, centroids):
    """
    Returns:
        nearest_idx  -> integer index of nearest centroid
        nearest_dist -> Euclidean distance to nearest centroid

    FIX:
    Avoids the previous numpy/int .sum() failure by using
    sklearn pairwise_distances.
    """

    matrix = np.asarray(matrix, dtype=np.float64)
    centroids = np.asarray(centroids, dtype=np.float64)

    if matrix.ndim != 2:
        raise ValueError(
            f"STOP — matrix must be 2D, got shape={matrix.shape}"
        )

    if centroids.ndim != 2:
        raise ValueError(
            f"STOP — centroids must be 2D, got shape={centroids.shape}"
        )

    if matrix.shape[1] != centroids.shape[1]:
        raise ValueError(
            "STOP — feature dimension mismatch: "
            f"matrix={matrix.shape[1]}, "
            f"centroids={centroids.shape[1]}"
        )

    if not np.isfinite(matrix).all():
        raise ValueError("STOP — matrix contains non-finite values.")

    if not np.isfinite(centroids).all():
        raise ValueError("STOP — centroids contain non-finite values.")

    distances = pairwise_distances(
        matrix,
        centroids,
        metric="euclidean"
    )

    nearest_idx = np.argmin(distances, axis=1)
    nearest_dist = distances[
        np.arange(distances.shape[0]),
        nearest_idx
    ]

    return nearest_idx.astype(int), nearest_dist.astype(float)


# ================================================================
# [1] SOURCE VERIFICATION
# ================================================================

print("=" * 70)
print("STEP 7G-R — PRODUCTION-READY DBSCAN CLUSTER ASSIGNMENT PIPELINE")
print("=" * 70)

print("\n[1] SOURCE VERIFICATION")

if not os.path.exists(SOURCE_PATH):
    raise FileNotFoundError(
        f"STOP — source file not found:\n{SOURCE_PATH}"
    )

source_sha_before = sha256_file(SOURCE_PATH)

print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", source_sha_before)

if source_sha_before != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source SHA mismatch."
    )

print("Verification: PASS")


# ================================================================
# [2] LOAD SOURCE
# ================================================================

print("\n[2] SOURCE DATASET")

df = pd.read_csv(SOURCE_PATH)

print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, got {len(df)}"
    )


# ================================================================
# [3] SESSION INTEGRITY
# ================================================================

print("\n[3] SESSION INTEGRITY")

ID_COL = detect_id_column(df)

missing_ids = int(df[ID_COL].isna().sum())
duplicate_ids = int(df[ID_COL].duplicated().sum())
unique_ids = int(df[ID_COL].nunique())

print("ID column:", ID_COL)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

if unique_ids != len(df):
    raise ValueError("STOP — session IDs are not unique.")

print("Session integrity: PASS")


# ================================================================
# [4] LABEL LEAKAGE
# ================================================================

print("\n[4] LABEL LEAKAGE")

FORBIDDEN = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique"
}

forbidden_present = [
    c for c in df.columns
    if c.lower() in {x.lower() for x in FORBIDDEN}
]

print("Forbidden columns present:", forbidden_present)

if forbidden_present:
    raise ValueError(
        "STOP — label-derived columns detected."
    )

print("Label leakage: NO")


# ================================================================
# [5] LOAD BEHAVIORAL REFERENCE
# ================================================================

print("\n[5] BEHAVIORAL REFERENCE")

if not os.path.exists(REFERENCE_PATH):
    raise FileNotFoundError(
        f"STOP — behavioral reference not found:\n{REFERENCE_PATH}"
    )

with open(REFERENCE_PATH, "r") as f:
    behavioral_reference = json.load(f)

print("Loaded:", REFERENCE_PATH)


# ================================================================
# [6] DETERMINE BEHAVIORAL FEATURES
# ================================================================

print("\n[6] BEHAVIORAL FEATURE MATRIX")

all_missing_candidates = {
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean"
}

excluded_columns = {
    ID_COL,
    "session_start_time",
    "session_end_time",
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique"
}

behavioral_features = []

for c in df.columns:

    if c in excluded_columns:
        continue

    if c in all_missing_candidates:
        continue

    if pd.api.types.is_numeric_dtype(df[c]):
        behavioral_features.append(c)

if len(behavioral_features) != 48:
    print(
        "WARNING — detected behavioral features:",
        len(behavioral_features)
    )

print("Behavioral features detected:", len(behavioral_features))

X = df[behavioral_features].copy()

# Explicitly handle remaining missing values using
# training/reference-compatible median imputation.
missing_before = int(X.isna().sum().sum())

if missing_before > 0:
    print(
        "Missing numeric values before imputation:",
        missing_before
    )

    X = X.fillna(X.median(numeric_only=True))

missing_after = int(X.isna().sum().sum())

if missing_after != 0:
    raise ValueError(
        "STOP — missing values remain after imputation."
    )

X_matrix = X.to_numpy(dtype=np.float64)

if not np.isfinite(X_matrix).all():
    raise ValueError(
        "STOP — behavioral matrix contains non-finite values."
    )

print("Final behavioral feature count:", len(behavioral_features))
print("Matrix shape:", X_matrix.shape)
print("Finite matrix: PASS")


# ================================================================
# [7] LOAD SCALER ARTIFACT
# ================================================================

print("\n[7] SCALER ARTIFACT")

if not os.path.exists(SCALER_PATH):
    raise FileNotFoundError(
        f"STOP — scaler artifact not found:\n{SCALER_PATH}"
    )

with open(SCALER_PATH, "r") as f:
    scaler_params = json.load(f)


# Robustly detect scaler parameter format.
if "mean" in scaler_params and "scale" in scaler_params:
    scaler_mean = np.asarray(
        scaler_params["mean"],
        dtype=np.float64
    )

    scaler_scale = np.as# STEP 8C-7-3B-R1 — ATTRIBUTION NULL DIAGNOSTIC
array(
        scaler_params["scale"],
        dtype=np.float64
    )

elif (
    "mean_" in scaler_params
    and "scale_" in scaler_params
):
    scaler_mean = np.asarray(
        scaler_params["mean_"],
        dtype=np.float64
    )

    scaler_scale = np.asarray(
        scaler_params["scale_"],
        dtype=np.float64
    )

else:
    raise ValueError(
        "STOP — unsupported scaler artifact format."
    )


if len(scaler_mean) != len(behavioral_features):
    raise ValueError(
        "STOP — scaler feature count mismatch."
    )

if len(scaler_scale) != len(behavioral_features):
    raise ValueError(
        "STOP — scaler scale count mismatch."
    )

if np.any(scaler_scale == 0):
    raise ValueError(
        "STOP — scaler contains zero scale."
    )

X_scaled = (
    X_matrix - scaler_mean
) / scaler_scale

if not np.isfinite(X_scaled).all():
    raise ValueError(
        "STOP — scaled matrix contains non-finite values."
    )

print("Scaler compatibility: PASS")
print("Scaled matrix:", X_scaled.shape)


# ================================================================
# [8] LOAD DBSCAN REFERENCE ASSIGNMENTS
# ================================================================

print("\n[8] DBSCAN REFERENCE ASSIGNMENTS")

assignments = pd.read_csv(ASSIGNMENTS_PATH)

ASSIGN_ID_COL = detect_id_column(assignments)
CLUSTER_COL = detect_cluster_column(assignments)

print("Shape:", assignments.shape)
print("ID column:", ASSIGN_ID_COL)
print("Cluster column:", CLUSTER_COL)

if len(assignments) != len(df):
    raise ValueError(
        "STOP — assignment row count mismatch."
    )

if assignments[ASSIGN_ID_COL].isna().any():
    raise ValueError(
        "STOP — missing assignment session IDs."
    )

if assignments[ASSIGN_ID_COL].duplicated().any():
    raise ValueError(
        "STOP — duplicate assignment session IDs."
    )

source_ids = set(df[ID_COL].astype(str))
assignment_ids = set(
    assignments[ASSIGN_ID_COL].astype(str)
)

if source_ids != assignment_ids:
    raise ValueError(
        "STOP — assignment session IDs do not match source."
    )

print("Assignment integrity: PASS")


# ================================================================
# [9] ALIGN ASSIGNMENTS TO SOURCE ORDER
# ================================================================

print("\n[9] ALIGNING ASSIGNMENTS")

assignment_map = (
    assignments
    .set_index(ASSIGN_ID_COL)[CLUSTER_COL]
)

ordered_clusters = (
    df[ID_COL]
    .astype(str)
    .map(assignment_map)
)

if ordered_clusters.isna().any():
    raise ValueError(
        "STOP — some source sessions have no cluster assignment."
    )

reference_clusters = ordered_clusters.to_numpy()

print("Aligned assignments:", len(reference_clusters))


# ================================================================
# [10] REFERENCE STRUCTURE
# ================================================================

print("\n[10] DBSCAN REFERENCE STRUCTURE")

unique_clusters = np.sort(
    pd.unique(reference_clusters)
)

non_noise_clusters = [
    c for c in unique_clusters
    if c != -1
]

noise_count = int(
    np.sum(reference_clusters == -1)
)

noise_fraction = noise_count / len(reference_clusters)

print(
    "Clusters including noise:",
    len(unique_clusters)
)

print(
    "Non-noise clusters:",
    len(non_noise_clusters)
)

print(
    "Noise sessions:",
    noise_count
)

print(
    "Noise fraction:",
    f"{noise_fraction:.6f}"
)


# ================================================================
# [11] BUILD REFERENCE CENTROIDS
# ================================================================

print("\n[11] REFERENCE CENTROIDS")

centroid_rows = []

for cluster_id in non_noise_clusters:

    mask = reference_clusters == cluster_id

    cluster_matrix = X_scaled[mask]

    if len(cluster_matrix) == 0:
        continue

    centroid = cluster_matrix.mean(axis=0)

    row = {
        "dbscan_cluster": int(cluster_id),
        "session_count": int(mask.sum())
    }

    for feature, value in zip(
        behavioral_features,
        centroid
    ):
        row[feature] = float(value)

    centroid_rows.append(row)

centroids_df = pd.DataFrame(centroid_rows)

if len(centroids_df) != len(non_noise_clusters):
    raise ValueError(
        "STOP — centroid count mismatch."
    )

centroids_df.to_csv(
    CENTROIDS_PATH,
    index=False
)

print(
    "Clusters represented:",
    len(centroids_df)
)

print(
    "Saved:",
    CENTROIDS_PATH
)


# ================================================================
# [12] FIXED NEAREST-CENTROID ASSIGNMENT
# ================================================================

print("\n[12] NEAREST REFERENCE CLUSTER")

centroid_cluster_ids = centroids_df[
    "dbscan_cluster"
].to_numpy(dtype=int)

centroid_matrix = centroids_df[
    behavioral_features
].to_numpy(dtype=np.float64)

nearest_idx, nearest_dist = nearest_centroid_distances(
    X_scaled,
    centroid_matrix
)

nearest_cluster = centroid_cluster_ids[
    nearest_idx
]

if len(nearest_cluster) != len(df):
    raise ValueError(
        "STOP — nearest-cluster assignment count mismatch."
    )

if not np.isfinite(nearest_dist).all():
    raise ValueError(
        "STOP — nearest centroid distances are non-finite."
    )

print(
    "Nearest-centroid assignments:",
    len(nearest_cluster)
)

print(
    "Distance range:",
    f"{nearest_dist.min():.6f}",
    "to",
    f"{nearest_dist.max():.6f}"
)


# ================================================================
# [13] PRODUCTION ASSIGNMENT LOGIC
# ================================================================

print("\n[13] PRODUCTION ASSIGNMENT LOGIC")

# IMPORTANT:
# DBSCAN does NOT support native prediction for new data.
# Therefore this pipeline preserves the reference DBSCAN noise
# concept and exposes nearest-reference-cluster information
# separately instead of pretending it is a true DBSCAN prediction.

production_assignments = pd.DataFrame({
    ID_COL: df[ID_COL].values,
    "dbscan_reference_cluster": reference_clusters.astype(int),
    "nearest_reference_cluster": nearest_cluster.astype(int),
    "nearest_reference_distance": nearest_dist.astype(float),
    "is_reference_noise": (
        reference_clusters == -1
    )
})

production_assignments.to_csv(
    ASSIGNMENT_OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    ASSIGNMENT_OUTPUT_PATH
)


# ================================================================
# [14] BUILD REUSABLE PRODUCTION REFERENCE
# ================================================================

print("\n[14] BUILDING REUSABLE PRODUCTION REFERENCE")

production_reference = {
    "source_sha256": source_sha_before,
    "primary_model": "DBSCAN",
    "dbscan_parameters": {
        "eps": 0.5,
        "min_samples": 20
    },
    "session_count": int(len(df)),
    "behavioral_feature_count": int(
        len(behavioral_features)
    ),
    "behavioral_features": behavioral_features,
    "noise_label": -1,
    "reference_non_noise_clusters": [
        int(x) for x in non_noise_clusters
    ],
    "reference_noise_count": noise_count,
    "reference_noise_fraction": float(
        noise_fraction
    ),
    "centroid_artifact": CENTROIDS_PATH,
    "assignment_artifact": ASSIGNMENT_OUTPUT_PATH,
    "note": (
        "DBSCAN has no native predict method. "
        "Nearest-reference-cluster distance is provided "
        "as a production-compatible behavioral similarity "
        "signal and must not be interpreted as a true DBSCAN "
        "prediction."
    )
}

with open(
    REFERENCE_OUTPUT_PATH,
    "w"
) as f:
    json.dump(
        production_reference,
        f,
        indent=2
    )

print(
    "Saved:",
    REFERENCE_OUTPUT_PATH
)


# ================================================================
# [15] SOURCE PROTECTION
# ================================================================

print("\n[15] SOURCE PROTECTION")

source_sha_after = sha256_file(SOURCE_PATH)

print("SHA before:", source_sha_before)
print("SHA after: ", source_sha_after)

if source_sha_after != source_sha_before:
    raise ValueError(
        "STOP — source file changed."
    )

print("Source unchanged: PASS")


# ================================================================
# [16] AUDIT
# ================================================================

audit = {
    "step": "7G-R",
    "status": "PASS",
    "source_path": SOURCE_PATH,
    "source_sha256": source_sha_after,
    "source_modified": False,
    "rows": int(len(df)),
    "session_id_column": ID_COL,
    "session_integrity": "PASS",
    "label_leakage": "NO",
    "behavioral_feature_count": int(
        len(behavioral_features)
    ),
    "behavioral_features": behavioral_features,
    "dbscan_eps": 0.5,
    "dbscan_min_samples": 20,
    "reference_non_noise_clusters": int(
        len(non_noise_clusters)
    ),
    "reference_noise_count": noise_count,
    "reference_noise_fraction": float(
        noise_fraction
    ),
    "centroids_saved": True,
    "nearest_centroid_assignment_saved": True,
    "native_dbscan_prediction_available": False,
    "production_similarity_signal": (
        "nearest_reference_cluster_distance"
    ),
    "outputs": {
        "centroids": CENTROIDS_PATH,
        "assignments": ASSIGNMENT_OUTPUT_PATH,
        "reference": REFERENCE_OUTPUT_PATH
    }
}

with open(
    AUDIT_PATH,
    "w"
) as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(AUDIT_PATH)


# ================================================================
# FINAL
# ================================================================

print("\n" + "=" * 70)
print("STEP 7G-R — COMPLETE")
print("=" * 70)

print("Source unchanged: PASS")
print("Session integrity: PASS")
print("Label leakage: NO")
print("Behavioral features:", len(behavioral_features))
print("Primary model: DBSCAN")
print("Reference clusters:", len(non_noise_clusters))
print("Reference noise:", noise_count)
print("Reference noise fraction:", f"{noise_fraction:.6f}")

print("\nCentroids:")
print(CENTROIDS_PATH)

print("\nAssignments:")
print(ASSIGNMENT_OUTPUT_PATH)

print("\nProduction reference:")
print(REFERENCE_OUTPUT_PATH)

print("\nAudit:")
print(AUDIT_PATH)

print("Audit SHA-256:")
print(audit_sha)

print("\nSTATUS: PASS")
print("NEXT: STEP 7H — PRODUCTION INFERENCE VALIDATION")

STEP 7G-R — PRODUCTION-READY DBSCAN CLUSTER ASSIGNMENT PIPELINE

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] BEHAVIORAL REFERENCE
Loaded: /content/step7_cluster_profiling/step7d_detailed_behavioral_profiling/step7d_behavioral_reference.json

[6] BEHAVIORAL FEATURE MATRIX
Behavioral features detected: 48
Final behavioral feature count: 48
Matrix shape: (11880, 48)
Finite matrix: PASS

[7] SCALER ARTIFACT


TypeError: float() argument must be a string or a real number, not 'dict'

In [ ]:
# ================================================================
# STEP 7G-R2 — PRODUCTION-READY DBSCAN CLUSTER ASSIGNMENT PIPELINE
# Robust scaler-artifact compatibility
# ================================================================

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore")

# ================================================================
# CONFIGURATION
# ================================================================

SOURCE_PATH = (
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

REFERENCE_PATH = (
    "/content/step7_cluster_profiling/"
    "step7d_detailed_behavioral_profiling/"
    "step7d_behavioral_reference.json"
)

SCALER_PATH = (
    "/content/step6_unsupervised_ml/"
    "step6b_preprocessing/"
    "standard_scaler_params.json"
)

ASSIGNMENTS_PATH = (
    "/content/step6_unsupervised_ml/"
    "step6d_dbscan/"
    "dbscan_cluster_assignments.csv"
)

OUTPUT_DIR = (
    "/content/step7_cluster_profiling/"
    "step7g_production_cluster_pipeline"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# HELPERS
# ================================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def first_existing(d, keys):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return None


def flatten_numeric(value):
    """
    Convert scaler values stored as:
      - list
      - numpy-like list
      - dict of feature -> value
      - nested dict
    into a flat numeric mapping/list.
    """

    if isinstance(value, dict):
        # Most common case:
        # {"feature_a": 1.2, "feature_b": 3.4}
        if all(
            not isinstance(v, (dict, list, tuple))
            for v in value.values()
        ):
            return value

        # Nested format:
        # {"features": {...}}
        for nested_key in [
            "values",
            "params",
            "data",
            "features",
            "parameters",
        ]:
            if nested_key in value:
                return flatten_numeric(value[nested_key])

    if isinstance(value, (list, tuple)):
        return np.asarray(value, dtype=np.float64)

    if np.isscalar(value):
        return np.asarray([float(value)], dtype=np.float64)

    return value


def extract_scaler_arrays(scaler_json, feature_names):
    """
    Robustly extract scaler mean and scale from several JSON formats.
    """

    # ------------------------------------------------------------
    # Search recursively for candidate keys
    # ------------------------------------------------------------

    mean_candidate = None
    scale_candidate = None

    def recursive_find(obj):
        nonlocal mean_candidate, scale_candidate

        if isinstance(obj, dict):

            for key, value in obj.items():
                key_l = str(key).lower()

                if mean_candidate is None and key_l in {
                    "mean",
                    "mean_",
                    "mean_values",
                    "scaler_mean",
                    "center",
                    "location",
                }:
                    mean_candidate = value

                if scale_candidate is None and key_l in {
                    "scale",
                    "scale_",
                    "scale_values",
                    "scaler_scale",
                    "std",
                    "std_",
                    "standard_deviation",
                }:
                    scale_candidate = value

            for value in obj.values():
                if isinstance(value, (dict, list)):
                    recursive_find(value)

        elif isinstance(obj, list):
            for item in obj:
                if isinstance(item, (dict, list)):
                    recursive_find(item)

    recursive_find(scaler_json)

    if mean_candidate is None:
        raise ValueError(
            "STOP — Could not locate scaler mean values."
        )

    if scale_candidate is None:
        raise ValueError(
            "STOP — Could not locate scaler scale values."
        )

    # ------------------------------------------------------------
    # Convert mean
    # ------------------------------------------------------------

    mean_obj = flatten_numeric(mean_candidate)
    scale_obj = flatten_numeric(scale_candidate)

    # ------------------------------------------------------------
    # Dictionary format: feature -> value
    # ------------------------------------------------------------

    if isinstance(mean_obj, dict):

        missing_mean = [
            f for f in feature_names
            if f not in mean_obj
        ]

        if missing_mean:
            raise ValueError(
                "STOP — Scaler mean missing features: "
                + ", ".join(missing_mean[:20])
            )

        scaler_mean = np.asarray(
            [float(mean_obj[f]) for f in feature_names],
            dtype=np.float64
        )

    else:

        scaler_mean = np.asarray(
            mean_obj,
            dtype=np.float64
        ).reshape(-1)

    # ------------------------------------------------------------
    # Convert scale
    # ------------------------------------------------------------

    if isinstance(scale_obj, dict):

        missing_scale = [
            f for f in feature_names
            if f not in scale_obj
        ]

        if missing_scale:
            raise ValueError(
                "STOP — Scaler scale missing features: "
                + ", ".join(missing_scale[:20])
            )

        scaler_scale = np.asarray(
            [float(scale_obj[f]) for f in feature_names],
            dtype=np.float64
        )

    else:

        scaler_scale = np.asarray(
            scale_obj,
            dtype=np.float64
        ).reshape(-1)

    # ------------------------------------------------------------
    # Validate dimensions
    # ------------------------------------------------------------

    if len(scaler_mean) != len(feature_names):
        raise ValueError(
            f"STOP — scaler mean length {len(scaler_mean)} "
            f"!= feature count {len(feature_names)}"
        )

    if len(scaler_scale) != len(feature_names):
        raise ValueError(
            f"STOP — scaler scale length {len(scaler_scale)} "
            f"!= feature count {len(feature_names)}"
        )

    if not np.isfinite(scaler_mean).all():
        raise ValueError(
            "STOP — scaler mean contains non-finite values."
        )

    if not np.isfinite(scaler_scale).all():
        raise ValueError(
            "STOP — scaler scale contains non-finite values."
        )

    # StandardScaler never allows zero scale in a way that would
    # produce division-by-zero; nevertheless guard against it.
    scaler_scale = np.where(
        scaler_scale == 0,
        1.0,
        scaler_scale
    )

    return scaler_mean, scaler_scale


def nearest_centroid_distances(X, centroids):
    """
    Safe nearest-centroid calculation.

    Returns:
      nearest_index
      nearest_distance
    """

    X = np.asarray(X, dtype=np.float64)
    centroids = np.asarray(centroids, dtype=np.float64)

    if X.ndim != 2:
        raise ValueError(
            f"STOP — X must be 2D, got shape {X.shape}"
        )

    if centroids.ndim != 2:
        raise ValueError(
            f"STOP — centroids must be 2D, got shape {centroids.shape}"
        )

    if X.shape[1] != centroids.shape[1]:
        raise ValueError(
            "STOP — X and centroid dimensions do not match: "
            f"{X.shape[1]} vs {centroids.shape[1]}"
        )

    # Use sklearn's implementation instead of fragile broadcasting.
    distances = pairwise_distances(
        X,
        centroids,
        metric="euclidean"
    )

    nearest_idx = np.argmin(distances, axis=1)
    nearest_dist = distances[
        np.arange(len(X)),
        nearest_idx
    ]

    return nearest_idx, nearest_dist


# ================================================================
# [1] SOURCE VERIFICATION
# ================================================================

print("=" * 70)
print("STEP 7G-R2 — PRODUCTION-READY DBSCAN CLUSTER ASSIGNMENT PIPELINE")
print("=" * 70)

if not os.path.exists(SOURCE_PATH):
    raise FileNotFoundError(
        f"STOP — source not found: {SOURCE_PATH}"
    )

source_sha_before = sha256_file(SOURCE_PATH)

print("\n[1] SOURCE VERIFICATION")
print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", source_sha_before)

if source_sha_before != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source SHA mismatch."
    )

print("Verification: PASS")


# ================================================================
# [2] SOURCE DATASET
# ================================================================

df = pd.read_csv(SOURCE_PATH)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)

# Detect ID
id_candidates = [
    "reconstructed_session_id",
    "session_id",
]

id_col = next(
    (c for c in id_candidates if c in df.columns),
    None
)

if id_col is None:
    raise ValueError(
        "STOP — no session ID column detected."
    )


# ================================================================
# [3] SESSION INTEGRITY
# ================================================================

missing_ids = int(df[id_col].isna().sum())
duplicate_ids = int(df[id_col].duplicated().sum())
unique_ids = int(df[id_col].nunique())

print("\n[3] SESSION INTEGRITY")
print("ID column:", id_col)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0:
    raise ValueError("STOP — missing session IDs.")

if duplicate_ids != 0:
    raise ValueError("STOP — duplicate session IDs.")

print("Session integrity: PASS")


# ================================================================
# [4] LABEL LEAKAGE
# ================================================================

FORBIDDEN_COLUMNS = {
    "attack_type",
    "severity",
    "attack_stage",
    "mitre_technique",
}

forbidden_present = sorted(
    FORBIDDEN_COLUMNS.intersection(df.columns)
)

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns present:", forbidden_present)

if forbidden_present:
    raise ValueError(
        "STOP — forbidden label columns detected."
    )

print("Label leakage: NO")


# ================================================================
# [5] BEHAVIORAL REFERENCE
# ================================================================

print("\n[5] BEHAVIORAL REFERENCE")
print("Loaded:", REFERENCE_PATH)

if not os.path.exists(REFERENCE_PATH):
    raise FileNotFoundError(
        "STOP — behavioral reference not found."
    )

with open(REFERENCE_PATH, "r") as f:
    reference = json.load(f)

# Detect feature list
behavioral_features = None

candidate_feature_keys = [
    "behavioral_features",
    "features",
    "feature_names",
    "usable_behavioral_features",
]

for key in candidate_feature_keys:
    value = reference.get(key)

    if isinstance(value, list):
        behavioral_features = value
        break

# Recursive fallback
if behavioral_features is None:

    def find_feature_list(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if (
                    isinstance(v, list)
                    and len(v) > 5
                    and all(isinstance(x, str) for x in v)
                ):
                    k_l = k.lower()

                    if (
                        "feature" in k_l
                        or "behavior" in k_l
                    ):
                        return v

                result = find_feature_list(v)

                if result is not None:
                    return result

        elif isinstance(obj, list):

            for item in obj:
                result = find_feature_list(item)

                if result is not None:
                    return result

        return None

    behavioral_features = find_feature_list(reference)

if behavioral_features is None:
    raise ValueError(
        "STOP — behavioral feature list could not be detected."
    )

# Only retain features that exist in source
behavioral_features = [
    f for f in behavioral_features
    if f in df.columns
]

if len(behavioral_features) == 0:
    raise ValueError(
        "STOP — no behavioral features found in source."
    )

print(
    "Behavioral features detected:",
    len(behavioral_features)
)


# ================================================================
# [6] BEHAVIORAL FEATURE MATRIX
# ================================================================

X_df = df[behavioral_features].copy()

# Force numeric
for col in behavioral_features:
    X_df[col] = pd.to_numeric(
        X_df[col],
        errors="coerce"
    )

# Remove all-missing features
all_missing = [
    c for c in behavioral_features
    if X_df[c].isna().all()
]

if all_missing:
    X_df = X_df.drop(columns=all_missing)

behavioral_features = X_df.columns.tolist()

# Median imputation ONLY for actual missing values.
# No synthetic threat/intelligence values are invented.
for col in behavioral_features:

    if X_df[col].isna().any():

        median_value = X_df[col].median()

        if pd.isna(median_value):
            raise ValueError(
                f"STOP — cannot impute feature: {col}"
            )

        X_df[col] = X_df[col].fillna(
            median_value
        )

X = X_df.to_numpy(
    dtype=np.float64
)

print("\n[6] BEHAVIORAL FEATURE MATRIX")
print(
    "Final behavioral feature count:",
    len(behavioral_features)
)
print("Matrix shape:", X.shape)

if not np.isfinite(X).all():
    raise ValueError(
        "STOP — behavioral matrix contains non-finite values."
    )

print("Finite matrix: PASS")


# ================================================================
# [7] SCALER ARTIFACT — ROBUST FORMAT HANDLING
# ================================================================

print("\n[7] SCALER ARTIFACT")
print("Path:", SCALER_PATH)

if not os.path.exists(SCALER_PATH):
    raise FileNotFoundError(
        "STOP — scaler artifact not found."
    )

with open(SCALER_PATH, "r") as f:
    scaler_params = json.load(f)

scaler_mean, scaler_scale = extract_scaler_arrays(
    scaler_params,
    behavioral_features
)

print(
    "Scaler mean length:",
    len(scaler_mean)
)

print(
    "Scaler scale length:",
    len(scaler_scale)
)

if len(scaler_mean) != X.shape[1]:
    raise ValueError(
        "STOP — scaler/source feature mismatch."
    )

X_scaled = (
    X - scaler_mean
) / scaler_scale

if not np.isfinite(X_scaled).all():
    raise ValueError(
        "STOP — scaled matrix contains non-finite values."
    )

print("Scaler compatibility: PASS")
print("Scaled matrix:", X_scaled.shape)


# ================================================================
# [8] DBSCAN REFERENCE ASSIGNMENTS
# ================================================================

print("\n[8] DBSCAN REFERENCE ASSIGNMENTS")

if not os.path.exists(ASSIGNMENTS_PATH):
    raise FileNotFoundError(
        "STOP — DBSCAN assignments not found."
    )

assignments = pd.read_csv(
    ASSIGNMENTS_PATH
)

print("Shape:", assignments.shape)
print("Columns:", assignments.columns.tolist())

assign_id_col = next(
    (
        c for c in [
            "reconstructed_session_id",
            "session_id"
        ]
        if c in assignments.columns
    ),
    None
)

cluster_col = next(
    (
        c for c in [
            "dbscan_cluster",
            "DBSCAN_cluster"
        ]
        if c in assignments.columns
    ),
    None
)

if assign_id_col is None:
    raise ValueError(
        "STOP — assignment ID column missing."
    )

if cluster_col is None:
    raise ValueError(
        "STOP — DBSCAN cluster column missing."
    )

if assignments[assign_id_col].isna().any():
    raise ValueError(
        "STOP — assignment IDs contain missing values."
    )

if assignments[assign_id_col].duplicated().any():
    raise ValueError(
        "STOP — duplicate assignment IDs."
    )

source_ids = set(
    df[id_col].astype(str)
)

assignment_ids = set(
    assignments[assign_id_col].astype(str)
)

if source_ids != assignment_ids:
    raise ValueError(
        "STOP — source and DBSCAN assignment IDs do not match."
    )

print("Assignment integrity: PASS")


# ================================================================
# [9] REFERENCE STRUCTURE
# ================================================================

cluster_values = assignments[cluster_col].to_numpy()

non_noise_clusters = sorted(
    [
        int(c)
        for c in pd.unique(cluster_values)
        if int(c) != -1
    ]
)

noise_count = int(
    np.sum(cluster_values == -1)
)

print("\n[9] DBSCAN REFERENCE STRUCTURE")
print(
    "Clusters including noise:",
    len(pd.unique(cluster_values))
)

print(
    "Non-noise clusters:",
    len(non_noise_clusters)
)

print(
    "Noise sessions:",
    noise_count
)

print(
    "Noise fraction:",
    round(
        noise_count / len(assignments),
        6
    )
)


# ================================================================
# [10] BUILD REFERENCE CENTROIDS
# ================================================================

print("\n[10] REFERENCE CENTROIDS")

assignment_map = (
    assignments
    .set_index(assign_id_col)[cluster_col]
)

aligned_clusters = (
    df[id_col]
    .astype(str)
    .map(
        assignment_map.astype(
            "int64"
        )
    )
    .to_numpy()
)

if len(aligned_clusters) != len(X_scaled):
    raise ValueError(
        "STOP — assignment alignment failed."
    )

centroid_rows = []

for cluster_id in non_noise_clusters:

    mask = (
        aligned_clusters == cluster_id
    )

    count = int(mask.sum())

    if count == 0:
        continue

    centroid = X_scaled[mask].mean(axis=0)

    row = {
        "dbscan_cluster": cluster_id,
        "session_count": count,
    }

    for feature, value in zip(
        behavioral_features,
        centroid
    ):
        row[feature] = float(value)

    centroid_rows.append(row)

centroids_df = pd.DataFrame(
    centroid_rows
)

centroid_path = os.path.join(
    OUTPUT_DIR,
    "step7g_dbscan_behavioral_reference_centroids.csv"
)

centroids_df.to_csv(
    centroid_path,
    index=False
)

print(
    "Clusters represented:",
    len(centroids_df)
)

print(
    "Saved:",
    centroid_path
)


# ================================================================
# [11] NEAREST CENTROID REFERENCE
# ================================================================

print("\n[11] NEAREST CENTROID REFERENCE")

centroid_matrix = centroids_df[
    behavioral_features
].to_numpy(
    dtype=np.float64
)

nearest_idx, nearest_dist = (
    nearest_centroid_distances(
        X_scaled,
        centroid_matrix
    )
)

nearest_cluster = (
    centroids_df
    .iloc[nearest_idx]["dbscan_cluster"]
    .to_numpy()
    .astype(int)
)

# IMPORTANT:
# This is NOT replacing DBSCAN labels.
# It is a production-compatible behavioral
# reference assignment for future logs.

reference_assignment_df = pd.DataFrame({
    id_col: df[id_col].values,
    "dbscan_cluster": aligned_clusters,
    "nearest_behavioral_cluster": nearest_cluster,
    "nearest_behavioral_distance": nearest_dist,
})

reference_assignment_path = os.path.join(
    OUTPUT_DIR,
    "step7g_reference_behavioral_assignments.csv"
)

reference_assignment_df.to_csv(
    reference_assignment_path,
    index=False
)

print(
    "Saved:",
    reference_assignment_path
)


# ================================================================
# [12] DISTANCE DISTRIBUTION
# ================================================================

distance_summary = pd.DataFrame({
    "statistic": [
        "min",
        "p01",
        "p05",
        "median",
        "p95",
        "p99",
        "max",
        "mean",
    ],
    "nearest_centroid_distance": [
        float(np.min(nearest_dist)),
        float(np.quantile(nearest_dist, 0.01)),
        float(np.quantile(nearest_dist, 0.05)),
        float(np.median(nearest_dist)),
        float(np.quantile(nearest_dist, 0.95)),
        float(np.quantile(nearest_dist, 0.99)),
        float(np.max(nearest_dist)),
        float(np.mean(nearest_dist)),
    ],
})

distance_path = os.path.join(
    OUTPUT_DIR,
    "step7g_reference_distance_summary.csv"
)

distance_summary.to_csv(
    distance_path,
    index=False
)

print(
    "Saved:",
    distance_path
)


# ================================================================
# [13] PRODUCTION CONFIG
# ================================================================

production_config = {
    "primary_model": "DBSCAN",
    "eps": 0.5,
    "min_samples": 20,
    "source_sha256": source_sha_before,
    "session_id_column": id_col,
    "cluster_column": "dbscan_cluster",
    "behavioral_feature_count": len(
        behavioral_features
    ),
    "behavioral_features": behavioral_features,
    "noise_label": -1,
    "reference_non_noise_clusters": len(
        non_noise_clusters
    ),
    "reference_noise_count": noise_count,
    "reference_noise_fraction": float(
        noise_count / len(assignments)
    ),
    "assignment_method_for_future_logs": (
        "nearest_behavioral_centroid"
    ),
    "important_note": (
        "Nearest-centroid assignment is a production "
        "reference assignment and does not recreate "
        "DBSCAN density fitting. Future logs should "
        "use the identical feature engineering and "
        "scaling pipeline."
    ),
}

config_path = os.path.join(
    OUTPUT_DIR,
    "step7g_production_config.json"
)

with open(
    config_path,
    "w"
) as f:
    json.dump(
        production_config,
        f,
        indent=2
    )

print(
    "Saved:",
    config_path
)


# ================================================================
# [14] AUDIT
# ================================================================

source_sha_after = sha256_file(
    SOURCE_PATH
)

audit = {
    "step": "7G-R2",
    "status": "PASS",
    "source_path": SOURCE_PATH,
    "source_sha256_before": source_sha_before,
    "source_sha256_after": source_sha_after,
    "source_unchanged": (
        source_sha_before ==
        source_sha_after
    ),
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "session_integrity": {
        "missing_ids": missing_ids,
        "duplicate_ids": duplicate_ids,
        "unique_ids": unique_ids,
    },
    "label_leakage": {
        "forbidden_columns_present":
            forbidden_present
    },
    "behavioral_features": behavioral_features,
    "behavioral_feature_count":
        len(behavioral_features),
    "scaler_compatible": True,
    "scaled_matrix_shape":
        list(X_scaled.shape),
    "dbscan": {
        "eps": 0.5,
        "min_samples": 20,
        "non_noise_clusters":
            len(non_noise_clusters),
        "noise_count":
            noise_count,
        "noise_fraction":
            float(
                noise_count /
                len(assignments)
            ),
    },
    "production_assignment": {
        "method":
            "nearest_behavioral_centroid",
        "reference_centroids":
            len(centroids_df),
    },
    "outputs": {
        "centroids":
            centroid_path,
        "reference_assignments":
            reference_assignment_path,
        "distance_summary":
            distance_path,
        "production_config":
            config_path,
    },
}

audit_path = os.path.join(
    OUTPUT_DIR,
    "step7g_production_cluster_pipeline_audit.json"
)

with open(
    audit_path,
    "w"
) as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(
    audit_path
)


# ================================================================
# [15] FINAL PROTECTION CHECK
# ================================================================

print("\n[15] FINAL AUDIT")

print(
    "Source SHA before:",
    source_sha_before
)

print(
    "Source SHA after: ",
    source_sha_after
)

print(
    "Source unchanged:",
    source_sha_before ==
    source_sha_after
)

print(
    "Session integrity: PASS"
)

print(
    "Label leakage: NO"
)

print(
    "Behavioral features:",
    len(behavioral_features)
)

print(
    "Scaler compatibility: PASS"
)

print(
    "Primary model: DBSCAN"
)

print(
    "Non-noise clusters:",
    len(non_noise_clusters)
)

print(
    "Noise sessions:",
    noise_count
)

print(
    "Noise fraction:",
    round(
        noise_count /
        len(assignments),
        6
    )
)

print(
    "Audit:",
    audit_path
)

print(
    "Audit SHA-256:",
    audit_sha
)

if source_sha_before != source_sha_after:
    raise ValueError(
        "STOP — source changed during STEP 7G-R2."
    )

print("\n" + "=" * 70)
print("STEP 7G-R2 — COMPLETE")
print("=" * 70)

print("STATUS: PASS")
print(
    "NEXT: STEP 7H — PRODUCTION ARTIFACT PACKAGE "
    "+ FUTURE-LOG VALIDATION"
)

STEP 7G-R2 — PRODUCTION-READY DBSCAN CLUSTER ASSIGNMENT PIPELINE

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] BEHAVIORAL REFERENCE
Loaded: /content/step7_cluster_profiling/step7d_detailed_behavioral_profiling/step7d_behavioral_reference.json
Behavioral features detected: 48

[6] BEHAVIORAL FEATURE MATRIX
Final behavioral feature count: 48
Matrix shape: (11880, 48)
Finite matrix: PASS

[7] SCALER ARTIFACT
Path: /content/step6_unsupervised_ml/step6b_preprocessing/standard_scaler_params.json
Scaler mean length: 48
Scaler scale length: 48
Scaler compatibility: PASS
Scaled matrix: (11880, 48)

[8]

In [ ]:
# ================================================================
# STEP 7H — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION
# ================================================================

import os
import json
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------

SOURCE = Path(
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

STEP7_ROOT = Path("/content/step7_cluster_profiling")

PRODUCTION_DIR = (
    STEP7_ROOT /
    "step7g_production_cluster_pipeline"
)

VALIDATION_DIR = (
    STEP7_ROOT /
    "step7h_production_validation"
)

PACKAGE_DIR = (
    STEP7_ROOT /
    "step7h_production_package"
)

ZIP_PATH = Path(
    "/content/STEP_7_ML_PRODUCTION_COMPLETE.zip"
)

# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def require_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"STOP — required {label} missing:\n{path}"
        )


def detect_column(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"STOP — {label} not found.\n"
        f"Available columns: {list(df.columns)}"
    )


def json_dump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ----------------------------------------------------------------
# START
# ----------------------------------------------------------------

print("=" * 70)
print("STEP 7H — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION")
print("=" * 70)

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# [1] SOURCE VERIFICATION
# ----------------------------------------------------------------

print("\n[1] SOURCE VERIFICATION")

require_file(SOURCE, "source dataset")

source_sha_before = sha256_file(SOURCE)

print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", source_sha_before)

if source_sha_before != EXPECTED_SOURCE_SHA:
    raise ValueError(
        "STOP — source SHA mismatch."
    )

print("Verification: PASS")

# ----------------------------------------------------------------
# [2] LOAD SOURCE
# ----------------------------------------------------------------

print("\n[2] SOURCE DATASET")

df = pd.read_csv(SOURCE)

print("Shape:", df.shape)

# ----------------------------------------------------------------
# [3] SESSION INTEGRITY
# ----------------------------------------------------------------

print("\n[3] SESSION INTEGRITY")

id_col = detect_column(
    df,
    [
        "reconstructed_session_id",
        "session_id",
        "session"
    ],
    "session ID column"
)

missing_ids = int(df[id_col].isna().sum())
duplicate_ids = int(df[id_col].duplicated().sum())
unique_ids = int(df[id_col].nunique())

print("ID column:", id_col)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)
print("Unique IDs:", unique_ids)

if missing_ids != 0 or duplicate_ids != 0:
    raise ValueError(
        "STOP — session integrity failure."
    )

print("Session integrity: PASS")

# ----------------------------------------------------------------
# [4] LOAD PRODUCTION REFERENCE
# ----------------------------------------------------------------

print("\n[4] LOADING PRODUCTION REFERENCE")

REFERENCE_JSON = (
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_behavioral_reference.json"
)

CENTROIDS = (
    PRODUCTION_DIR /
    "step7g_dbscan_behavioral_reference_centroids.csv"
)

REFERENCE_ASSIGNMENTS = (
    PRODUCTION_DIR /
    "step7g_reference_behavioral_assignments.csv"
)

DISTANCE_SUMMARY = (
    PRODUCTION_DIR /
    "step7g_reference_distance_summary.csv"
)

CONFIG = (
    PRODUCTION_DIR /
    "step7g_production_config.json"
)

AUDIT = (
    PRODUCTION_DIR /
    "step7g_production_cluster_pipeline_audit.json"
)

for p, label in [
    (REFERENCE_JSON, "behavioral reference"),
    (CENTROIDS, "reference centroids"),
    (REFERENCE_ASSIGNMENTS, "reference assignments"),
    (DISTANCE_SUMMARY, "distance summary"),
    (CONFIG, "production config"),
    (AUDIT, "production audit"),
]:
    require_file(p, label)

print("Reference: PASS")
print("Centroids: PASS")
print("Reference assignments: PASS")
print("Distance summary: PASS")
print("Production config: PASS")
print("Production audit: PASS")

# ----------------------------------------------------------------
# [5] LOAD CONFIG
# ----------------------------------------------------------------

print("\n[5] PRODUCTION CONFIG")

with open(CONFIG, "r", encoding="utf-8") as f:
    production_config = json.load(f)

print(
    json.dumps(
        production_config,
        indent=2,
        ensure_ascii=False
    )[:4000]
)

# ----------------------------------------------------------------
# [6] LOAD BEHAVIORAL REFERENCE
# ----------------------------------------------------------------

print("\n[6] BEHAVIORAL FEATURE SCHEMA")

with open(REFERENCE_JSON, "r", encoding="utf-8") as f:
    reference = json.load(f)

behavioral_features = None

possible_keys = [
    "behavioral_features",
    "features",
    "feature_names",
    "usable_behavioral_features"
]

for key in possible_keys:
    value = reference.get(key)
    if isinstance(value, list) and len(value) > 0:
        behavioral_features = value
        break

if behavioral_features is None:
    # Fallback: infer from centroid columns.
    centroid_probe = pd.read_csv(CENTROIDS)
    excluded = {
        "dbscan_cluster",
        "cluster",
        "reconstructed_session_id",
        "session_id"
    }

    behavioral_features = [
        c for c in centroid_probe.columns
        if c not in excluded
        and pd.api.types.is_numeric_dtype(
            centroid_probe[c]
        )
    ]

behavioral_features = list(dict.fromkeys(behavioral_features))

print(
    "Behavioral features detected:",
    len(behavioral_features)
)

if len(behavioral_features) != 48:
    raise ValueError(
        f"STOP — expected 48 behavioral features, "
        f"found {len(behavioral_features)}."
    )

missing_features = [
    c for c in behavioral_features
    if c not in df.columns
]

if missing_features:
    raise ValueError(
        "STOP — behavioral feature schema mismatch.\n"
        f"Missing features: {missing_features}"
    )

print("Feature schema: PASS")

# ----------------------------------------------------------------
# [7] BUILD FUTURE-LOG INPUT MATRIX
# ----------------------------------------------------------------

print("\n[7] FUTURE-LOG SIMULATION MATRIX")

future_df = df.copy()

# Simulate a future batch:
# deterministic selection avoids modifying the source.
future_n = max(1, int(round(len(df) * 0.20)))

future_df = (
    df.sample(
        n=future_n,
        random_state=2026
    )
    .copy()
)

print("Simulated future rows:", len(future_df))

X_future = future_df[behavioral_features].copy()

# Numeric conversion
for c in behavioral_features:
    X_future[c] = pd.to_numeric(
        X_future[c],
        errors="coerce"
    )

# ----------------------------------------------------------------
# [8] MISSING VALUE HANDLING
# ----------------------------------------------------------------

print("\n[8] MISSING VALUE HANDLING")

missing_before = int(X_future.isna().sum().sum())

print(
    "Missing values before imputation:",
    missing_before
)

# Use reference/global medians where available.
# If unavailable, use current source median only as a
# deterministic fallback for validation.
global_medians = {}

if isinstance(reference, dict):
    for key in [
        "global_feature_medians",
        "feature_medians",
        "medians"
    ]:
        if isinstance(reference.get(key), dict):
            global_medians = reference[key]
            break

for c in behavioral_features:
    if c in global_medians:
        median_value = global_medians[c]
    else:
        median_value = pd.to_numeric(
            df[c],
            errors="coerce"
        ).median()

    if not np.isfinite(median_value):
        median_value = 0.0

    X_future[c] = X_future[c].fillna(
        float(median_value)
    )

X_future = X_future.replace(
    [np.inf, -np.inf],
    np.nan
)

# Final fallback
X_future = X_future.fillna(0.0)

if not np.isfinite(
    X_future.to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "STOP — future feature matrix contains non-finite values."
    )

print("Finite matrix: PASS")

# ----------------------------------------------------------------
# [9] LOAD SCALER
# ----------------------------------------------------------------

print("\n[9] SCALER")

SCALER_PATH = (
    STEP7_ROOT /
    "step6_unsupervised_ml" /
    "step6b_preprocessing" /
    "standard_scaler_params.json"
)

require_file(
    SCALER_PATH,
    "StandardScaler parameters"
)

with open(SCALER_PATH, "r", encoding="utf-8") as f:
    scaler_raw = json.load(f)


def extract_vector(obj, keys):
    for key in keys:
        if key not in obj:
            continue

        value = obj[key]

        if isinstance(value, list):
            return np.asarray(
                value,
                dtype=np.float64
            )

        if isinstance(value, dict):
            # Support feature -> value format
            try:
                return np.asarray(
                    [
                        value[k]
                        for k in behavioral_features
                    ],
                    dtype=np.float64
                )
            except Exception:
                pass

    return None


scaler_mean = extract_vector(
    scaler_raw,
    ["mean", "mean_", "center"]
)

scaler_scale = extract_vector(
    scaler_raw,
    ["scale", "scale_", "std"]
)

if scaler_mean is None or scaler_scale is None:
    raise ValueError(
        "STOP — unable to extract scaler mean/scale."
    )

if len(scaler_mean) != 48:
    raise ValueError(
        "STOP — scaler mean length mismatch."
    )

if len(scaler_scale) != 48:
    raise ValueError(
        "STOP — scaler scale length mismatch."
    )

if np.any(~np.isfinite(scaler_mean)):
    raise ValueError(
        "STOP — scaler mean contains non-finite values."
    )

if np.any(~np.isfinite(scaler_scale)):
    raise ValueError(
        "STOP — scaler scale contains non-finite values."
    )

if np.any(scaler_scale == 0):
    raise ValueError(
        "STOP — scaler contains zero scale."
    )

X_future_np = X_future.to_numpy(
    dtype=np.float64
)

X_future_scaled = (
    X_future_np - scaler_mean
) / scaler_scale

if not np.isfinite(X_future_scaled).all():
    raise ValueError(
        "STOP — scaled future matrix is non-finite."
    )

print("Scaler features:", len(scaler_mean))
print("Scaled matrix:", X_future_scaled.shape)
print("Scaler compatibility: PASS")

# ----------------------------------------------------------------
# [10] LOAD REFERENCE CENTROIDS
# ----------------------------------------------------------------

print("\n[10] REFERENCE CENTROIDS")

centroid_df = pd.read_csv(CENTROIDS)

cluster_col = detect_column(
    centroid_df,
    [
        "dbscan_cluster",
        "cluster"
    ],
    "centroid cluster column"
)

centroid_feature_cols = [
    c for c in behavioral_features
    if c in centroid_df.columns
]

if len(centroid_feature_cols) != 48:
    raise ValueError(
        "STOP — centroid feature schema mismatch.\n"
        f"Found {len(centroid_feature_cols)} features."
    )

centroid_matrix = centroid_df[
    centroid_feature_cols
].to_numpy(
    dtype=np.float64
)

centroid_labels = centroid_df[
    cluster_col
].to_numpy()

if centroid_matrix.shape[1] != 48:
    raise ValueError(
        "STOP — centroid matrix width mismatch."
    )

if not np.isfinite(centroid_matrix).all():
    raise ValueError(
        "STOP — centroid matrix contains non-finite values."
    )

print(
    "Reference clusters:",
    len(centroid_labels)
)

print(
    "Centroid matrix:",
    centroid_matrix.shape
)

print("Centroids: PASS")

# ----------------------------------------------------------------
# [11] NORMALIZE CENTROIDS IF NEEDED
# ----------------------------------------------------------------
# Determine whether centroids appear to be in raw or scaled space.
# The production reference pipeline uses standardized behavioral
# space. We therefore compare dimensions only and preserve the
# stored reference representation.

print("\n[11] CENTROID SPACE VALIDATION")

# The reference centroid artifact is assumed to be generated from
# the production standardized matrix. No re-fitting is performed.

if centroid_matrix.shape[1] != X_future_scaled.shape[1]:
    raise ValueError(
        "STOP — future matrix and centroid dimensions differ."
    )

print("Centroid compatibility: PASS")

# ----------------------------------------------------------------
# [12] FUTURE-LOG ASSIGNMENT
# ----------------------------------------------------------------

print("\n[12] FUTURE-LOG CLUSTER ASSIGNMENT")


def nearest_centroids(matrix, centroids):
    """
    Robust vectorized nearest-centroid calculation.
    Returns nearest centroid index and Euclidean distance.
    """

    matrix = np.asarray(
        matrix,
        dtype=np.float64
    )

    centroids = np.asarray(
        centroids,
        dtype=np.float64
    )

    if matrix.ndim != 2:
        raise ValueError(
            "Matrix must be 2-dimensional."
        )

    if centroids.ndim != 2:
        raise ValueError(
            "Centroids must be 2-dimensional."
        )

    if matrix.shape[1] != centroids.shape[1]:
        raise ValueError(
            "Feature dimensions do not match."
        )

    # Stable squared Euclidean distance:
    #
    # ||x-c||² = ||x||² + ||c||² - 2*x.c
    #
    # This avoids the previous axis/sum bug.
    x_norm = np.sum(
        matrix * matrix,
        axis=1,
        keepdims=True
    )

    c_norm = np.sum(
        centroids * centroids,
        axis=1,
        keepdims=True
    ).T

    distances_sq = (
        x_norm
        + c_norm
        - 2.0 * np.matmul(
            matrix,
            centroids.T
        )
    )

    distances_sq = np.maximum(
        distances_sq,
        0.0
    )

    nearest_idx = np.argmin(
        distances_sq,
        axis=1
    )

    nearest_dist = np.sqrt(
        distances_sq[
            np.arange(len(matrix)),
            nearest_idx
        ]
    )

    return nearest_idx, nearest_dist


nearest_idx, nearest_dist = nearest_centroids(
    X_future_scaled,
    centroid_matrix
)

future_assignments = pd.DataFrame({
    id_col: future_df[id_col].values,
    "production_dbscan_cluster": centroid_labels[
        nearest_idx
    ],
    "distance_to_reference_centroid": nearest_dist
})

# ----------------------------------------------------------------
# [13] FUTURE ASSIGNMENT SANITY
# ----------------------------------------------------------------

print("\n[13] FUTURE ASSIGNMENT SANITY")

if len(future_assignments) != len(future_df):
    raise ValueError(
        "STOP — assignment row count mismatch."
    )

if future_assignments[id_col].isna().any():
    raise ValueError(
        "STOP — assigned session IDs contain missing values."
    )

if future_assignments[id_col].duplicated().any():
    raise ValueError(
        "STOP — future assignment contains duplicate IDs."
    )

if not np.isfinite(
    future_assignments[
        "distance_to_reference_centroid"
    ].to_numpy()
).all():
    raise ValueError(
        "STOP — assignment distances are non-finite."
    )

print("Rows assigned:", len(future_assignments))
print("Missing IDs:", future_assignments[id_col].isna().sum())
print(
    "Unique clusters assigned:",
    future_assignments[
        "production_dbscan_cluster"
    ].nunique()
)

print("Assignment sanity: PASS")

# ----------------------------------------------------------------
# [14] DISTANCE SUMMARY
# ----------------------------------------------------------------

print("\n[14] DISTANCE SUMMARY")

distance_summary = pd.DataFrame({
    "metric": [
        "count",
        "mean_distance",
        "median_distance",
        "p95_distance",
        "max_distance"
    ],
    "value": [
        len(nearest_dist),
        float(np.mean(nearest_dist)),
        float(np.median(nearest_dist)),
        float(np.percentile(nearest_dist, 95)),
        float(np.max(nearest_dist))
    ]
})

print(distance_summary.to_string(index=False))

# ----------------------------------------------------------------
# [15] SAVE FUTURE VALIDATION ARTIFACTS
# ----------------------------------------------------------------

print("\n[15] SAVING VALIDATION ARTIFACTS")

future_assignment_path = (
    VALIDATION_DIR /
    "step7h_future_log_simulated_assignments.csv"
)

distance_path = (
    VALIDATION_DIR /
    "step7h_future_log_distance_summary.csv"
)

feature_schema_path = (
    VALIDATION_DIR /
    "step7h_production_feature_schema.json"
)

future_assignments.to_csv(
    future_assignment_path,
    index=False
)

distance_summary.to_csv(
    distance_path,
    index=False
)

json_dump(
    {
        "feature_count": len(behavioral_features),
        "behavioral_features": behavioral_features,
        "scaler": str(SCALER_PATH),
        "centroids": str(CENTROIDS),
        "primary_model": "DBSCAN",
        "eps": 0.5,
        "min_samples": 20,
        "reference_non_noise_clusters": 38,
        "reference_noise_sessions": 1676,
        "reference_noise_fraction": 0.141077,
        "assignment_mode": (
            "nearest_reference_centroid"
        )
    },
    feature_schema_path
)

print("Future assignments:", future_assignment_path)
print("Distance summary:", distance_path)
print("Feature schema:", feature_schema_path)

# ----------------------------------------------------------------
# [16] PACKAGE PRODUCTION ARTIFACTS
# ----------------------------------------------------------------

print("\n[16] BUILDING PRODUCTION PACKAGE")

# Clean package directory
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

artifact_paths = [
    REFERENCE_JSON,
    SCALER_PATH,
    CENTROIDS,
    REFERENCE_ASSIGNMENTS,
    DISTANCE_SUMMARY,
    CONFIG,
    AUDIT,
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_cluster_sizes.csv",
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_detailed_cluster_profiles.csv",
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_behavioral_signatures.csv",
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_top_behavioral_features.csv",
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_cluster_interpretation_summary.csv",
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_noise_behavioral_profile.csv",
    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_corrected_stability_results.csv",
    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_pairwise_stability_results.csv",
    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_corrected_cluster_validation_audit.json",
    future_assignment_path,
    distance_path,
    feature_schema_path,
]

# Keep only existing files.
artifact_paths = [
    p for p in artifact_paths
    if p.exists()
]

if len(artifact_paths) == 0:
    raise ValueError(
        "STOP — no production artifacts found."
    )

manifest_files = []

for src in artifact_paths:

    # Preserve filename, but prevent collisions.
    dst = PACKAGE_DIR / src.name

    if dst.exists():
        stem = src.parent.name + "__" + src.name
        dst = PACKAGE_DIR / stem

    shutil.copy2(
        src,
        dst
    )

    manifest_files.append({
        "source": str(src),
        "packaged_name": dst.name,
        "bytes": dst.stat().st_size,
        "sha256": sha256_file(dst)
    })

print(
    "Artifacts packaged:",
    len(manifest_files)
)

# ----------------------------------------------------------------
# [17] MANIFEST
# ----------------------------------------------------------------

manifest = {
    "package": "STEP_7_ML_PRODUCTION_COMPLETE",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": str(SOURCE),
    "source_sha256": source_sha_before,
    "source_verified": (
        source_sha_before == EXPECTED_SOURCE_SHA
    ),
    "primary_model": "DBSCAN",
    "dbscan_eps": 0.5,
    "dbscan_min_samples": 20,
    "reference_non_noise_clusters": 38,
    "reference_noise_sessions": 1676,
    "reference_noise_fraction": 0.141077,
    "behavioral_feature_count": 48,
    "future_simulation_rows": len(future_df),
    "assignment_strategy": (
        "nearest_reference_centroid"
    ),
    "artifacts": manifest_files
}

MANIFEST_PATH = (
    PACKAGE_DIR /
    "step7h_manifest.json"
)

json_dump(
    manifest,
    MANIFEST_PATH
)

# ----------------------------------------------------------------
# [18] ZIP PACKAGE
# ----------------------------------------------------------------

print("\n[18] CREATING ZIP")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for file_path in PACKAGE_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(
                file_path,
                arcname=file_path.relative_to(
                    PACKAGE_DIR
                )
            )

print("ZIP:", ZIP_PATH)
print(
    "ZIP size:",
    ZIP_PATH.stat().st_size,
    "bytes"
)

zip_sha = sha256_file(
    ZIP_PATH
)

print("ZIP SHA-256:", zip_sha)

# ----------------------------------------------------------------
# [19] ZIP INTEGRITY
# ----------------------------------------------------------------

print("\n[19] ZIP INTEGRITY")

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    bad_file = zf.testzip()

    if bad_file is not None:
        raise ValueError(
            f"STOP — corrupted ZIP member: {bad_file}"
        )

    zip_members = zf.namelist()

print(
    "ZIP members:",
    len(zip_members)
)

print("ZIP integrity: PASS")

# ----------------------------------------------------------------
# [20] SOURCE PROTECTION
# ----------------------------------------------------------------

print("\n[20] SOURCE PROTECTION")

source_sha_after = sha256_file(
    SOURCE
)

print(
    "SHA before:",
    source_sha_before
)

print(
    "SHA after: ",
    source_sha_after
)

if source_sha_before != source_sha_after:
    raise ValueError(
        "STOP — source dataset changed."
    )

print("Source unchanged: PASS")

# ----------------------------------------------------------------
# [21] FINAL AUDIT
# ----------------------------------------------------------------

audit = {
    "source_sha_expected": EXPECTED_SOURCE_SHA,
    "source_sha_before": source_sha_before,
    "source_sha_after": source_sha_after,
    "source_unchanged": (
        source_sha_before == source_sha_after
    ),

    "source_shape": list(df.shape),

    "session_id_column": id_col,
    "missing_ids": missing_ids,
    "duplicate_ids": duplicate_ids,
    "unique_ids": unique_ids,

    "behavioral_feature_count": len(
        behavioral_features
    ),

    "behavioral_feature_schema_pass": True,

    "future_simulation_rows": len(
        future_df
    ),

    "future_assignment_pass": True,

    "reference_clusters": 38,
    "reference_noise_sessions": 1676,
    "reference_noise_fraction": 0.141077,

    "primary_model": "DBSCAN",
    "eps": 0.5,
    "min_samples": 20,

    "artifact_count": len(
        manifest_files
    ),

    "zip_path": str(ZIP_PATH),
    "zip_sha256": zip_sha,

    "status": "PASS"
}

AUDIT_PATH = (
    VALIDATION_DIR /
    "step7h_production_validation_audit.json"
)

json_dump(
    audit,
    AUDIT_PATH
)

audit_sha = sha256_file(
    AUDIT_PATH
)

# ----------------------------------------------------------------
# FINAL OUTPUT
# ----------------------------------------------------------------

print("\n")
print("=" * 70)
print("STEP 7H — COMPLETE")
print("=" * 70)

print("\nSource SHA:")
print(source_sha_after)

print("\nBehavioral features:")
print(len(behavioral_features))

print("\nFuture-log simulation:")
print("Rows:", len(future_df))
print("Assignments:", len(future_assignments))
print("Assignment validation: PASS")

print("\nProduction model:")
print("DBSCAN")
print("eps:", 0.5)
print("min_samples:", 20)
print("Reference clusters:", 38)
print("Reference noise:", 1676)
print("Reference noise fraction:", 0.141077)

print("\nArtifacts packaged:")
print(len(manifest_files))

print("\nZIP:")
print(ZIP_PATH)

print("ZIP size:", ZIP_PATH.stat().st_size)
print("ZIP SHA-256:", zip_sha)

print("\nAudit:")
print(AUDIT_PATH)

print("Audit SHA-256:", audit_sha)

print("\nSTATUS: PASS")
print("NEXT: STEP 8 — GEOIP / ATTACKER / MITRE / SESSION REPLAY / DASHBOARD DATA")

STEP 7H — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LOADING PRODUCTION REFERENCE
Reference: PASS
Centroids: PASS
Reference assignments: PASS
Distance summary: PASS
Production config: PASS
Production audit: PASS

[5] PRODUCTION CONFIG
{
  "primary_model": "DBSCAN",
  "eps": 0.5,
  "min_samples": 20,
  "source_sha256": "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534",
  "session_id_column": "reconstructed_session_id",
  "cluster_column": "dbscan_cluster",
  "behavioral_feature_count": 48,
  "behavioral_features": [
    "event_count",
    "session_duration_seconds",
    "auth_attempt_total

FileNotFoundError: STOP — required StandardScaler parameters missing:
/content/step7_cluster_profiling/step6_unsupervised_ml/step6b_preprocessing/standard_scaler_params.json

In [ ]:
# ================================================================
# STEP 7H-R — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION
#                 PATH-CORRECTED + SAFE ARTIFACT DISCOVERY
# ================================================================

import os
import json
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------

SOURCE = Path(
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

STEP7_ROOT = Path("/content/step7_cluster_profiling")

# IMPORTANT:
# step6_unsupervised_ml is under /content, NOT under step7_cluster_profiling
STEP6_ROOT = Path("/content/step6_unsupervised_ml")

PRODUCTION_DIR = (
    STEP7_ROOT /
    "step7g_production_cluster_pipeline"
)

VALIDATION_DIR = (
    STEP7_ROOT /
    "step7h_production_validation"
)

PACKAGE_DIR = (
    STEP7_ROOT /
    "step7h_production_package"
)

ZIP_PATH = Path(
    "/content/STEP_7_ML_PRODUCTION_COMPLETE.zip"
)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def require_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"STOP — required {label} missing:\n{path}"
        )


def find_file(candidates, label):
    """
    Safely search known absolute paths first.
    No files are modified.
    """

    checked = []

    for p in candidates:

        p = Path(p)
        checked.append(str(p))

        if p.exists() and p.is_file():
            print(f"{label} found:")
            print(p)
            return p

    raise FileNotFoundError(
        f"STOP — required {label} not found.\n\n"
        "Checked paths:\n" +
        "\n".join(
            f" - {x}"
            for x in checked
        )
    )


def detect_column(df, candidates, label):

    for c in candidates:

        if c in df.columns:
            return c

    raise ValueError(
        f"STOP — {label} not found.\n"
        f"Available columns: {list(df.columns)}"
    )


def json_dump(obj, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


# ----------------------------------------------------------------
# START
# ----------------------------------------------------------------

print("=" * 70)
print("STEP 7H-R — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION")
print("=" * 70)


VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ----------------------------------------------------------------
# [1] SOURCE VERIFICATION
# ----------------------------------------------------------------

print("\n[1] SOURCE VERIFICATION")

require_file(
    SOURCE,
    "source dataset"
)

source_sha_before = sha256_file(
    SOURCE
)

print(
    "Expected SHA:",
    EXPECTED_SOURCE_SHA
)

print(
    "Actual SHA:  ",
    source_sha_before
)

if source_sha_before != EXPECTED_SOURCE_SHA:

    raise ValueError(
        "STOP — source SHA mismatch."
    )

print("Verification: PASS")


# ----------------------------------------------------------------
# [2] SOURCE DATASET
# ----------------------------------------------------------------

print("\n[2] SOURCE DATASET")

df = pd.read_csv(
    SOURCE
)

print(
    "Shape:",
    df.shape
)


# ----------------------------------------------------------------
# [3] SESSION INTEGRITY
# ----------------------------------------------------------------

print("\n[3] SESSION INTEGRITY")

id_col = detect_column(
    df,
    [
        "reconstructed_session_id",
        "session_id",
        "session"
    ],
    "session ID column"
)

missing_ids = int(
    df[id_col].isna().sum()
)

duplicate_ids = int(
    df[id_col].duplicated().sum()
)

unique_ids = int(
    df[id_col].nunique()
)

print(
    "ID column:",
    id_col
)

print(
    "Missing IDs:",
    missing_ids
)

print(
    "Duplicate IDs:",
    duplicate_ids
)

print(
    "Unique IDs:",
    unique_ids
)

if missing_ids != 0:
    raise ValueError(
        "STOP — missing session IDs."
    )

if duplicate_ids != 0:
    raise ValueError(
        "STOP — duplicate session IDs."
    )

print(
    "Session integrity: PASS"
)


# ----------------------------------------------------------------
# [4] LOAD PRODUCTION REFERENCE
# ----------------------------------------------------------------

print("\n[4] LOADING PRODUCTION REFERENCE")

REFERENCE_JSON = (
    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_behavioral_reference.json"
)

CENTROIDS = (
    PRODUCTION_DIR /
    "step7g_dbscan_behavioral_reference_centroids.csv"
)

REFERENCE_ASSIGNMENTS = (
    PRODUCTION_DIR /
    "step7g_reference_behavioral_assignments.csv"
)

DISTANCE_SUMMARY = (
    PRODUCTION_DIR /
    "step7g_reference_distance_summary.csv"
)

CONFIG = (
    PRODUCTION_DIR /
    "step7g_production_config.json"
)

PRODUCTION_AUDIT = (
    PRODUCTION_DIR /
    "step7g_production_cluster_pipeline_audit.json"
)


for p, label in [

    (
        REFERENCE_JSON,
        "behavioral reference"
    ),

    (
        CENTROIDS,
        "reference centroids"
    ),

    (
        REFERENCE_ASSIGNMENTS,
        "reference assignments"
    ),

    (
        DISTANCE_SUMMARY,
        "distance summary"
    ),

    (
        CONFIG,
        "production config"
    ),

    (
        PRODUCTION_AUDIT,
        "production audit"
    )

]:

    require_file(
        p,
        label
    )


print(
    "Reference: PASS"
)

print(
    "Centroids: PASS"
)

print(
    "Reference assignments: PASS"
)

print(
    "Distance summary: PASS"
)

print(
    "Production config: PASS"
)

print(
    "Production audit: PASS"
)


# ----------------------------------------------------------------
# [5] PRODUCTION CONFIG
# ----------------------------------------------------------------

print("\n[5] PRODUCTION CONFIG")

with open(
    CONFIG,
    "r",
    encoding="utf-8"
) as f:

    production_config = json.load(f)


print(
    json.dumps(
        production_config,
        indent=2,
        ensure_ascii=False
    )[:5000]
)


# ----------------------------------------------------------------
# [6] BEHAVIORAL FEATURE SCHEMA
# ----------------------------------------------------------------

print("\n[6] BEHAVIORAL FEATURE SCHEMA")

with open(
    REFERENCE_JSON,
    "r",
    encoding="utf-8"
) as f:

    reference = json.load(f)


behavioral_features = None


for key in [

    "behavioral_features",
    "features",
    "feature_names",
    "usable_behavioral_features"

]:

    value = reference.get(
        key
    )

    if (
        isinstance(value, list)
        and len(value) > 0
    ):

        behavioral_features = value

        break


# Fallback to production config
if behavioral_features is None:

    value = production_config.get(
        "behavioral_features"
    )

    if (
        isinstance(value, list)
        and len(value) > 0
    ):

        behavioral_features = value


# Final fallback: centroid schema
if behavioral_features is None:

    centroid_probe = pd.read_csv(
        CENTROIDS
    )

    excluded = {
        "dbscan_cluster",
        "cluster",
        "reconstructed_session_id",
        "session_id"
    }

    behavioral_features = [

        c for c in centroid_probe.columns

        if c not in excluded

        and pd.api.types.is_numeric_dtype(
            centroid_probe[c]
        )

    ]


behavioral_features = list(
    dict.fromkeys(
        behavioral_features
    )
)


print(
    "Behavioral features detected:",
    len(behavioral_features)
)


if len(behavioral_features) != 48:

    raise ValueError(
        "STOP — expected exactly 48 "
        "behavioral features."
    )


missing_features = [

    c for c in behavioral_features

    if c not in df.columns

]


if missing_features:

    raise ValueError(
        "STOP — behavioral schema mismatch.\n"
        f"Missing features:\n{missing_features}"
    )


print(
    "Feature schema: PASS"
)


# ----------------------------------------------------------------
# [7] FUTURE LOG SIMULATION
# ----------------------------------------------------------------

print("\n[7] FUTURE-LOG SIMULATION")

future_n = max(
    1,
    int(round(len(df) * 0.20))
)

future_df = (

    df.sample(
        n=future_n,
        random_state=2026
    )

    .copy()

)


print(
    "Simulated future rows:",
    len(future_df)
)


X_future = future_df[
    behavioral_features
].copy()


# ----------------------------------------------------------------
# [8] NUMERIC + MISSING VALUE HANDLING
# ----------------------------------------------------------------

print(
    "\n[8] MISSING VALUE HANDLING"
)


for c in behavioral_features:

    X_future[c] = pd.to_numeric(
        X_future[c],
        errors="coerce"
    )


missing_before = int(
    X_future.isna().sum().sum()
)

print(
    "Missing values before handling:",
    missing_before
)


# Try reference medians first
global_medians = {}

for key in [

    "global_feature_medians",
    "feature_medians",
    "medians"

]:

    if isinstance(
        reference.get(key),
        dict
    ):

        global_medians = reference[key]

        break


for c in behavioral_features:

    if c in global_medians:

        median_value = global_medians[c]

    else:

        median_value = pd.to_numeric(
            df[c],
            errors="coerce"
        ).median()


    if not np.isfinite(
        median_value
    ):

        median_value = 0.0


    X_future[c] = (
        X_future[c]
        .fillna(
            float(median_value)
        )
    )


X_future = X_future.replace(
    [np.inf, -np.inf],
    np.nan
)

X_future = X_future.fillna(
    0.0
)


X_future_np = X_future.to_numpy(
    dtype=np.float64
)


if not np.isfinite(
    X_future_np
).all():

    raise ValueError(
        "STOP — future matrix is non-finite."
    )


print(
    "Finite matrix: PASS"
)


# ----------------------------------------------------------------
# [9] SCALER ARTIFACT — CORRECTED PATH
# ----------------------------------------------------------------

print(
    "\n[9] SCALER ARTIFACT"
)

print(
    "Searching outside step7 root "
    "because scaler belongs to Step 6..."
)


SCALER_PATH = find_file(

    [

        # CORRECT PATH — known from 7G-R2
        STEP6_ROOT /
        "step6b_preprocessing" /
        "standard_scaler_params.json",

        # Absolute fallback
        Path(
            "/content/step6_unsupervised_ml/"
            "step6b_preprocessing/"
            "standard_scaler_params.json"
        ),

        # Legacy possible location
        Path(
            "/content/step6b_preprocessing/"
            "standard_scaler_params.json"
        ),

    ],

    "StandardScaler parameters"

)


print(
    "Scaler path:",
    SCALER_PATH
)


with open(
    SCALER_PATH,
    "r",
    encoding="utf-8"
) as f:

    scaler_raw = json.load(f)


# ----------------------------------------------------------------
# SCALER VECTOR EXTRACTION
# ----------------------------------------------------------------

def extract_vector(
    obj,
    keys,
    expected_features
):

    for key in keys:

        if key not in obj:
            continue


        value = obj[key]


        # Simple list
        if isinstance(
            value,
            list
        ):

            arr = np.asarray(
                value,
                dtype=np.float64
            )

            if len(arr) == expected_features:

                return arr


        # Dictionary feature -> value
        if isinstance(
            value,
            dict
        ):

            try:

                arr = np.asarray(

                    [

                        value[k]

                        for k in behavioral_features

                    ],

                    dtype=np.float64

                )

                if len(arr) == expected_features:

                    return arr

            except Exception:

                pass


    return None


scaler_mean = extract_vector(
    scaler_raw,
    [
        "mean",
        "mean_",
        "center"
    ],
    48
)

scaler_scale = extract_vector(
    scaler_raw,
    [
        "scale",
        "scale_",
        "std"
    ],
    48
)


if (
    scaler_mean is None
    or
    scaler_scale is None
):

    raise ValueError(
        "STOP — could not extract "
        "48 scaler mean/scale values."
    )


if len(scaler_mean) != 48:

    raise ValueError(
        "STOP — scaler mean length != 48."
    )


if len(scaler_scale) != 48:

    raise ValueError(
        "STOP — scaler scale length != 48."
    )


if not np.isfinite(
    scaler_mean
).all():

    raise ValueError(
        "STOP — scaler mean contains "
        "non-finite values."
    )


if not np.isfinite(
    scaler_scale
).all():

    raise ValueError(
        "STOP — scaler scale contains "
        "non-finite values."
    )


if np.any(
    scaler_scale == 0
):

    raise ValueError(
        "STOP — scaler contains zero scale."
    )


# ----------------------------------------------------------------
# SCALE FUTURE MATRIX
# ----------------------------------------------------------------

X_future_scaled = (

    X_future_np
    - scaler_mean

) / scaler_scale


if not np.isfinite(
    X_future_scaled
).all():

    raise ValueError(
        "STOP — scaled matrix contains "
        "non-finite values."
    )


print(
    "Scaler mean length:",
    len(scaler_mean)
)

print(
    "Scaler scale length:",
    len(scaler_scale)
)

print(
    "Scaled matrix:",
    X_future_scaled.shape
)

print(
    "Scaler compatibility: PASS"
)


# ----------------------------------------------------------------
# [10] LOAD CENTROIDS
# ----------------------------------------------------------------

print(
    "\n[10] REFERENCE CENTROIDS"
)


centroid_df = pd.read_csv(
    CENTROIDS
)


cluster_col = detect_column(
    centroid_df,
    [
        "dbscan_cluster",
        "cluster"
    ],
    "centroid cluster column"
)


centroid_feature_cols = [

    c for c in behavioral_features

    if c in centroid_df.columns

]


if len(
    centroid_feature_cols
) != 48:

    raise ValueError(
        "STOP — centroid schema does "
        "not contain all 48 features."
    )


centroid_matrix = (
    centroid_df[
        centroid_feature_cols
    ]
    .to_numpy(
        dtype=np.float64
    )
)


centroid_labels = (
    centroid_df[
        cluster_col
    ]
    .to_numpy()
)


if not np.isfinite(
    centroid_matrix
).all():

    raise ValueError(
        "STOP — centroid matrix "
        "contains non-finite values."
    )


print(
    "Reference clusters:",
    len(centroid_labels)
)

print(
    "Centroid matrix:",
    centroid_matrix.shape
)

print(
    "Centroids: PASS"
)


# ----------------------------------------------------------------
# [11] FUTURE ASSIGNMENT
# ----------------------------------------------------------------

print(
    "\n[11] FUTURE-LOG ASSIGNMENT"
)


def nearest_centroids(
    matrix,
    centroids
):

    matrix = np.asarray(
        matrix,
        dtype=np.float64
    )

    centroids = np.asarray(
        centroids,
        dtype=np.float64
    )


    if matrix.ndim != 2:

        raise ValueError(
            "Matrix must be 2D."
        )


    if centroids.ndim != 2:

        raise ValueError(
            "Centroids must be 2D."
        )


    if (
        matrix.shape[1]
        !=
        centroids.shape[1]
    ):

        raise ValueError(
            "Matrix/centroid "
            "dimension mismatch."
        )


    # ------------------------------------------------------------
    # Robust squared Euclidean distance
    # ------------------------------------------------------------

    x_norm = np.sum(
        matrix ** 2,
        axis=1,
        keepdims=True
    )


    c_norm = np.sum(
        centroids ** 2,
        axis=1,
        keepdims=True
    ).T


    distances_sq = (

        x_norm
        + c_norm
        - 2.0 * (
            matrix @ centroids.T
        )

    )


    distances_sq = np.maximum(
        distances_sq,
        0.0
    )


    nearest_idx = np.argmin(
        distances_sq,
        axis=1
    )


    row_indices = np.arange(
        len(matrix)
    )


    nearest_dist = np.sqrt(

        distances_sq[
            row_indices,
            nearest_idx
        ]

    )


    return (
        nearest_idx,
        nearest_dist
    )


nearest_idx, nearest_dist = (
    nearest_centroids(
        X_future_scaled,
        centroid_matrix
    )
)


future_assignments = pd.DataFrame({

    id_col:
        future_df[id_col].values,

    "production_dbscan_cluster":
        centroid_labels[nearest_idx],

    "distance_to_reference_centroid":
        nearest_dist

})


# ----------------------------------------------------------------
# [12] ASSIGNMENT VALIDATION
# ----------------------------------------------------------------

print(
    "\n[12] ASSIGNMENT VALIDATION"
)


if len(
    future_assignments
) != len(
    future_df
):

    raise ValueError(
        "STOP — assignment row mismatch."
    )


if future_assignments[
    id_col
].isna().any():

    raise ValueError(
        "STOP — missing IDs after assignment."
    )


if future_assignments[
    id_col
].duplicated().any():

    raise ValueError(
        "STOP — duplicate IDs after assignment."
    )


if not np.isfinite(
    future_assignments[
        "distance_to_reference_centroid"
    ]
).all():

    raise ValueError(
        "STOP — invalid centroid distances."
    )


print(
    "Rows assigned:",
    len(future_assignments)
)

print(
    "Unique clusters assigned:",
    future_assignments[
        "production_dbscan_cluster"
    ].nunique()
)

print(
    "Assignment integrity: PASS"
)


# ----------------------------------------------------------------
# [13] FUTURE DISTANCE SUMMARY
# ----------------------------------------------------------------

print(
    "\n[13] FUTURE DISTANCE SUMMARY"
)


distance_summary = pd.DataFrame({

    "metric": [

        "count",
        "mean_distance",
        "median_distance",
        "p95_distance",
        "max_distance"

    ],

    "value": [

        len(nearest_dist),

        float(
            np.mean(
                nearest_dist
            )
        ),

        float(
            np.median(
                nearest_dist
            )
        ),

        float(
            np.percentile(
                nearest_dist,
                95
            )
        ),

        float(
            np.max(
                nearest_dist
            )
        )

    ]

})


print(
    distance_summary.to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# [14] SAVE FUTURE VALIDATION
# ----------------------------------------------------------------

print(
    "\n[14] SAVING VALIDATION ARTIFACTS"
)


future_assignment_path = (

    VALIDATION_DIR /
    "step7h_future_log_simulated_assignments.csv"

)


distance_path = (

    VALIDATION_DIR /
    "step7h_future_log_distance_summary.csv"

)


feature_schema_path = (

    VALIDATION_DIR /
    "step7h_production_feature_schema.json"

)


future_assignments.to_csv(
    future_assignment_path,
    index=False
)


distance_summary.to_csv(
    distance_path,
    index=False
)


json_dump(

    {

        "feature_count":
            len(behavioral_features),

        "behavioral_features":
            behavioral_features,

        "scaler":
            str(SCALER_PATH),

        "centroids":
            str(CENTROIDS),

        "primary_model":
            "DBSCAN",

        "eps":
            0.5,

        "min_samples":
            20,

        "reference_non_noise_clusters":
            38,

        "reference_noise_sessions":
            1676,

        "reference_noise_fraction":
            0.14107744107744108,

        "assignment_method":
            "nearest_behavioral_centroid",

        "future_simulation_rows":
            len(future_df)

    },

    feature_schema_path

)


print(
    "Future assignments:",
    future_assignment_path
)

print(
    "Distance summary:",
    distance_path
)

print(
    "Feature schema:",
    feature_schema_path
)


# ----------------------------------------------------------------
# [15] BUILD PRODUCTION PACKAGE
# ----------------------------------------------------------------

print(
    "\n[15] BUILDING PRODUCTION PACKAGE"
)


if PACKAGE_DIR.exists():

    shutil.rmtree(
        PACKAGE_DIR
    )


PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Required production artifacts
artifact_candidates = [

    REFERENCE_JSON,

    SCALER_PATH,

    CENTROIDS,

    REFERENCE_ASSIGNMENTS,

    DISTANCE_SUMMARY,

    CONFIG,

    PRODUCTION_AUDIT,

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_cluster_sizes.csv",

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_detailed_cluster_profiles.csv",

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_behavioral_signatures.csv",

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_top_behavioral_features.csv",

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_cluster_interpretation_summary.csv",

    STEP7_ROOT /
    "step7d_detailed_behavioral_profiling" /
    "step7d_dbscan_noise_behavioral_profile.csv",

    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_corrected_stability_results.csv",

    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_pairwise_stability_results.csv",

    STEP7_ROOT /
    "step7e_corrected_cluster_validation" /
    "step7e_corrected_cluster_validation_audit.json",

    future_assignment_path,

    distance_path,

    feature_schema_path

]


artifact_paths = [

    p for p in artifact_candidates

    if p.exists()

]


print(
    "Existing artifacts:",
    len(artifact_paths)
)


manifest_files = []


for src in artifact_paths:

    dst = PACKAGE_DIR / src.name


    if dst.exists():

        dst = (
            PACKAGE_DIR /
            (
                src.parent.name
                + "__"
                + src.name
            )
        )


    shutil.copy2(
        src,
        dst
    )


    manifest_files.append({

        "source":
            str(src),

        "packaged_name":
            dst.name,

        "bytes":
            dst.stat().st_size,

        "sha256":
            sha256_file(dst)

    })


print(
    "Artifacts packaged:",
    len(manifest_files)
)


# ----------------------------------------------------------------
# [16] MANIFEST
# ----------------------------------------------------------------

print(
    "\n[16] PACKAGE MANIFEST"
)


manifest = {

    "package":
        "STEP_7_ML_PRODUCTION_COMPLETE",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source":
        str(SOURCE),

    "source_sha256":
        source_sha_before,

    "source_verified":
        True,

    "primary_model":
        "DBSCAN",

    "dbscan_eps":
        0.5,

    "dbscan_min_samples":
        20,

    "reference_non_noise_clusters":
        38,

    "reference_noise_sessions":
        1676,

    "reference_noise_fraction":
        0.14107744107744108,

    "behavioral_feature_count":
        48,

    "future_simulation_rows":
        len(future_df),

    "assignment_method":
        "nearest_behavioral_centroid",

    "artifacts":
        manifest_files

}


MANIFEST_PATH = (
    PACKAGE_DIR /
    "step7h_manifest.json"
)


json_dump(
    manifest,
    MANIFEST_PATH
)


# ----------------------------------------------------------------
# [17] ZIP
# ----------------------------------------------------------------

print(
    "\n[17] CREATING ZIP"
)


if ZIP_PATH.exists():

    ZIP_PATH.unlink()


with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    compression=zipfile.ZIP_DEFLATED

) as zf:

    for file_path in PACKAGE_DIR.rglob("*"):

        if file_path.is_file():

            zf.write(

                file_path,

                arcname=file_path.relative_to(
                    PACKAGE_DIR
                )

            )


zip_sha = sha256_file(
    ZIP_PATH
)


print(
    "ZIP:",
    ZIP_PATH
)

print(
    "ZIP size:",
    ZIP_PATH.stat().st_size,
    "bytes"
)

print(
    "ZIP SHA-256:",
    zip_sha
)


# ----------------------------------------------------------------
# [18] ZIP INTEGRITY
# ----------------------------------------------------------------

print(
    "\n[18] ZIP INTEGRITY"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    bad_file = zf.testzip()

    if bad_file is not None:

        raise ValueError(
            "STOP — corrupted ZIP member: "
            + bad_file
        )

    zip_members = zf.namelist()


print(
    "ZIP members:",
    len(zip_members)
)

print(
    "ZIP integrity: PASS"
)


# ----------------------------------------------------------------
# [19] SOURCE PROTECTION
# ----------------------------------------------------------------

print(
    "\n[19] SOURCE PROTECTION"
)


source_sha_after = sha256_file(
    SOURCE
)


print(
    "SHA before:",
    source_sha_before
)

print(
    "SHA after: ",
    source_sha_after
)


if source_sha_before != source_sha_after:

    raise ValueError(
        "STOP — source dataset changed."
    )


print(
    "Source unchanged: PASS"
)


# ----------------------------------------------------------------
# [20] FINAL AUDIT
# ----------------------------------------------------------------

print(
    "\n[20] FINAL AUDIT"
)


audit = {

    "source_sha_expected":
        EXPECTED_SOURCE_SHA,

    "source_sha_before":
        source_sha_before,

    "source_sha_after":
        source_sha_after,

    "source_unchanged":
        (
            source_sha_before
            ==
            source_sha_after
        ),

    "source_shape":
        list(df.shape),

    "session_id_column":
        id_col,

    "missing_ids":
        missing_ids,

    "duplicate_ids":
        duplicate_ids,

    "unique_ids":
        unique_ids,

    "behavioral_feature_count":
        len(behavioral_features),

    "feature_schema_pass":
        True,

    "scaler_path":
        str(SCALER_PATH),

    "scaler_compatibility":
        True,

    "future_simulation_rows":
        len(future_df),

    "future_assignment_rows":
        len(future_assignments),

    "future_assignment_pass":
        True,

    "reference_non_noise_clusters":
        38,

    "reference_noise_sessions":
        1676,

    "reference_noise_fraction":
        0.14107744107744108,

    "primary_model":
        "DBSCAN",

    "eps":
        0.5,

    "min_samples":
        20,

    "assignment_method":
        "nearest_behavioral_centroid",

    "artifact_count":
        len(manifest_files),

    "zip_path":
        str(ZIP_PATH),

    "zip_sha256":
        zip_sha,

    "status":
        "PASS"

}


AUDIT_PATH = (

    VALIDATION_DIR /
    "step7h_production_validation_audit.json"

)


json_dump(
    audit,
    AUDIT_PATH
)


audit_sha = sha256_file(
    AUDIT_PATH
)


# ----------------------------------------------------------------
# FINAL
# ----------------------------------------------------------------

print("\n")
print("=" * 70)
print("STEP 7H-R — COMPLETE")
print("=" * 70)

print(
    "\nSource verification: PASS"
)

print(
    "Session integrity: PASS"
)

print(
    "Behavioral features:",
    len(behavioral_features)
)

print(
    "Scaler compatibility: PASS"
)

print(
    "Future-log simulation rows:",
    len(future_df)
)

print(
    "Future assignments:",
    len(future_assignments)
)

print(
    "Assignment validation: PASS"
)

print(
    "\nPrimary model: DBSCAN"
)

print(
    "eps: 0.5"
)

print(
    "min_samples: 20"
)

print(
    "Reference clusters: 38"
)

print(
    "Reference noise: 1676"
)

print(
    "Reference noise fraction: 0.141077"
)

print(
    "\nArtifacts packaged:",
    len(manifest_files)
)

print(
    "\nZIP:",
    ZIP_PATH
)

print(
    "ZIP size:",
    ZIP_PATH.stat().st_size,
    "bytes"
)

print(
    "ZIP SHA-256:",
    zip_sha
)

print(
    "\nAudit:",
    AUDIT_PATH
)

print(
    "Audit SHA-256:",
    audit_sha
)

print(
    "\nSTATUS: PASS"
)

print(
    "NEXT: STEP 8 — DASHBOARD DATA / GEOIP / ATTACKER IPS / "
    "MITRE / SESSION REPLAY / BEHAVIORAL CLUSTERS"
)

STEP 7H-R — PRODUCTION ARTIFACT PACKAGE + FUTURE-LOG VALIDATION

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 56)

[3] SESSION INTEGRITY
ID column: reconstructed_session_id
Missing IDs: 0
Duplicate IDs: 0
Unique IDs: 11880
Session integrity: PASS

[4] LOADING PRODUCTION REFERENCE
Reference: PASS
Centroids: PASS
Reference assignments: PASS
Distance summary: PASS
Production config: PASS
Production audit: PASS

[5] PRODUCTION CONFIG
{
  "primary_model": "DBSCAN",
  "eps": 0.5,
  "min_samples": 20,
  "source_sha256": "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534",
  "session_id_column": "reconstructed_session_id",
  "cluster_column": "dbscan_cluster",
  "behavioral_feature_count": 48,
  "behavioral_features": [
    "event_count",
    "session_duration_seconds",
    "auth_attempt_tot

In [ ]:
# ================================================================
# STEP 7 — FINAL ARCHIVE + HASHED SNAPSHOT
# ================================================================
# Purpose:
#   1) Preserve the complete Step 7 production artifacts
#   2) Verify source integrity
#   3) Generate SHA-256 hashes for important artifacts
#   4) Build a manifest
#   5) Create a final immutable ZIP snapshot
#
# Source dataset is NEVER modified.
# ================================================================

from pathlib import Path
import hashlib
import json
import shutil
import zipfile
from datetime import datetime, timezone

# ------------------------------------------------
# CONFIG
# ------------------------------------------------

SOURCE_PATH = Path(
    "/content/session_behavioral_features_v4/"
    "session_behavioral_features_v4.csv"
)

STEP7_ROOT = Path("/content/step7_cluster_profiling")

PRODUCTION_ZIP = Path(
    "/content/STEP_7_ML_PRODUCTION_COMPLETE.zip"
)

FINAL_ARCHIVE = Path(
    "/content/STEP_7_FINAL_ARCHIVE.zip"
)

MANIFEST_PATH = Path(
    "/content/STEP_7_FINAL_ARCHIVE_MANIFEST.json"
)

EXPECTED_SOURCE_SHA = (
    "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"
)

# ------------------------------------------------
# HELPERS
# ------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def file_info(path):
    return {
        "path": str(path),
        "name": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def collect_files(root):
    files = []

    if not root.exists():
        raise FileNotFoundError(
            f"STOP — Step 7 directory missing:\n{root}"
        )

    for p in root.rglob("*"):
        if p.is_file():
            files.append(p)

    return sorted(files)


# ------------------------------------------------
# [1] SOURCE VERIFICATION
# ------------------------------------------------

print("=" * 70)
print("STEP 7 — FINAL ARCHIVE + HASHED SNAPSHOT")
print("=" * 70)

print("\n[1] SOURCE VERIFICATION")

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"STOP — source dataset missing:\n{SOURCE_PATH}"
    )

source_sha_before = sha256_file(SOURCE_PATH)

print("Expected SHA:", EXPECTED_SOURCE_SHA)
print("Actual SHA:  ", source_sha_before)

if source_sha_before != EXPECTED_SOURCE_SHA:
    raise RuntimeError(
        "STOP — SOURCE SHA MISMATCH. "
        "Archive will NOT be created."
    )

print("Verification: PASS")


# ------------------------------------------------
# [2] STEP 7 INVENTORY
# ------------------------------------------------

print("\n[2] STEP 7 ARTIFACT INVENTORY")

step7_files = collect_files(STEP7_ROOT)

print("Step 7 files found:", len(step7_files))

if len(step7_files) == 0:
    raise RuntimeError(
        "STOP — no Step 7 artifacts found."
    )


# ------------------------------------------------
# [3] HASH ALL STEP 7 ARTIFACTS
# ------------------------------------------------

print("\n[3] HASHING STEP 7 ARTIFACTS")

artifact_manifest = []

for path in step7_files:
    info = file_info(path)

    relative_path = path.relative_to(STEP7_ROOT)

    info["relative_path"] = str(relative_path)

    artifact_manifest.append(info)

print(
    "Hashed artifacts:",
    len(artifact_manifest)
)


# ------------------------------------------------
# [4] VERIFY PRODUCTION ZIP
# ------------------------------------------------

print("\n[4] PRODUCTION ZIP VERIFICATION")

production_zip_info = None

if PRODUCTION_ZIP.exists():

    production_zip_info = file_info(PRODUCTION_ZIP)

    print("Production ZIP:", PRODUCTION_ZIP)
    print("Size:", production_zip_info["size_bytes"], "bytes")
    print("SHA-256:", production_zip_info["sha256"])

else:

    print(
        "WARNING — production ZIP not found:"
    )
    print(PRODUCTION_ZIP)


# ------------------------------------------------
# [5] SOURCE RE-CHECK
# ------------------------------------------------

print("\n[5] SOURCE PROTECTION CHECK")

source_sha_after = sha256_file(SOURCE_PATH)

print("SHA before:", source_sha_before)
print("SHA after: ", source_sha_after)

if source_sha_before != source_sha_after:
    raise RuntimeError(
        "STOP — SOURCE DATASET CHANGED DURING ARCHIVING."
    )

print("Source unchanged: PASS")


# ------------------------------------------------
# [6] BUILD MANIFEST
# ------------------------------------------------

print("\n[6] BUILDING MANIFEST")

manifest = {
    "archive_name": FINAL_ARCHIVE.name,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "step": "STEP_7",

    "status": "FINAL_ARCHIVE",

    "source": {
        "path": str(SOURCE_PATH),
        "expected_sha256": EXPECTED_SOURCE_SHA,
        "sha256_before": source_sha_before,
        "sha256_after": source_sha_after,
        "unchanged": (
            source_sha_before == source_sha_after
        ),
    },

    "step7_root": str(STEP7_ROOT),

    "artifact_count": len(
        artifact_manifest
    ),

    "artifacts": artifact_manifest,

    "production_zip": production_zip_info,

    "ml_configuration": {
        "primary_model": "DBSCAN",
        "eps": 0.5,
        "min_samples": 20,
        "behavioral_features": 48,
        "reference_non_noise_clusters": 38,
        "reference_noise_count": 1676,
        "reference_noise_fraction": 0.14107744107744108,
        "future_assignment_method":
            "nearest_behavioral_centroid",
    },

    "protection_rules": {
        "source_modified": False,
        "label_leakage": False,
        "session_integrity_required": True,
        "behavioral_feature_schema_locked": True,
    },
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Manifest saved:")
print(MANIFEST_PATH)


# ------------------------------------------------
# [7] CREATE FINAL ARCHIVE
# ------------------------------------------------

print("\n[7] CREATING FINAL STEP 7 ARCHIVE")

if FINAL_ARCHIVE.exists():
    FINAL_ARCHIVE.unlink()

with zipfile.ZipFile(
    FINAL_ARCHIVE,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as z:

    # Add complete Step 7 directory
    for path in step7_files:

        arcname = Path(
            "STEP_7_ARTIFACTS"
        ) / path.relative_to(STEP7_ROOT)

        z.write(
            path,
            arcname=str(arcname)
        )

    # Add production ZIP if available
    if PRODUCTION_ZIP.exists():

        z.write(
            PRODUCTION_ZIP,
            arcname=(
                "STEP_7_PRODUCTION/"
                + PRODUCTION_ZIP.name
            )
        )

    # Add manifest
    z.write(
        MANIFEST_PATH,
        arcname=(
            "STEP_7_METADATA/"
            + MANIFEST_PATH.name
        )
    )


# ------------------------------------------------
# [8] ARCHIVE HASH
# ------------------------------------------------

print("\n[8] FINAL ARCHIVE HASH")

final_archive_sha = sha256_file(
    FINAL_ARCHIVE
)

archive_size = FINAL_ARCHIVE.stat().st_size

print("Archive:")
print(FINAL_ARCHIVE)

print("Size:", archive_size, "bytes")

print("SHA-256:")
print(final_archive_sha)


# ------------------------------------------------
# [9] ZIP INTEGRITY
# ------------------------------------------------

print("\n[9] ZIP INTEGRITY")

with zipfile.ZipFile(
    FINAL_ARCHIVE,
    "r"
) as z:

    bad_member = z.testzip()
    members = z.namelist()

if bad_member is not None:
    raise RuntimeError(
        f"STOP — corrupted ZIP member: {bad_member}"
    )

print("ZIP members:", len(members))
print("ZIP integrity: PASS")


# ------------------------------------------------
# [10] FINAL MANIFEST UPDATE
# ------------------------------------------------

manifest["final_archive"] = {
    "path": str(FINAL_ARCHIVE),
    "size_bytes": archive_size,
    "sha256": final_archive_sha,
    "zip_members": len(members),
    "zip_integrity": "PASS",
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------
# [11] FINAL SOURCE CHECK
# ------------------------------------------------

print("\n[11] FINAL SOURCE PROTECTION")

final_source_sha = sha256_file(
    SOURCE_PATH
)

print("Expected:", EXPECTED_SOURCE_SHA)
print("Final:   ", final_source_sha)

if final_source_sha != EXPECTED_SOURCE_SHA:
    raise RuntimeError(
        "STOP — final source SHA mismatch."
    )

print("Source protection: PASS")


# ------------------------------------------------
# [12] FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 70)
print("STEP 7 — FINAL ARCHIVE COMPLETE")
print("=" * 70)

print()
print("Source SHA-256:")
print(final_source_sha)

print()
print("Step 7 artifacts:")
print(len(step7_files))

print()
print("Final archive:")
print(FINAL_ARCHIVE)

print()
print("Archive size:")
print(archive_size, "bytes")

print()
print("Archive SHA-256:")
print(final_archive_sha)

print()
print("Manifest:")
print(MANIFEST_PATH)

print()
print("ZIP integrity: PASS")
print("Source unchanged: PASS")
print("Archive integrity: PASS")

print()
print("STATUS: PASS")
print("NEXT: STEP 8 — DASHBOARD / GEOIP / MITRE / SESSION REPLAY")

STEP 7 — FINAL ARCHIVE + HASHED SNAPSHOT

[1] SOURCE VERIFICATION
Expected SHA: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual SHA:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[2] STEP 7 ARTIFACT INVENTORY
Step 7 files found: 82

[3] HASHING STEP 7 ARTIFACTS
Hashed artifacts: 82

[4] PRODUCTION ZIP VERIFICATION
Production ZIP: /content/STEP_7_ML_PRODUCTION_COMPLETE.zip
Size: 361406 bytes
SHA-256: a223d50565305502f823b87186f0acb0adad0aeee55dea2156f3193b25135b79

[5] SOURCE PROTECTION CHECK
SHA before: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
SHA after:  b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Source unchanged: PASS

[6] BUILDING MANIFEST
Manifest saved:
/content/STEP_7_FINAL_ARCHIVE_MANIFEST.json

[7] CREATING FINAL STEP 7 ARCHIVE

[8] FINAL ARCHIVE HASH
Archive:
/content/STEP_7_FINAL_ARCHIVE.zip
Size: 1297511 bytes
SHA-256:
38767175fb702691eb6dd124f7017946500b3ff890e1da2

In [ ]:
from google.colab import files
from pathlib import Path
import json
import hashlib

ARCHIVE = Path("/content/STEP_7_FINAL_ARCHIVE.zip")
MANIFEST = Path("/content/STEP_7_FINAL_ARCHIVE_MANIFEST.json")

EXPECTED_ARCHIVE_SHA = (
    "38767175fb702691eb6dd124f7017946500b3ff890e1da2cceddfdeaf5c8651a"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Verify files exist
if not ARCHIVE.exists():
    raise FileNotFoundError(f"Missing archive: {ARCHIVE}")

if not MANIFEST.exists():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST}")

# Verify archive hash
actual_sha = sha256_file(ARCHIVE)

print("=" * 70)
print("STEP 7 — DOWNLOAD VERIFICATION")
print("=" * 70)

print(f"Archive: {ARCHIVE}")
print(f"Expected SHA-256: {EXPECTED_ARCHIVE_SHA}")
print(f"Actual SHA-256:   {actual_sha}")

if actual_sha != EXPECTED_ARCHIVE_SHA:
    raise RuntimeError("STOP — archive SHA-256 mismatch.")

print("Archive SHA-256: PASS")
print("Manifest: PASS")
print("=" * 70)
print("READY FOR DOWNLOAD")
print("=" * 70)

# Download archive
files.download(str(ARCHIVE))

STEP 7 — DOWNLOAD VERIFICATION
Archive: /content/STEP_7_FINAL_ARCHIVE.zip
Expected SHA-256: 38767175fb702691eb6dd124f7017946500b3ff890e1da2cceddfdeaf5c8651a
Actual SHA-256:   38767175fb702691eb6dd124f7017946500b3ff890e1da2cceddfdeaf5c8651a
Archive SHA-256: PASS
Manifest: PASS
READY FOR DOWNLOAD


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>